# Bagheria 2026 — thread Genere

Gap di genere fra i giovani a Bagheria, confrontato con Palermo, Sicilia e Italia.

**Prerequisiti**: `pipeline.fetch`, `pipeline.build` e `notebooks/analisi.ipynb` già eseguiti.
Le definizioni delle misure (tasso di occupazione, disoccupazione, fuori da lavoro e studio)
non vengono ridefinite qui: si riusano quelle del notebook condiviso, leggendo
`data/processed/analisi_condizione_15_24.csv`. Serve a garantire che i numeri di questo thread
e quelli degli altri due si sommino nella stessa proposal.

**Fasce d'età**: 15-24 sul lavoro, 9-24 sull'istruzione. Non sono scelte, sono le uniche
classi giovanili disponibili a livello comunale — la cella di verifica qui sotto lo mostra.

# **Caricamento**

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.width", 150)
pd.set_option("display.max_columns", 30)

RADICE = Path.cwd()
if RADICE.name == "notebooks":
    RADICE = RADICE.parent
PROCESSED = RADICE / "data" / "processed"


def leggi(nome: str) -> pd.DataFrame:
    """Codici come stringhe: `condizione` contiene 1, 12, 99 e pandas li leggerebbe come interi."""
    tabella = pd.read_csv(PROCESSED / nome, dtype=str)
    for colonna in ("valore", "eta_anni", "popolazione", "occupati", "tasso_occupazione",
                    "tasso_disoccupazione", "fuori_da_lavoro_e_studio"):
        if colonna in tabella.columns:
            tabella[colonna] = pd.to_numeric(tabella[colonna])
    if "anno" in tabella.columns:
        tabella["anno"] = tabella["anno"].astype(int)
    return tabella


atteso = PROCESSED / "analisi_condizione_15_24.csv"
if not atteso.exists():
    raise SystemExit(f"manca {atteso.name}: esegui prima notebooks/analisi.ipynb")

istr_lav = leggi("censpop_istr_lav_long.csv")
ottomila = leggi("ottomilacensus_long.csv")
indicatori = leggi("indicatori.csv")
codici = leggi("codici.csv")
territori = leggi("territori.csv")
condizione = leggi("analisi_condizione_15_24.csv")
popolazione = leggi("censpop_popolazione_long.csv")

BAGHERIA, PALERMO, SICILIA, ITALIA = "082006", "082053", "ITG1", "IT"
CONFRONTO = [BAGHERIA, PALERMO, SICILIA, ITALIA]
NOMI = territori.set_index("territorio")["nome_territorio"].to_dict()
ORDINE = ["Bagheria", "Palermo", "Sicilia", "Italia"]

etichette = codici.set_index(["dimensione", "codice"])["etichetta"].to_dict()
print("misure disponibili dal notebook condiviso:", [c for c in condizione.columns if "tasso" in c or "fuori" in c])

misure disponibili dal notebook condiviso: ['tasso_occupazione', 'tasso_disoccupazione', 'fuori_da_lavoro_e_studio']


# **Verifica di fattibilità del thread**
Il piano iniziale del thread prevedeva la decomposizione del gap occupazionale per titolo di
studio. Prima di scrivere l'analisi, si controlla che l'incrocio esista.

In [2]:
lavoro = istr_lav[istr_lav["tavola"].eq("lavoro")]
istruzione = istr_lav[istr_lav["tavola"].eq("istruzione")]

print("Tavola LAVORO — titoli di studio presenti:", sorted(lavoro["titolo_studio"].unique()))
print("Tavola ISTRUZIONE — titoli di studio presenti:", sorted(istruzione["titolo_studio"].unique()))
print("Tavola ISTRUZIONE — condizioni professionali presenti:", sorted(istruzione["condizione"].unique()))
print()
print("Classi d'età per tavola:")
print("  lavoro:    ", sorted(lavoro["eta"].unique()))
print("  istruzione:", sorted(istruzione["eta"].unique()))
print()
print("Anni con dato sulla classe 15-24 (lavoro):", sorted(lavoro[lavoro["eta"].eq("Y15-24")]["anno"].unique().tolist()))
print("Anni con dato sulla classe 9-24 (istruzione):", sorted(istruzione[istruzione["eta"].eq("Y9-24")]["anno"].unique().tolist()))

Tavola LAVORO — titoli di studio presenti: ['ALL']
Tavola ISTRUZIONE — titoli di studio presenti: ['ALL', 'BL', 'IL', 'LBNA', 'LSE', 'ML', 'ML_RDD', 'NED', 'PSE', 'RDD', 'USE_IF']
Tavola ISTRUZIONE — condizioni professionali presenti: ['99']

Classi d'età per tavola:
  lavoro:     ['Y15-24', 'Y25-49', 'Y50-64', 'Y_GE15', 'Y_GE65']
  istruzione: ['Y25-49', 'Y50-64', 'Y9-24', 'Y_GE65', 'Y_GE9']

Anni con dato sulla classe 15-24 (lavoro): [2018, 2019, 2021, 2022, 2023, 2024]
Anni con dato sulla classe 9-24 (istruzione): [2018, 2019, 2020, 2021, 2022, 2023, 2024]


**📌 Risultato chiave** — L'incrocio *titolo di studio × condizione professionale* non esiste a livello comunale: nella tavola lavoro il titolo è solo `ALL`, in quella istruzione la condizione è solo `99`. La decomposizione del gap per titolo di studio non è difficile, è impossibile con questi dati — il piano del thread cambia di conseguenza.

> **La decomposizione per titolo di studio non è possibile.** Nella tavola sul lavoro il titolo
> di studio esiste solo come `ALL`: il censimento permanente non pubblica a livello comunale
> l'incrocio condizione professionale × titolo di studio. Nella tavola sull'istruzione vale il
> simmetrico: i titoli di studio ci sono tutti, ma la condizione professionale è solo `99`, il totale.
>
> Le due tavole si toccano solo sui totali, quindi *"tra le diplomate, quante lavorano"* non è una
> domanda a cui questi dati rispondono. Non è aggirabile con un'aggregazione diversa.
>
> **Cosa si fa invece.** Si misurano i due gap separatamente — occupazione (15-24) e istruzione
> (9-24) — e si guarda se vanno nella stessa direzione. È una domanda diversa e più debole della
> decomposizione, ma è onesta: dice se le ragazze di Bagheria sono svantaggiate *anche* nello
> studio o *solo* nel lavoro.
>
> Nota sulle fasce: lavoro 15-24, istruzione 9-24. Non sono sovrapponibili e non vanno mai
> mostrate come se fossero la stessa popolazione.

# **Gap di genere sull'occupazione, 15-24**
Le tre misure definite nel notebook condiviso, lette per genere. Il 2020 manca alla fonte.

In [3]:
giovani = condizione[condizione["territorio"].isin(CONFRONTO)].copy()

# Serie di Bagheria: le tre misure per maschi e femmine.
bagheria = (giovani[giovani["territorio"].eq(BAGHERIA) & giovani["genere"].isin(["M", "F"])]
    .pivot_table(index="anno", columns="genere",
                 values=["tasso_occupazione", "tasso_disoccupazione", "fuori_da_lavoro_e_studio"]))
bagheria.round(1)

fuori_da_lavoro_e_studio       tasso_disoccupazione       tasso_occupazione      
genere                        F     M                    F     M                 F     M
anno                                                                                    
2018                       38.0  37.4                 79.0  61.7               4.7  11.5
2019                       32.7  32.8                 76.8  61.9               5.2  11.8
2021                       29.9  29.7                 59.2  47.0               6.2  13.8
2022                       27.8  26.8                 54.3  41.8               7.7  15.4
2023                       31.1  30.4                 56.2  45.2               8.0  15.2
2024                       26.7  27.1                 45.6  35.1               8.2  16.5

In [4]:
# Il gap in punti percentuali (maschi meno femmine) sulle tre misure, per i quattro territori.
def gap_di_genere(misura: str) -> pd.DataFrame:
    largo = (giovani[giovani["genere"].isin(["M", "F"])]
             .pivot_table(index=["nome_territorio", "anno"], columns="genere", values=misura))
    return (largo["M"] - largo["F"]).rename(misura)


gap = pd.concat([gap_di_genere(m) for m in
                 ["tasso_occupazione", "tasso_disoccupazione", "fuori_da_lavoro_e_studio"]], axis=1)
gap = gap.round(1).reset_index()
gap["nome_territorio"] = pd.Categorical(gap["nome_territorio"], ORDINE, ordered=True)
gap = gap.sort_values(["nome_territorio", "anno"])
gap.to_csv(PROCESSED / "genere_gap_occupazione.csv", index=False)

gap.pivot(index="anno", columns="nome_territorio", values="tasso_occupazione")

nome_territorio,Bagheria,Palermo,Sicilia,Italia
anno,,,,
2018,6.8,5.7,7.3,8.2
2019,6.6,5.2,7.4,8.5
2021,7.6,6.4,8.8,9.8
2022,7.7,6.7,9.3,9.7
2023,7.2,6.3,9.4,9.5
2024,8.3,6.9,9.9,9.6


**📌 Risultato chiave** — A Bagheria fra 2018 e 2024 il tasso di occupazione femminile 15-24 sale da 4.7% a 8.2% e quello maschile da 11.5% a 16.5%: entrambi crescono, ma il gap in punti non si chiude, si allarga (6.8 → 8.3 pp).

> Un gap positivo sul tasso di occupazione significa che gli uomini lavorano di più. Sul
> `fuori_da_lavoro_e_studio` un gap positivo significa il contrario di quel che sembra: sono
> *gli uomini* a essere più spesso fuori da lavoro e studio. Le due misure vanno lette insieme,
> perché una fascia dove molti studiano ancora nasconde metà del fenomeno.

# **Quanto è preciso il gap? Intervalli di confidenza**
Prima di confrontare i gap fra territori serve l'ordine di grandezza dell'errore di ogni
numero: Bagheria è un comune, e i suoi conteggi sono piccoli. CI di Wilson sui tassi,
CI di Newcombe sulla differenza M-F.

Per gap occupazionale (o divario di genere nell'occupazione) si intende la differenza 
tra il tasso di occupazione maschile e quello femminile all'interno 
di una determinata popolazione o fascia d'età.

Avvertenza di metodo: i conteggi comunali non sono interi perché il censimento permanente
è una stima da registro + campione. I CI binomiali qui sotto trattano
i conteggi come esatti e sono quindi un **limite inferiore** dell'incertezza vera.

In [5]:
from statsmodels.stats.proportion import proportion_confint


def wilson(successi, totale):
    return proportion_confint(successi, totale, alpha=0.05, method="wilson")


def gap_con_ci(occ_m, pop_m, occ_f, pop_f):
    """Gap M-F fra proporzioni con CI 95% di Newcombe, costruito sui limiti di Wilson."""
    p_m, p_f = occ_m / pop_m, occ_f / pop_f
    l_m, u_m = wilson(occ_m, pop_m)
    l_f, u_f = wilson(occ_f, pop_f)
    g = p_m - p_f
    return (g,
            g - ((p_m - l_m) ** 2 + (u_f - p_f) ** 2) ** 0.5,
            g + ((u_m - p_m) ** 2 + (p_f - l_f) ** 2) ** 0.5)


# Conteggi non arrotondati dal notebook condiviso: per questo qualche decimale può
# differire dalla tabella dei gap sopra, che parte dai tassi già arrotondati.
conteggi_occ = (giovani[giovani["genere"].isin(["M", "F"])]
                .pivot_table(index=["territorio", "nome_territorio", "anno"],
                             columns="genere", values=["occupati", "popolazione"]))
conteggi_occ.columns = [f"{misura}_{gen}" for misura, gen in conteggi_occ.columns]

righe = []
for (territorio, nome, anno), r in conteggi_occ.iterrows():
    g, lo, hi = gap_con_ci(r["occupati_M"], r["popolazione_M"], r["occupati_F"], r["popolazione_F"])
    righe.append({"territorio": territorio, "nome_territorio": nome, "anno": anno,
                  "tasso_F": 100 * r["occupati_F"] / r["popolazione_F"],
                  "tasso_M": 100 * r["occupati_M"] / r["popolazione_M"],
                  "gap": 100 * g, "gap_lo": 100 * lo, "gap_hi": 100 * hi})
ci_gap = pd.DataFrame(righe)
ci_gap["rapporto_M_F"] = ci_gap["tasso_M"] / ci_gap["tasso_F"]
ci_gap = ci_gap.round({"tasso_F": 1, "tasso_M": 1, "gap": 1, "gap_lo": 1, "gap_hi": 1, "rapporto_M_F": 2})
ci_gap.to_csv(PROCESSED / "genere_gap_occupazione_ci.csv", index=False)

print("Bagheria — gap occupazionale M-F (punti) con CI 95%:")
print(ci_gap[ci_gap["territorio"].eq(BAGHERIA)][["anno", "tasso_F", "tasso_M", "gap", "gap_lo", "gap_hi"]]
      .to_string(index=False))
# CI di Wilson anche sul LIVELLO femminile: il claim "tasso più basso del panel" non è
# testato dai CI sul gap; il test formale sta nella sezione LPM (effetti principali).
livelli_2024 = []
for (territorio, nome, anno), r in conteggi_occ.iterrows():
    if anno != 2024:
        continue
    basso, alto = wilson(r["occupati_F"], r["popolazione_F"])
    livelli_2024.append({"nome_territorio": nome,
                         "tasso_F": 100 * r["occupati_F"] / r["popolazione_F"],
                         "CI 95% basso": 100 * basso, "CI 95% alto": 100 * alto})
print("\nTasso di occupazione femminile 2024 con CI 95% di Wilson:")
print(pd.DataFrame(livelli_2024).set_index("nome_territorio").reindex(ORDINE).round(1).to_string())

print("\nAnno 2024, i quattro territori:")
ci_gap[ci_gap["anno"].eq(2024)].set_index("nome_territorio").reindex(ORDINE)[
    ["tasso_F", "tasso_M", "gap", "gap_lo", "gap_hi"]]

Bagheria — gap occupazionale M-F (punti) con CI 95%:
 anno  tasso_F  tasso_M  gap  gap_lo  gap_hi
 2018      4.7     11.5  6.9     5.5     8.2
 2019      5.2     11.8  6.7     5.3     8.1
 2021      6.2     13.8  7.6     6.1     9.1
 2022      7.7     15.4  7.7     6.1     9.3
 2023      8.0     15.2  7.2     5.5     8.8
 2024      8.2     16.5  8.3     6.6    10.0

Tasso di occupazione femminile 2024 con CI 95% di Wilson:
                 tasso_F  CI 95% basso  CI 95% alto
nome_territorio                                    
Bagheria             8.2           7.2          9.2
Palermo              9.6           9.3          9.9
Sicilia             10.4          10.3         10.5
Italia              17.3          17.2         17.3

Anno 2024, i quattro territori:


,tasso_F,tasso_M,gap,gap_lo,gap_hi
nome_territorio,,,,,
Bagheria,8.2,16.5,8.3,6.6,10.0
Palermo,9.6,16.5,6.9,6.4,7.5
Sicilia,10.4,20.3,9.9,9.7,10.1
Italia,17.3,26.9,9.7,9.6,9.7


**📌 Risultato chiave** — Gap occupazionale 2024 di Bagheria: **8.3 pp** [CI 95% 6.6-10.0]. Il gap è solido — nessun CI tocca lo zero, in nessun anno e in nessun territorio — ma la precisione comunale è di ±1.5-1.7 pp: le oscillazioni annue (7.7 → 7.2 → 8.3) sono rumore, non trend.

> Il gap esiste ed è solido: in nessun anno e in nessun territorio il CI 95% tocca lo zero.
> Ma la precisione comunale è di ±1.5-1.7 punti: le oscillazioni anno su anno di Bagheria
> (7.7 → 7.2 → 8.3 fra 2022 e 2024) stanno tutte dentro gli intervalli e non vanno
> raccontate come peggioramenti o recuperi annuali. Il confronto sensato è fra territori
> e su più anni, ed è quello che fanno le sezioni successive.

# **Punti percentuali o rapporto? Entrambi**
Il gap in punti risponde a "quanti punti separano i tassi"; il rapporto M/F a "quante
volte è più probabile che un ragazzo lavori rispetto a una coetanea". Con tassi base
molto diversi fra territori le due scale possono ordinare i territori in modo opposto:
riportarne una sola sarebbe una scelta di comodo.

In [6]:
# Il gap in punti e il rapporto fra i tassi, fianco a fianco.
print("Rapporto M/F sul tasso di occupazione 15-24, serie:")
print(ci_gap.pivot(index="anno", columns="nome_territorio", values="rapporto_M_F")[ORDINE].to_string())
print("\nAnno 2024, le due scale:")
ci_gap[ci_gap["anno"].eq(2024)].set_index("nome_territorio").reindex(ORDINE)[
    ["tasso_F", "tasso_M", "gap", "rapporto_M_F"]]

Rapporto M/F sul tasso di occupazione 15-24, serie:
nome_territorio  Bagheria  Palermo  Sicilia  Italia
anno                                               
2018                 2.47     1.90     1.97    1.58
2019                 2.29     1.75     1.93    1.59
2021                 2.23     1.84     2.04    1.65
2022                 2.00     1.77     2.00    1.59
2023                 1.89     1.68     1.94    1.56
2024                 2.01     1.72     1.95    1.56

Anno 2024, le due scale:


,tasso_F,tasso_M,gap,rapporto_M_F
nome_territorio,,,,
Bagheria,8.2,16.5,8.3,2.01
Palermo,9.6,16.5,6.9,1.72
Sicilia,10.4,20.3,9.9,1.95
Italia,17.3,26.9,9.7,1.56


**📌 Risultato chiave** — Le due scale ordinano i territori in modo opposto. **In punti** Bagheria (8.3) sta sopra Palermo (6.9) ma sotto Sicilia (9.9) e Italia (9.7); **in rapporto M/F** (2.01) è la peggiore del panel (Palermo 1.72, Sicilia 1.95, Italia 1.56). Il tratto locale non è l'ampiezza del divario: è il **livello** del tasso femminile, 8.2%, il più basso dei quattro territori — CI 95% di Wilson [7.2-9.2], interamente sotto il 9.6 di Palermo; il test formale dei livelli sta nella sezione LPM (tutti i confronti p ≤ 0.009).

> Le due scale raccontano storie opposte, ed è il motivo per cui vanno dichiarate entrambe:
>
> - **in punti**, il gap 2024 di Bagheria (8.3) è sopra Palermo (6.9) ma sotto Sicilia (9.9)
>   e Italia (9.7);
> - **in rapporto**, Bagheria è la peggiore delle quattro: un ragazzo ha il doppio della
>   probabilità di lavorare di una coetanea (2.0, contro l'1.6 nazionale) — con una cautela: verso la Sicilia lo scarto
>   (2.01 contro 1.95) è dentro il rumore campionario e nel 2023 l'ordine era
>   invertito (1.89 contro 1.94); il primato nel rapporto è robusto verso Palermo
>   e Italia, quello nel livello verso tutti.
>
> Quando i tassi sono bassi per entrambi i generi, gli stessi punti percentuali pesano
> molto di più. Le scale divergono anche nel tempo: dal 2018 il gap in punti sale
> (6.9 → 8.3) mentre il rapporto scende (2.47 → 2.01), perché entrambi i tassi crescono.
> Ogni claim della proposal deve dire quale scala sta usando.

# **Il gap di Bagheria è un'anomalia locale? Modello lineare di probabilità**
La domanda di policy del thread — gap locale o regionale — testata formalmente invece che
a occhio. GLM binomiale con link identità sui conteggi aggregati: è il modello lineare di
probabilità, quindi i coefficienti si leggono direttamente in punti percentuali.
L'interazione genere × territorio stima la differenza fra il gap di Bagheria e quello di
ciascun benchmark; gli effetti principali del territorio (con F come riferimento) stimano
la differenza fra i **tassi femminili** — gap e livello escono dallo stesso modello. Confronti pre-specificati, per non pescare a strascico: anno di
riferimento 2024, e pooled 2022-2024 come robustezza.

Caveat sul pooled: tre annualità contano più volte le stesse persone, quindi i suoi CI
sono un po' ottimisti. Il pooled serve a stabilizzare il punto, non a moltiplicare i dati.

In [7]:
import warnings

import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.tools.sm_exceptions import DomainWarning, PerfectSeparationWarning

# Il modello è saturo (una media per cella genere × territorio): il link identità fuori
# dal dominio [0,1] e la "separazione perfetta" sono attesi qui, non problemi di stima.
warnings.filterwarnings("ignore", category=DomainWarning)
warnings.filterwarnings("ignore", category=PerfectSeparationWarning)
warnings.filterwarnings("ignore", message="divide by zero", category=RuntimeWarning)

BENCHMARK = ["Palermo", "Sicilia", "Italia"]


def lpm_gap_e_livelli(anni: list, periodo: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    """LPM sui conteggi aggregati: interazioni = eccesso di gap, effetti principali = livelli F.

    Con F e Bagheria come riferimenti, l'effetto principale del territorio è la differenza
    fra i tassi FEMMINILI (benchmark − Bagheria): è il test formale del claim sul livello.
    """
    d = (giovani[giovani["genere"].isin(["M", "F"]) & giovani["anno"].isin(anni)]
         .groupby(["nome_territorio", "genere"], as_index=False)[["occupati", "popolazione"]].sum())
    d["p"] = d["occupati"] / d["popolazione"]
    modello = smf.glm("p ~ C(nome_territorio, Treatment('Bagheria')) * C(genere, Treatment('F'))",
                      data=d, family=sm.families.Binomial(link=sm.families.links.Identity()),
                      var_weights=d["popolazione"]).fit()
    intervalli = modello.conf_int()
    gap, livelli = [], []
    for nome in BENCHMARK:
        principale = f"C(nome_territorio, Treatment('Bagheria'))[T.{nome}]"
        interazione = f"{principale}:C(genere, Treatment('F'))[T.M]"
        basso, alto = intervalli.loc[interazione]
        # L'interazione stima gap_benchmark - gap_Bagheria: segno invertito per leggere
        # l'eccesso locale (positivo = il gap di Bagheria è più largo).
        gap.append({"periodo": periodo, "confronto": f"vs {nome}",
                    "eccesso gap Bagheria (pp)": -100 * modello.params[interazione],
                    "CI 95% basso": -100 * alto, "CI 95% alto": -100 * basso,
                    "p": modello.pvalues[interazione]})
        basso, alto = intervalli.loc[principale]
        livelli.append({"periodo": periodo, "confronto": f"vs {nome}",
                        "livello F: benchmark − Bagheria (pp)": 100 * modello.params[principale],
                        "CI 95% basso": 100 * basso, "CI 95% alto": 100 * alto,
                        "p": modello.pvalues[principale]})
    return pd.DataFrame(gap), pd.DataFrame(livelli)


coppie = [lpm_gap_e_livelli([2024], "2024"),
          lpm_gap_e_livelli([2022, 2023, 2024], "pooled 2022-2024")]
eccessi = pd.concat([g for g, _ in coppie], ignore_index=True)
livelli_f = pd.concat([l for _, l in coppie], ignore_index=True)

print("Livello femminile: di quanto il tasso F di ciascun benchmark supera quello di Bagheria.")
print("Positivo e significativo su ogni confronto e periodo: il primato negativo del livello è testato.")
print(livelli_f.round({"livello F: benchmark − Bagheria (pp)": 2, "CI 95% basso": 2,
                       "CI 95% alto": 2, "p": 4}).to_string(index=False))
print("\nEccesso di gap (interazioni): il gap di Bagheria meno quello di ciascun benchmark.")
eccessi.round({"eccesso gap Bagheria (pp)": 1, "CI 95% basso": 1, "CI 95% alto": 1, "p": 3})

Livello femminile: di quanto il tasso F di ciascun benchmark supera quello di Bagheria.
Positivo e significativo su ogni confronto e periodo: il primato negativo del livello è testato.
         periodo  confronto  livello F: benchmark − Bagheria (pp)  CI 95% basso  CI 95% alto      p
            2024 vs Palermo                                  1.40          0.35         2.45 0.0090
            2024 vs Sicilia                                  2.22          1.21         3.23 0.0000
            2024  vs Italia                                  9.09          8.08        10.09 0.0000
pooled 2022-2024 vs Palermo                                  1.18          0.58         1.78 0.0001
pooled 2022-2024 vs Sicilia                                  1.93          1.35         2.50 0.0000
pooled 2022-2024  vs Italia                                  8.89          8.32         9.46 0.0000

Eccesso di gap (interazioni): il gap di Bagheria meno quello di ciascun benchmark.


,periodo,confronto,eccesso gap Bagheria (pp),CI 95% basso,CI 95% alto,p
0,2024,vs Palermo,1.3,-0.4,3.1,0.130
1,2024,vs Sicilia,-1.6,-3.2,0.1,0.066
2,2024,vs Italia,-1.4,-3.0,0.3,0.107
3,pooled 2022-2024,vs Palermo,1.1,0.1,2.1,0.031
4,pooled 2022-2024,vs Sicilia,-1.8,-2.7,-0.8,0.000
5,pooled 2022-2024,vs Italia,-1.9,-2.9,-1.0,0.000


**📌 Risultato chiave** — Pooled 2022-2024, il gap di Bagheria è **+1.1 pp più largo di Palermo** (p=0.03) e **1.8-1.9 pp più stretto di Sicilia e Italia** (p<0.001); sul solo 2024 nessuna differenza è significativa. In punti percentuali il gap di Bagheria **non è un'anomalia locale**. Ma il "vantaggio" sulla Sicilia nasce dal tasso maschile più basso (16.5 contro 20.3), non da un tasso femminile migliore. Gli **effetti principali dello stesso modello testano i livelli femminili**: Bagheria sotto Palermo di 1.2 pp [CI 0.6-1.8, p=0.0001], sotto la Sicilia di 1.9 [1.3-2.5] e sotto l'Italia di 8.9 [8.3-9.5] (pooled 2022-2024; anche sul solo 2024 tutti p ≤ 0.009). Il gap non distingue Bagheria; **il livello sì**.

> Sul solo 2024 nessuna differenza fra gap è significativa al 5%: un anno singolo di un
> comune non ha la precisione per distinguerli. Sul pooled 2022-2024 il quadro si separa:
>
> - il gap di Bagheria è **più largo di quello di Palermo** (+1.1 punti, p=0.03);
> - ed è **più stretto di quello di Sicilia e Italia** (-1.8 e -1.9 punti, p<0.001).
>
> Quindi, in punti percentuali, il gap di genere di Bagheria non è un'anomalia locale: sta
> dentro il paesaggio regionale. Ma il "vantaggio" verso la Sicilia non è una buona
> notizia: nasce dal tasso maschile più basso (16.5 contro 20.3), non da un tasso femminile
> migliore. Messo insieme alla sezione sulle scale, il tratto distintivo di Bagheria non è
> l'ampiezza del divario: è il **livello** — le ragazze hanno il tasso di occupazione più
> basso del panel (8.2%) e lo svantaggio relativo più alto (M/F = 2.0). È questo il
> bersaglio per la proposal, non la chiusura di un gap "anomalo" che i dati non mostrano. La tabella dei livelli
> qui sopra lo formalizza: tutti e tre i confronti sono positivi e significativi, in
> entrambi i periodi.

# **Il gap si sta allargando? Trend 2018-2024**
OLS sul gap in punti, anno centrato sul 2021, interazione col territorio per confrontare
le pendenze. Sei punti temporali per territorio e ogni punto è a sua volta una stima:
il modello dà direzione e ordine di grandezza, non inferenza fine.

In [8]:
serie_gap = ci_gap[["nome_territorio", "anno", "gap"]].assign(anno_c=lambda d: d["anno"] - 2021)
trend = smf.ols("gap ~ anno_c * C(nome_territorio, Treatment('Bagheria'))", data=serie_gap).fit()

pendenze = (pd.DataFrame({"pendenza (pp/anno)": trend.params, "p": trend.pvalues})
            .join(trend.conf_int().set_axis(["CI 95% basso", "CI 95% alto"], axis=1)))
pendenze = pendenze[pendenze.index.str.startswith("anno_c")]
pendenze.index = (pendenze.index
    .str.replace("anno_c:C(nome_territorio, Treatment('Bagheria'))[T.",
                 "differenza vs Bagheria: ", regex=False)
    .str.replace("]", "", regex=False)
    .str.replace("anno_c", "Bagheria", regex=False))
pendenze.round(3)[["pendenza (pp/anno)", "CI 95% basso", "CI 95% alto", "p"]]

,pendenza (pp/anno),CI 95% basso,CI 95% alto,p
Bagheria,0.205,0.060,0.350,0.008
differenza vs Bagheria: Italia,0.058,-0.147,0.262,0.558
differenza vs Bagheria: Palermo,0.030,-0.174,0.235,0.757
differenza vs Bagheria: Sicilia,0.252,0.047,0.456,0.019


**📌 Risultato chiave** — Il gap in punti di Bagheria cresce di **+0.21 pp/anno** (CI 0.06-0.35, p=0.008): l'allargamento è reale, non rumore. Il passo è indistinguibile da Palermo e Italia; solo la Sicilia allarga più in fretta (+0.25 pp/anno in più). Nello stesso periodo il rapporto M/F **scende** (2.47 → 2.01): le due scale divergono perché entrambi i tassi salgono.

> Il gap in punti di Bagheria cresce di ~0.2 punti l'anno (CI 0.06-0.35, p=0.008):
> l'allargamento è reale, non rumore. Il passo è indistinguibile da Palermo e Italia;
> solo la Sicilia allarga significativamente più in fretta (+0.25 punti/anno in più).
> Coerente con la sezione sulle scale: il divario in punti si allarga mentre quello
> relativo si riduce, perché entrambi i tassi stanno salendo e quello maschile sale di
> più in valore assoluto.

# **Dentro il "fuori da lavoro e studio": la popolazione per stato**
Il tasso di disoccupazione ha per denominatore le sole forze di lavoro, che per le
ragazze di Bagheria sono qualche centinaio di persone: è la misura più fragile del
thread e da sola non regge conclusioni. Al suo posto: la popolazione 15-24 ripartita
in quattro stati esaustivi — occupati, in cerca, studenti, altri inattivi — tutti su
denominatore-popolazione. La partizione è verificata contro i totali prima dell'uso;
per costruzione "in cerca + altri inattivi" coincide con la misura condivisa
`fuori_da_lavoro_e_studio`.

In [9]:
lavoro_15_24 = istr_lav[
    istr_lav["tavola"].eq("lavoro")
    & istr_lav["territorio"].isin(CONFRONTO)
    & istr_lav["eta"].eq("Y15-24")
    & istr_lav["cittadinanza"].eq("TOTAL")
    & istr_lav["titolo_studio"].eq("ALL")
    & istr_lav["genere"].isin(["M", "F", "T"])]
largo = lavoro_15_24.pivot_table(index=["territorio", "anno", "genere"],
                                 columns="condizione", values="valore", aggfunc="sum")

# Verifica della partizione. Tolleranza e non uguaglianza: i conteggi comunali non sono
# interi, il censimento permanente è una stima registro + campione.
DETTAGLIO = ["1", "12", "5", "4", "24", "7"]  # occupato, in cerca, studente, casalinga/o, pensione, altro
assert (largo[DETTAGLIO].sum(axis=1) - largo["99"]).abs().max() < 0.001, "la partizione non ricostruisce il totale"
assert (largo[["1", "12"]].sum(axis=1) - largo["22"]).abs().max() < 0.001, "occupati + in cerca != forze di lavoro"
per_genere = largo["99"].unstack("genere")
assert (per_genere["M"] + per_genere["F"] - per_genere["T"]).abs().max() < 0.001, "M + F != T"

stati = pd.DataFrame({
    "occupati": largo["1"], "in cerca": largo["12"], "studenti": largo["5"],
    "altri inattivi": largo[["4", "24", "7"]].sum(axis=1),
})
quote = (100 * stati.div(largo["99"], axis=0)).reset_index()
quote["nome_territorio"] = quote["territorio"].map(NOMI)

lungo = (quote.melt(id_vars=["territorio", "nome_territorio", "anno", "genere"],
                    var_name="stato", value_name="quota")
         .assign(quota=lambda d: d["quota"].round(1)))
lungo.to_csv(PROCESSED / "genere_composizione_stato.csv", index=False)

forze_f = largo.loc[(BAGHERIA, slice(None), "F"), "22"].droplevel([0, 2]).round(0).astype(int)
print("Forze di lavoro femminili 15-24 a Bagheria (denominatore del tasso di disoccupazione):")
print(forze_f.to_dict())
print("\nComposizione 2024 (% della popolazione 15-24):")
(lungo[lungo["anno"].eq(2024) & lungo["genere"].isin(["M", "F"])]
    .pivot_table(index="nome_territorio", columns=["stato", "genere"], values="quota")
    .reindex(ORDINE)[["occupati", "in cerca", "studenti", "altri inattivi"]])

Forze di lavoro femminili 15-24 a Bagheria (denominatore del tasso di disoccupazione):
{2018: 676, 2019: 663, 2021: 432, 2022: 479, 2023: 528, 2024: 434}

Composizione 2024 (% della popolazione 15-24):


stato           occupati       in cerca      studenti       altri inattivi      
genere                 F     M        F    M        F     M              F     M
nome_territorio                                                                 
Bagheria             8.2  16.5      6.9  8.9     65.1  56.4           19.9  18.2
Palermo              9.6  16.5      7.2  9.3     66.8  59.8           16.4  14.3
Sicilia             10.4  20.3      6.6  8.3     67.8  57.0           15.2  14.4
Italia              17.3  26.9      5.9  6.4     67.8  57.4            9.0   9.3

**📌 Risultato chiave** — Il minor tasso di occupazione femminile a 15-24 è assorbito dallo **studio**, non dall'inattività: studenti 65.1% F contro 56.4% M, in cerca 6.9% contro 8.9%, altri inattivi 19.9% contro 18.2%. E sotto il gap di genere ce n'è uno territoriale che colpisce **entrambi** i generi: "altri inattivi" al 14-20% a Bagheria, Palermo e Sicilia contro il ~9% nazionale.

> Tre letture:
>
> 1. Le ragazze **non** sono più spesso "fuori da tutto" dei ragazzi: altri inattivi
>    19.9% contro 18.2%, in cerca 6.9% contro 8.9%. Il tasso di occupazione più basso è
>    assorbito quasi per intero da più studio (65.1% contro 56.4%). A 15-24 il gap
>    occupazionale fotografa soprattutto ragazze ancora nel sistema formativo, non
>    inattività femminile.
> 2. Ciò che separa Bagheria, Palermo e la Sicilia dall'Italia è la quota di "altri
>    inattivi" per **entrambi** i generi: ~14-20% contro il ~9% nazionale.
> 3. Il denominatore del tasso di disoccupazione femminile di Bagheria è di 430-680
>    persone: i suoi sbalzi annuali sono in gran parte rumore. Le conclusioni del thread
>    usano occupazione e composizione, non quel tasso.
>
> Il punto 1 sposta la domanda di policy in avanti: se a 15-24 le ragazze studiano di più
> e il vantaggio non si converte poi in occupazione, il nodo sta all'uscita dal percorso
> formativo — che i dati comunali non permettono di osservare oltre i 24 anni (la classe
> successiva, 25-49, sfora il target giovani e non è scomponibile).

# **Dentro gli "altri inattivi": casalinghe a 15-24 anni**
La decomposizione sopra mostra "altri inattivi" quasi uguali fra i generi (19.9% F contro
18.2% M nel 2024). È un'uguaglianza apparente: i codici che la compongono — casalinga/o,
percettore/rice di pensione, altra condizione — hanno distribuzioni di genere opposte,
e tenerli aggregati nasconde il meccanismo.

In [10]:
# Scissione degli "altri inattivi" nei tre codici che li compongono.
inattivi_dettaglio = pd.DataFrame({
    "casalinghe_o_i": largo["4"],
    "percettori_pensione": largo["24"],
    "altra_condizione": largo["7"],
})
quote_inattivi = (100 * inattivi_dettaglio.div(largo["99"], axis=0)).round(1)
casalinghe = pd.concat([inattivi_dettaglio["casalinghe_o_i"].rename("conteggio"),
                        quote_inattivi.add_suffix("_%")], axis=1).reset_index()
casalinghe["nome_territorio"] = casalinghe["territorio"].map(NOMI)
casalinghe.to_csv(PROCESSED / "genere_casalinghe.csv", index=False)

# Versione a sei stati della composizione, per la figura: qui "altri inattivi" è scisso,
# mentre genere_composizione_stato.csv resta a quattro stati per gli altri thread.
sei_stati = pd.DataFrame({
    "occupati": largo["1"], "in cerca": largo["12"], "studenti": largo["5"],
    "casalinghe/i": largo["4"], "altra condizione": largo["7"], "pensione": largo["24"],
})
dettaglio = (100 * sei_stati.div(largo["99"], axis=0)).round(1).reset_index()
dettaglio["nome_territorio"] = dettaglio["territorio"].map(NOMI)
(dettaglio.melt(id_vars=["territorio", "nome_territorio", "anno", "genere"],
                var_name="stato", value_name="quota")
    .to_csv(PROCESSED / "genere_composizione_stato_dettaglio.csv", index=False))

serie_bag = (casalinghe[casalinghe["territorio"].eq(BAGHERIA) & casalinghe["genere"].eq("F")]
             .set_index("anno")[["conteggio", "casalinghe_o_i_%"]]
             .assign(conteggio=lambda d: d["conteggio"].round().astype(int)))
print("Bagheria — ragazze 15-24 che si dichiarano casalinghe:")
print(serie_bag.to_string())
print("\n2024, quote sulla popolazione 15-24 (%):")
(casalinghe[casalinghe["anno"].eq(2024) & casalinghe["genere"].isin(["M", "F"])]
    .pivot_table(index="nome_territorio", columns="genere",
                 values=["casalinghe_o_i_%", "altra_condizione_%"])
    .reindex(ORDINE)[["casalinghe_o_i_%", "altra_condizione_%"]])

Bagheria — ragazze 15-24 che si dichiarano casalinghe:
      conteggio  casalinghe_o_i_%
anno                             
2018        376              12.4
2019        324              10.9
2021        421              14.8
2022        364              12.8
2023        421              14.6
2024        387              13.4

2024, quote sulla popolazione 15-24 (%):


casalinghe_o_i_%      altra_condizione_%      
genere                         F    M                  F     M
nome_territorio                                               
Bagheria                    13.4  1.7                6.4  16.1
Palermo                     11.3  1.4                5.0  12.5
Sicilia                     10.1  1.2                5.1  12.7
Italia                       4.6  0.6                4.4   8.5

**📌 Risultato chiave** — Il **13.4% delle ragazze 15-24 di Bagheria si dichiara casalinga (387 persone)** contro l'1.7% dei ragazzi e il 4.6% delle coetanee italiane: quasi il triplo dell'incidenza nazionale. Gradiente territoriale netto (Italia 4.6 < Sicilia 10.1 < Palermo 11.3 < Bagheria 13.4) e serie 2018-2024 sempre fra 320 e 420 ragazze: è **strutturale**. Per i maschi gli "altri inattivi" sono invece quasi tutti "in altra condizione" (16.1%).

> L'uguaglianza era apparente: **il 13.4% delle ragazze 15-24 di Bagheria si dichiara
> casalinga (387 persone) contro l'1.7% dei ragazzi**; per i maschi gli "altri inattivi"
> sono quasi tutti "in altra condizione". E il fenomeno ha un gradiente territoriale
> netto — Italia 4.6%, Sicilia 10.1%, Palermo 11.3%, Bagheria 13.4%: il triplo
> dell'incidenza nazionale. La serie 2018-2024 (sempre fra 320 e 420 ragazze) dice che
> è strutturale, non episodico.
>
> È il carico di cura reso visibile nei dati, ed è un'evidenza da proposal: un target
> definito (le ~390 ragazze casalinghe), un meccanismo nominabile e un KPI naturale
> (la quota casalinghe 15-24, da portare prima al livello di Palermo, poi verso quello
> nazionale). Cautela: è la condizione autodichiarata al censimento, non una misura del
> lavoro di cura effettivo — va usata come marcatore del fenomeno, non come sua stima.

# **Chi sono le casalinghe? Un ragionamento di bounds per età**
La condizione professionale esiste solo sull'aggregato `Y15-24`; il registro demografico
dà però la popolazione femminile per **età singola**. Dal registro si prende solo la
*forma* della distribuzione per età e si delimita dove la quota di casalinghe può stare:
è un ragionamento di **bounds fra scenari estremi**, non una stima puntuale — l'età delle
387 non è osservata. Le due tavole condividono la stessa base demografica ufficiale (il
totale 15-24 coincide, come verifica la cella), quindi la ripartizione per età del
registro si applica direttamente alle quote della tavola lavoro.

In [11]:
# Struttura per età delle ragazze di Bagheria dal registro demografico (2024).
eta_f = (popolazione[popolazione["territorio"].eq(BAGHERIA)
                     & popolazione["anno"].eq(2024)
                     & popolazione["genere"].eq("F")
                     & popolazione["cittadinanza"].eq("TOTAL")
                     & popolazione["eta_anni"].notna()]
         .assign(eta=lambda d: d["eta_anni"].astype(int))
         .groupby("eta")["valore"].sum())
pop_15_24, pop_18_24, pop_20_24 = (eta_f.loc[15:24].sum(), eta_f.loc[18:24].sum(),
                                   eta_f.loc[20:24].sum())

# Denominatore della tavola lavoro: la sovrapponibilità col registro non si assume,
# si verifica — sotto l'unità le due basi coincidono e la ripartizione è trasferibile.
pop_f_lavoro = giovani.query("territorio == @BAGHERIA and genere == 'F' and anno == 2024")["popolazione"].item()
assert abs(pop_15_24 - pop_f_lavoro) < 1, f"registro {pop_15_24:.0f} != tavola lavoro {pop_f_lavoro:.0f}"

riga = casalinghe.query("territorio == @BAGHERIA and genere == 'F' and anno == 2024")
n_cas, quota_cas = riga["conteggio"].item(), riga["casalinghe_o_i_%"].item()

scenari = pd.DataFrame([
    ("distribuzione uniforme sulle età 15-24", quota_cas),
    ("nessuna sotto i 18 anni (tutte 18-24)", quota_cas / (pop_18_24 / pop_15_24)),
    ("nessuna sotto i 20 anni (tutte 20-24)", quota_cas / (pop_20_24 / pop_15_24)),
], columns=["scenario", "quota nella fascia interessata (%)"]).round(1)
scenari.to_csv(PROCESSED / "genere_casalinghe_bounds.csv", index=False)

# Eccesso rispetto all'incidenza italiana, in persone (denominatore della tavola lavoro).
cas_italia = casalinghe.query("territorio == @ITALIA and genere == 'F' and anno == 2024")["casalinghe_o_i_%"].item()
eccesso = (quota_cas - cas_italia) / 100 * pop_f_lavoro

print(f"Registro 2024, femmine di Bagheria: 15-24 = {pop_15_24:.0f}, "
      f"di cui 18-24 = {pop_18_24:.0f} e 20-24 = {pop_20_24:.0f}")
print(f"Casalinghe 15-24 (tavola lavoro): {n_cas:.0f} = {quota_cas}% della fascia; "
      f"eccesso sull'incidenza italiana ({cas_italia}%): {eccesso:.0f} ragazze")
scenari

Registro 2024, femmine di Bagheria: 15-24 = 2882, di cui 18-24 = 2053 e 20-24 = 1498
Casalinghe 15-24 (tavola lavoro): 387 = 13.4% della fascia; eccesso sull'incidenza italiana (4.6%): 254 ragazze


,scenario,quota nella fascia interessata (%)
0,distribuzione uniforme sulle età 15-24,13.4
1,nessuna sotto i 18 anni (tutte 18-24),18.8
2,nessuna sotto i 20 anni (tutte 20-24),25.8


**📌 Risultato chiave** — Il 13.4% sull'aggregato 15-24 è compatibile con concentrazioni molto diverse: se nessuna casalinga avesse meno di 20 anni sarebbero il **25.8% delle 20-24enni — una su quattro**; se nessuna ne avesse meno di 18, il 18.8% delle 18-24enni. L'eccesso rispetto all'incidenza italiana vale **254 ragazze**. L'età resta non osservata: sono bounds, non stime.

> Lo scenario uniforme (13.4% a ogni età, comprese le quindicenni in obbligo scolastico) è
> implausibile; quello tutto-20-24 è l'estremo opposto. La realtà sta in mezzo, e già
> l'estremo inferiore plausibile — nessuna minorenne, 18.8% — descrive quasi una
> diciotto-ventiquattrenne su cinque. Con la ritenzione di coorte (le ragazze sono ancora
> qui fino ai 25 anni) il quadro è coerente: la condizione si forma *prima* dell'uscita
> dal comune, dentro la finestra in cui un servizio può ancora intercettarla.
> Il fetch SDMX dello stato civile per età (oggi solo `ALL`) resta l'unico modo per dire
> se la concentrazione segue i matrimoni precoci: task "nuova fonte", da decidere in team.

# **Le casalinghe sono coniugate? Lo stato civile per età**

La quota di casalinghe dice *quante* ragazze stanno nel ruolo; non dice *da dove arriva*
il ruolo. Il candidato classico è il matrimonio precoce. A livello comunale il censimento
permanente non incrocia lo stato civile, ma la popolazione al 1° gennaio (`DCIS_POPRES1`,
base censuaria dal 2019) dà età singola × sesso × stato civile per tutti i comuni: fonte
diversa, dichiarata in ogni output, mai messa in serie con il censimento.

In [12]:
# Fonte diversa e dichiarata: DCIS_POPRES1 (popolazione al 1° gennaio, base censuaria dal
# 2019), perché il censimento permanente non incrocia lo stato civile a livello comunale
# (verificato in pipeline/fetch.py: la famiglia DEMCITMIG risponde NoRecordsFound).
# Stock al 1° gennaio contro media annua: mai in serie con SETA_1. Il riferimento è il
# 1.1.2025 — la fotografia di fine 2024, l'anno della tavola lavoro; il 1.1.2026 esiste
# ma pubblica solo il totale, senza dettaglio coniugale.
stato_civile = leggi("popres_stato_civile_long.csv")
ANNO_RIF = 2025

largo_sc = (stato_civile[stato_civile["eta_anni"].notna()]
            .pivot_table(index=["territorio", "anno", "genere", "eta_anni"],
                         columns="stato_civile", values="valore", aggfunc="sum"))

# Tre convenzioni della tavola, verificate e non assunte: (a) il dettaglio coniugale non è
# pubblicato dove è strutturalmente vuoto (coniugate sotto i 16 anni, unioni civili sotto
# i 18): NaN = 0; (b) sotto i 16 anni nubile == totale, esattamente; (c) con la (a) gli
# stati dettagliati ricostruiscono il totale 99 riga per riga, al centesimo.
DETTAGLIO_SC = ["1", "2", "3", "4", "15", "16", "17"]
dettaglio_sc = largo_sc[DETTAGLIO_SC].fillna(0)
con_dettaglio = largo_sc["1"].notna()          # tutto tranne il 1.1.2026, solo-totale
sotto_16 = largo_sc.index.get_level_values("eta_anni") < 16
assert (largo_sc.loc[sotto_16 & con_dettaglio, "1"]
        - largo_sc.loc[sotto_16 & con_dettaglio, "99"]).abs().max() < 0.001
assert (dettaglio_sc[con_dettaglio].sum(axis=1)
        - largo_sc.loc[con_dettaglio, "99"]).abs().max() < 0.001, \
    "gli stati dettagliati non ricostruiscono il totale"
anni_dettaglio = sorted(largo_sc[con_dettaglio].index.get_level_values("anno").unique())
assert ANNO_RIF in anni_dettaglio, f"manca il dettaglio al 1.1.{ANNO_RIF}"

# Sanità fra le fonti: le ragazze 15-24 al 1.1.2025 (POPRES) contro la media annua 2024
# (SETA_1). Uno scarto piccolo è atteso (momenti e fonti diverse), uno grande sarebbe un errore.
f_15_24_popres = largo_sc.loc[(BAGHERIA, ANNO_RIF, "F"), "99"].loc[15:24].sum()
f_15_24_seta = popolazione[
    popolazione["territorio"].eq(BAGHERIA) & popolazione["anno"].eq(2024)
    & popolazione["genere"].eq("F") & popolazione["cittadinanza"].eq("TOTAL")
    & popolazione["eta_anni"].between(15, 24)]["valore"].sum()
print(f"Controllo fonti, ragazze 15-24: POPRES 1.1.{ANNO_RIF} = {f_15_24_popres:,.0f} contro "
      f"media annua 2024 = {f_15_24_seta:,.0f} ({100 * (f_15_24_popres / f_15_24_seta - 1):+.1f}%)")

# "Già coniugate": coniugate, divorziate, vedove, unioni civili incluse. Lo stato civile
# osserva il matrimonio formale, NON le convivenze né la maternità.
GIA_CONIUGATE = ["2", "3", "4", "15", "16", "17"]
gia_largo = dettaglio_sc[GIA_CONIUGATE].sum(axis=1)
righe = []
for territorio in CONFRONTO:
    for genere in ("F", "M"):
        for anno in anni_dettaglio:
            for etichetta, (a0, a1) in {"15-24": (15, 24), "18-24": (18, 24),
                                        "20-24": (20, 24)}.items():
                gia = gia_largo.loc[(territorio, anno, genere)].loc[a0:a1].sum()
                tot = largo_sc.loc[(territorio, anno, genere), "99"].loc[a0:a1].sum()
                righe.append({"territorio": territorio, "nome_territorio": NOMI[territorio],
                              "anno": anno, "genere": genere, "fascia": etichetta,
                              "popolazione": round(tot), "gia_coniugate": round(gia),
                              "quota_gia_coniugate_pct": round(100 * gia / tot, 2)})
civile = pd.DataFrame(righe)
civile.to_csv(PROCESSED / "genere_stato_civile.csv", index=False)

print(f"\nQuota di ragazze già coniugate o unite civilmente al 1.1.{ANNO_RIF} (%):")
print(civile.query("genere == 'F' and anno == @ANNO_RIF")
      .pivot_table(index="nome_territorio", columns="fascia", values="quota_gia_coniugate_pct")
      .reindex(ORDINE)[["15-24", "18-24", "20-24"]].to_string())

# Il confronto che decide il meccanismo: anche se OGNI già-coniugata fosse casalinga,
# quante casalinghe restano nubili? Fonti diverse: è un bound, non un conto esatto.
gia_1524 = civile.query("territorio == @BAGHERIA and genere == 'F' "
                        "and anno == @ANNO_RIF and fascia == '15-24'")["gia_coniugate"].item()
cas_2024 = casalinghe.query("territorio == @BAGHERIA and genere == 'F' and anno == 2024")["conteggio"].item()
print(f"\nBagheria: {cas_2024:.0f} casalinghe 15-24 (media 2024) contro {gia_1524} ragazze "
      f"già coniugate al 1.1.{ANNO_RIF}: anche nell'ipotesi estrema che ogni coniugata "
      f"sia casalinga, almeno {cas_2024 - gia_1524:.0f} casalinghe "
      f"({100 * (cas_2024 - gia_1524) / cas_2024:.0f}%) sono nubili.")

serie_20_24 = civile.query("genere == 'F' and fascia == '20-24'")
print("\nSerie 20-24 (la fascia dove il fenomeno si concentra), quota già coniugate (%):")
print(serie_20_24.pivot_table(index="anno", columns="nome_territorio",
                              values="quota_gia_coniugate_pct")[ORDINE].to_string())

Controllo fonti, ragazze 15-24: POPRES 1.1.2025 = 2,882 contro media annua 2024 = 2,882 (+0.0%)



Quota di ragazze già coniugate o unite civilmente al 1.1.2025 (%):
fascia           15-24  18-24  20-24
nome_territorio                     
Bagheria          1.42   2.00   2.67
Palermo           1.61   2.30   3.21
Sicilia           1.52   2.15   2.90
Italia            1.16   1.65   2.25

Bagheria: 387 casalinghe 15-24 (media 2024) contro 41 ragazze già coniugate al 1.1.2025: anche nell'ipotesi estrema che ogni coniugata sia casalinga, almeno 346 casalinghe (89%) sono nubili.

Serie 20-24 (la fascia dove il fenomeno si concentra), quota già coniugate (%):
nome_territorio  Bagheria  Palermo  Sicilia  Italia
anno                                               
2019                 6.07     6.22     5.70    4.18
2020                 4.71     5.52     5.05    3.76
2021                 3.56     4.44     4.13    3.23
2022                 3.47     3.85     3.77    2.79
2023                 3.05     3.43     3.40    2.65
2024                 2.54     3.51     3.16    2.43
2025                 

**📌 Risultato chiave** — Il canale del matrimonio precoce **non regge i numeri**: al
1.1.2025 le ragazze 15-24 di Bagheria già coniugate (o in unione civile, o già uscite da
un matrimonio) sono **41, l'1.4%**, contro **387 casalinghe**: anche nell'ipotesi
estrema che ogni coniugata sia casalinga, **almeno l'89% delle casalinghe è nubile**. E
Bagheria non si sposa presto nemmeno in senso relativo: quota di già coniugate 20-24 al
2.7%, *sotto* Palermo (3.2%) e Sicilia (2.9%), con il matrimonio under-25 in caduta
ovunque (a Bagheria dal 6.1% del 1.1.2019 al 2.7%). Le due fonti combaciano sul
denominatore alla singola unità (2.882 ragazze in entrambe): lo scarto non è un
artefatto di misura. Il ruolo di casalinga a vent'anni, qui, è nella quasi totalità un
ruolo **da nubile, dentro la famiglia d'origine** — coerente con cura familiare e
scoraggiamento, non con la formazione di una famiglia propria. Limite dichiarato: lo
stato civile non osserva convivenze né maternità; il conteggio dei nati per età della
madre (demo.istat) è il check successivo, ed è un fetch nuovo. Per la policy la
conseguenza è direzionale: il servizio per le casalinghe ventenni è un servizio di
**attivazione** (rientro in formazione e lavoro), non solo di conciliazione.

# **Gap di genere sull'istruzione, 9-24**
Composizione per titolo di studio e quota con almeno il diploma.

In [13]:
# Categorie mutuamente esclusive: la loro somma torna esattamente al totale ALL (verificato sotto).
ALMENO_DIPLOMA = ["USE_IF", "BL", "ML_RDD"]   # diploma, terziario 1° livello, terziario 2° e dottorato
TITOLI = ["NED", "PSE", "LSE"] + ALMENO_DIPLOMA

scuola = istruzione[
    istruzione["territorio"].isin(CONFRONTO)
    & istruzione["eta"].eq("Y9-24")
    & istruzione["cittadinanza"].eq("TOTAL")
    & istruzione["genere"].isin(["M", "F", "T"])]

conteggi = scuola.pivot_table(index=["territorio", "anno", "genere"],
                              columns="titolo_studio", values="valore", aggfunc="sum")

# Controllo di coerenza: le categorie di dettaglio devono ricostruire il totale.
scarto = (conteggi[TITOLI].sum(axis=1) - conteggi["ALL"]).abs().max()
assert scarto == 0, f"le categorie non ricostruiscono il totale, scarto massimo {scarto}"

titoli = pd.DataFrame({
    "popolazione_9_24": conteggi["ALL"],
    "almeno_diploma_%": 100 * conteggi[ALMENO_DIPLOMA].sum(axis=1) / conteggi["ALL"],
    "licenza_media_%": 100 * conteggi["LSE"] / conteggi["ALL"],
    "nessun_titolo_o_elementare_%": 100 * conteggi[["NED", "PSE"]].sum(axis=1) / conteggi["ALL"],
}).round(1).reset_index()
titoli["nome_territorio"] = titoli["territorio"].map(NOMI)
titoli.to_csv(PROCESSED / "genere_istruzione.csv", index=False)

ultimo_anno = titoli["anno"].max()
(titoli[titoli["anno"].eq(ultimo_anno) & titoli["genere"].isin(["M", "F"])]
    .pivot(index="nome_territorio", columns="genere", values="almeno_diploma_%")
    .reindex(ORDINE)
    .assign(**{"gap M-F (punti)": lambda d: (d["M"] - d["F"]).round(1)}))

genere,F,M,gap M-F (punti)
nome_territorio,,,
Bagheria,33.4,29.2,-4.2
Palermo,30.5,28.7,-1.8
Sicilia,33.1,30.4,-2.7
Italia,34.5,32.3,-2.2


**📌 Risultato chiave** — Gap istruzione 9-24 = **-4.2 pp**: a Bagheria le ragazze arrivano almeno al diploma più dei coetanei (33.4% contro 29.2%), ed è il vantaggio femminile più ampio del panel (Palermo -1.8, Sicilia -2.7, Italia -2.2).

> Il gap sull'istruzione è **negativo**: le ragazze arrivano al diploma più spesso dei coetanei,
> a Bagheria come altrove.
>
> Il livello assoluto va letto con cautela — la fascia parte da 9 anni e include ragazzi che il
> diploma non possono ancora averlo, quindi la percentuale è strutturalmente bassa. Il confronto
> fra generi resta valido, perché il denominatore è distorto allo stesso modo per entrambi.

# **Verifica di composizione per età**
I tassi delle fasce 15-24 e 9-24 sono aggregati: se la struttura per età dentro la fascia
differisse fra generi o fra territori, parte dei gap sarebbe un artefatto di composizione
(chi ha 15 anni non lavora quasi mai, chi ne ha 12 non può avere un diploma). Le età
singole della demografia, disponibili dal 2021, permettono il controllo sul 2024.

In [14]:
singole_2024 = popolazione[
    popolazione["territorio"].isin(CONFRONTO)
    & popolazione["cittadinanza"].eq("TOTAL")
    & popolazione["genere"].isin(["M", "F"])
    & popolazione["eta_anni"].notna()
    & popolazione["anno"].eq(2024)]


def quota_fascia_alta(fascia, alta):
    """% della sottofascia più vecchia dentro la fascia, per territorio e genere (2024)."""
    dentro = singole_2024[singole_2024["eta_anni"].between(*fascia)]
    totale = dentro.groupby(["territorio", "genere"])["valore"].sum()
    parte = dentro[dentro["eta_anni"].between(*alta)].groupby(["territorio", "genere"])["valore"].sum()
    return (100 * parte / totale).unstack().rename(index=NOMI).reindex(ORDINE).round(1)


print("Fascia lavoro — quota dei 20-24 dentro i 15-24:")
print(quota_fascia_alta((15, 24), (20, 24)).to_string())
print("\nFascia istruzione — quota dei 19-24 dentro i 9-24:")
quota_fascia_alta((9, 24), (19, 24))

Fascia lavoro — quota dei 20-24 dentro i 15-24:
genere         F     M
territorio            
Bagheria    52.0  50.7
Palermo     49.2  49.6
Sicilia     50.8  50.6
Italia      50.2  50.7

Fascia istruzione — quota dei 19-24 dentro i 9-24:


genere,F,M
territorio,,
Bagheria,40.8,38.3
Palermo,38.5,38.7
Sicilia,39.5,40.1
Italia,38.8,39.7


**📌 Risultato chiave** — Il gap occupazionale **non** è un artefatto di composizione per età: le ragazze 15-24 di Bagheria sono semmai il gruppo più "vecchio" del panel (52.0% ha 20-24 anni), il che dovrebbe alzarne il tasso — il controllo è conservativo rispetto alla conclusione. Sul gap istruzione invece **~1.5 dei 4.2 punti sono mix per età**: il segno regge, la magnitudine di Bagheria va citata con questa cautela.

> - **Fascia lavoro 15-24**: la quota di 20-24enni sta fra il 49% e il 52% ovunque, per
>   entrambi i generi. Semmai le ragazze di Bagheria sono il gruppo leggermente più
>   "vecchio" (52.0%), il che dovrebbe *alzarne* il tasso di occupazione: il livello
>   femminile più basso del panel non è un artefatto di composizione, e il controllo è
>   conservativo rispetto alla conclusione.
> - **Fascia istruzione 9-24**: a Bagheria le femmine sono più concentrate nei 19-24 dei
>   maschi (40.8% contro 38.3%), mentre negli altri territori la differenza è sotto il
>   punto e spesso di segno opposto. Parte del gap di istruzione di Bagheria (-4.2) è
>   quindi composizione: assumendo che il diploma stia quasi tutto nei 19-24, l'ordine di
>   grandezza è 2.5 punti di mix × ~60 punti di differenza fra le sottofasce ≈ **1.5 punti
>   dei 4.2**. Il segno del vantaggio femminile regge (c'è anche dove il mix è pari), la
>   magnitudine di Bagheria va citata con questa cautela. La standardizzazione esatta non
>   è possibile: il titolo di studio per età singola non esiste a livello comunale.

# **I due gap a confronto**
La domanda del thread, nella forma che i dati permettono: le ragazze studiano di più e lavorano
di meno, e questo distingue Bagheria dai territori di riferimento?

In [15]:
anno_comune = min(giovani["anno"].max(), titoli["anno"].max())

occupazione_gap = (giovani[giovani["anno"].eq(anno_comune) & giovani["genere"].isin(["M", "F"])]
    .pivot_table(index="nome_territorio", columns="genere", values="tasso_occupazione"))
istruzione_gap = (titoli[titoli["anno"].eq(anno_comune) & titoli["genere"].isin(["M", "F"])]
    .pivot_table(index="nome_territorio", columns="genere", values="almeno_diploma_%"))

quadro = pd.DataFrame({
    "occupazione M": occupazione_gap["M"],
    "occupazione F": occupazione_gap["F"],
    "gap occupazione (M-F)": occupazione_gap["M"] - occupazione_gap["F"],
    "almeno diploma M": istruzione_gap["M"],
    "almeno diploma F": istruzione_gap["F"],
    "gap istruzione (M-F)": istruzione_gap["M"] - istruzione_gap["F"],
}).reindex(ORDINE).round(1)
quadro.to_csv(PROCESSED / "genere_quadro_sintesi.csv")

print(f"Anno: {anno_comune}   —   occupazione 15-24, istruzione 9-24")
quadro

Anno: 2024   —   occupazione 15-24, istruzione 9-24


,occupazione M,occupazione F,gap occupazione (M-F),almeno diploma M,almeno diploma F,gap istruzione (M-F)
nome_territorio,,,,,,
Bagheria,16.5,8.2,8.3,29.2,33.4,-4.2
Palermo,16.5,9.6,6.9,28.7,30.5,-1.8
Sicilia,20.3,10.4,9.9,30.4,33.1,-2.7
Italia,26.9,17.3,9.6,32.3,34.5,-2.2


**📌 Risultato chiave** — Il paradosso, nella sua forma più compatta: **istruzione -4.2 pp (a favore delle ragazze), occupazione +8.3 pp (a loro sfavore)**, e i due segni sono opposti in tutti e quattro i territori. Le ragazze di Bagheria studiano più dei coetanei e lavorano la metà.

> È il risultato centrale del thread: **il gap di istruzione favorisce le ragazze, quello
> sull'occupazione le penalizza**, e i due segni sono opposti in tutti e quattro i territori.
>
> Per la proposal conta il confronto, non il livello: se il gap occupazionale di Bagheria è vicino
> a quello siciliano, il problema è regionale e un intervento comunale può poco. Se è più largo,
> c'è qualcosa di locale su cui agire.
>
> La risposta formale sta nella sezione LPM sopra: in punti percentuali il gap di Bagheria non è
> un'anomalia (sopra Palermo, sotto Sicilia e Italia). Lo specifico locale è il **livello**
> dell'occupazione femminile, il più basso del panel — ed è quello, insieme al conteggio della
> sezione "Il gap in persone" qui sotto, che la proposal deve bersagliare.

# **Il quadrante per genere: dove si rompe la catena**
Le stesse due misure del quadro, come otto punti M/F nel piano istruzione × occupazione:
il segmento da M a F di ogni territorio mostra direzione e ampiezza dello scarto di
genere. Le fasce restano quelle dei dati (istruzione 9-24, occupazione 15-24) e vanno
dichiarate su ogni asse. Qui si esportano anche le tabelle per le figure R
(`fig05_forbice`, `fig06_quadrante`): la "forbice" usa il vantaggio educativo in punti e
lo svantaggio occupazionale **in rapporto M/F** — usare i punti anche sull'occupazione
contraddirebbe la sezione sulle scale, dove Bagheria in punti non è un'anomalia.

In [16]:
quadrante = (giovani[giovani["anno"].eq(anno_comune) & giovani["genere"].isin(["M", "F"])]
    [["territorio", "nome_territorio", "genere", "tasso_occupazione"]]
    .merge(titoli[titoli["anno"].eq(anno_comune) & titoli["genere"].isin(["M", "F"])]
           [["territorio", "genere", "almeno_diploma_%"]], on=["territorio", "genere"])
    .assign(anno=anno_comune))
quadrante.to_csv(PROCESSED / "genere_quadrante.csv", index=False)

rapporti = ci_gap[ci_gap["anno"].eq(anno_comune)].set_index("nome_territorio")["rapporto_M_F"]
forbice = pd.DataFrame({
    "vantaggio_istruzione_F_pp": quadro["almeno diploma F"] - quadro["almeno diploma M"],
    "rapporto_M_F_occupazione": rapporti,
    "tasso_occupazione_F": quadro["occupazione F"],
    "tasso_occupazione_M": quadro["occupazione M"],
    "almeno_diploma_F": quadro["almeno diploma F"],
    "almeno_diploma_M": quadro["almeno diploma M"],
}).round(2).assign(anno=anno_comune)
forbice.index.name = "nome_territorio"
forbice.to_csv(PROCESSED / "genere_forbice.csv")

print(f"Anno {anno_comune} — istruzione 9-24 (almeno diploma), occupazione 15-24")
print(quadrante.pivot_table(index="nome_territorio", columns="genere",
                            values=["almeno_diploma_%", "tasso_occupazione"]).reindex(ORDINE).round(1).to_string())
forbice[["vantaggio_istruzione_F_pp", "rapporto_M_F_occupazione"]]

Anno 2024 — istruzione 9-24 (almeno diploma), occupazione 15-24
                almeno_diploma_%       tasso_occupazione      
genere                         F     M                 F     M
nome_territorio                                               
Bagheria                    33.4  29.2               8.2  16.5
Palermo                     30.5  28.7               9.6  16.5
Sicilia                     33.1  30.4              10.4  20.3
Italia                      34.5  32.3              17.3  26.9


,vantaggio_istruzione_F_pp,rapporto_M_F_occupazione
nome_territorio,,
Bagheria,4.2,2.01
Italia,2.2,1.56
Palermo,1.8,1.72
Sicilia,2.7,1.95


**📌 Risultato chiave** — Nel piano istruzione × occupazione tutti i segmenti M→F puntano nella stessa direzione — più istruite, meno occupate — ma quello di Bagheria è il più ripido: **+4.2 punti di vantaggio educativo contro un rapporto occupazionale M/F di 2.01**, l'estremo femminile più basso del piano (8.2%). La forbice fra le due classifiche è la forma compatta del paradosso, ed è la coppia esportata per `fig05`/`fig06`.

> Bagheria è prima in entrambe le classifiche sbagliate: il vantaggio educativo femminile
> più ampio del panel e la conversione in lavoro più sbilanciata. Le due tabelle esportate
> (`genere_quadrante.csv`, `genere_forbice.csv`) alimentano le figure senza ricalcoli in R.

# **Il gap in persone**
Per il template della proposal (target e KPI) i tassi vanno tradotti in teste: quante
ragazze 15-24 occupate in più servirebbero a Bagheria sotto tre tassi-obiettivo. Calcolo
sui conteggi non arrotondati; la colonna "media 2022-2024" è la versione robusta alla
scelta dell'anno. Sono stock annui, non cumulati: "+40" significa 40 occupate in più
nell'anno di riferimento, a parità di popolazione.

In [17]:
def occupate_in_piu(anni: list) -> pd.Series:
    """Quante ragazze 15-24 occupate in più a Bagheria sotto ciascun tasso-obiettivo."""
    d = (giovani[giovani["genere"].isin(["M", "F"]) & giovani["anno"].isin(anni)]
         .groupby(["nome_territorio", "genere"])[["occupati", "popolazione"]].sum())
    p = d["occupati"] / d["popolazione"]
    pop_f = d.loc[("Bagheria", "F"), "popolazione"] / len(anni)  # popolazione media annua
    base = p[("Bagheria", "F")]
    return pd.Series({
        "parità con i coetanei maschi di Bagheria": int(round(pop_f * (p[("Bagheria", "M")] - base))),
        "tasso femminile di Palermo": int(round(pop_f * (p[("Palermo", "F")] - base))),
        "tasso femminile dell'Italia": int(round(pop_f * (p[("Italia", "F")] - base))),
    })


persone = pd.DataFrame({"occupate in più (2024)": occupate_in_piu([2024]),
                        "occupate in più (media 2022-2024)": occupate_in_piu([2022, 2023, 2024])})
persone.index.name = "scenario"
persone.to_csv(PROCESSED / "genere_gap_persone.csv")

base_2024 = giovani.query("nome_territorio == 'Bagheria' and genere == 'F' and anno == 2024")
# La base va anche su disco: fig09 traduce gli scenari in persone e non la ricalcola.
pd.DataFrame([{"anno": 2024,
               "popolazione_F_15_24": int(base_2024["popolazione"].item()),
               "occupate_F_15_24": int(base_2024["occupati"].item())}]).to_csv(
    PROCESSED / "genere_base_persone.csv", index=False)
print(f"Base 2024: {base_2024['popolazione'].item():.0f} ragazze 15-24, "
      f"{base_2024['occupati'].item():.0f} occupate "
      f"({base_2024['occupati'].item() / base_2024['popolazione'].item():.1%})")
persone

Base 2024: 2882 ragazze 15-24, 236 occupate (8.2%)


,occupate in più (2024),occupate in più (media 2022-2024)
scenario,,
parità con i coetanei maschi di Bagheria,239,221
tasso femminile di Palermo,40,34
tasso femminile dell'Italia,262,255


**📌 Risultato chiave** — Base 2024: **2.882 ragazze 15-24 a Bagheria, 236 occupate (8.2%)**. Allinearsi al tasso femminile di Palermo vale **+40 occupate** (KPI realistico a 2-3 anni); la parità con i coetanei o il tasso nazionale valgono **+239 / +262**, cioè più che raddoppiare le occupate attuali — misura del problema, non obiettivo.

> La scala dell'intervento, in persone:
>
> - allinearsi al tasso femminile di **Palermo** vale **+40 occupate**: è l'obiettivo che
>   un intervento comunale può realisticamente rivendicare come KPI a 2-3 anni;
> - la **parità con i coetanei** o il **tasso femminile nazionale** valgono +239 e +262:
>   più che raddoppiare le 236 occupate attuali. È la misura del problema, non un KPI.
>
> Per il template della proposal: l'evidenza è questa tabella (più i CI della sezione
> sugli intervalli), il target sono le ~2.900 ragazze 15-24 di Bagheria, il KPI sono i
> punti di tasso chiusi verso il benchmark scelto — e ogni cifra si rigenera da questa
> cella a ogni aggiornamento dei dati.

# **Il KPI si può misurare? Potenza statistica e finestre di lettura**
"+40 occupate" equivale a +1.4 punti sul tasso femminile. Prima di scriverlo come KPI va
chiesto se una variazione simile sarebbe **distinguibile dal rumore** nella fonte che
dovrebbe misurarla. Convenzione coerente con la sezione sugli intervalli: i conteggi
comunali trattati come binomiali — un limite inferiore dell'incertezza vera, quindi MDE
e potenze qui sotto sono **ottimisti**. Test a due proporzioni indipendenti, α=0.05
bilaterale, potenza obiettivo 80%; le finestre pooled contano più volte le stesse persone
(stesso caveat del pooled nella sezione LPM). La ritenzione di coorte è invece un
conteggio di registro, non una stima campionaria: lì il metro non è l'errore binomiale ma
la variabilità osservata anno su anno, riportata in fondo.

In [18]:
import numpy as np
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

analisi_potenza = NormalIndPower()


def mde_pp(p0, n, potenza_obiettivo=0.80, alpha=0.05):
    """Variazione minima (pp) rilevabile all'80% confrontando due proporzioni su n persone."""
    h = analisi_potenza.solve_power(effect_size=None, nobs1=n, alpha=alpha,
                                    power=potenza_obiettivo, ratio=1.0, alternative="two-sided")
    phi0 = 2 * np.arcsin(np.sqrt(p0))
    return 100 * (np.sin((phi0 + h) / 2) ** 2 - p0)


def potenza_pct(p0, p1, n, alpha=0.05):
    h = abs(proportion_effectsize(p1, p0))
    return 100 * analisi_potenza.power(effect_size=h, nobs1=n, alpha=alpha,
                                       ratio=1.0, alternative="two-sided")


base = giovani.query("territorio == @BAGHERIA and genere == 'F' and anno == 2024")
n_f = base["popolazione"].item()
p_occ = base["occupati"].item() / n_f
p_occ_obiettivo = (giovani.query("territorio == @PALERMO and genere == 'F' and anno == 2024")
                   .pipe(lambda d: d["occupati"].item() / d["popolazione"].item()))
q_cas = casalinghe.query("territorio == @BAGHERIA and genere == 'F' and anno == 2024")["casalinghe_o_i_%"].item() / 100
q_cas_obiettivo = casalinghe.query("territorio == @PALERMO and genere == 'F' and anno == 2024")["casalinghe_o_i_%"].item() / 100

righe = []
for kpi, p0, p1 in [("tasso di occupazione F 15-24 (obiettivo: Palermo)", p_occ, p_occ_obiettivo),
                    ("quota casalinghe F 15-24 (obiettivo: Palermo)", q_cas, q_cas_obiettivo)]:
    for finestra in (1, 2, 3):
        righe.append({"KPI": kpi, "attuale (%)": 100 * p0, "obiettivo (%)": 100 * p1,
                      "delta da rilevare (pp)": 100 * (p1 - p0), "anni pooled per lato": finestra,
                      "MDE 80% (pp)": mde_pp(p0, finestra * n_f),
                      "potenza per il delta (%)": potenza_pct(p0, p1, finestra * n_f)})
mde = pd.DataFrame(righe).round({"attuale (%)": 1, "obiettivo (%)": 1, "delta da rilevare (pp)": 2,
                                 "MDE 80% (pp)": 2, "potenza per il delta (%)": 0})
mde.to_csv(PROCESSED / "genere_mde.csv", index=False)

# Ritenzione (registro): variabilità annua osservata della coorte femminile 25-29.
eta_reg = (popolazione[popolazione["territorio"].eq(BAGHERIA)
                       & popolazione["genere"].eq("F")
                       & popolazione["cittadinanza"].eq("TOTAL")
                       & popolazione["eta_anni"].notna()]
           .assign(eta=lambda d: d["eta_anni"].astype(int))
           .groupby(["anno", "eta"])["valore"].sum())
rit_annua = {f"{a}->{a+1}": round(100 * eta_reg.loc[a + 1].loc[26:30].sum()
                                  / eta_reg.loc[a].loc[25:29].sum(), 1)
             for a in (2021, 2022, 2023)}
print("Ritenzione annua osservata, coorte F 25-29 di Bagheria (registro):", rit_annua)
mde

Ritenzione annua osservata, coorte F 25-29 di Bagheria (registro): {'2021->2022': np.float64(99.2), '2022->2023': np.float64(98.8), '2023->2024': np.float64(99.0)}


,KPI,attuale (%),obiettivo (%),delta da rilevare (pp),anni pooled per lato,MDE 80% (pp),potenza per il delta (%)
0,tasso di occupazione F 15-24 (obiettivo: Palermo),8.2,9.6,1.4,1,2.14,46.0
1,tasso di occupazione F 15-24 (obiettivo: Palermo),8.2,9.6,1.4,2,1.49,75.0
2,tasso di occupazione F 15-24 (obiettivo: Palermo),8.2,9.6,1.4,3,1.21,90.0
3,quota casalinghe F 15-24 (obiettivo: Palermo),13.4,11.3,-2.1,1,2.61,68.0
4,quota casalinghe F 15-24 (obiettivo: Palermo),13.4,11.3,-2.1,2,1.83,93.0
5,quota casalinghe F 15-24 (obiettivo: Palermo),13.4,11.3,-2.1,3,1.48,99.0


**📌 Risultato chiave** — Con la lettura annuale il KPI "+40 occupate" **non è verificabile**: il delta da rilevare è +1.40 pp contro un MDE dell'80% di **2.14 pp** (potenza: 46%, una moneta). Su finestre **pooled di 3 anni** la potenza sale al **90%** (MDE 1.21 pp). Per le casalinghe (-2.1 pp verso Palermo): 68% annuale, ~99% su tre anni. Il KPI primario si dichiara quindi con la sua finestra: **valutazione su trienni pooled**, lettura annuale solo per gli indicatori di processo.

> Tre conseguenze operative per la proposal:
>
> 1. il tasso comunale resta il KPI primario ma **si legge su finestre triennali**
>    (2022-2024 contro 2025-2027), non anno su anno — la banda del gap (±1.5-1.7 pp) lo
>    diceva già, qui c'è il numero: potenza 46% contro 90%;
> 2. serve almeno un **indicatore di processo** a lettura annuale (utenza dei servizi per
>    età e genere, ingressi nei percorsi), che oggi nessuno rileva — il che salda questa
>    sezione alla proposta del presidio di misurazione;
> 3. la ritenzione di coorte si monitora contro la variabilità osservata: le tre letture
>    annue della coorte F 25-29 (99.2, 98.8, 99.0) stanno in mezzo punto — il segnale,
>    una perdita netta di ~1% l'anno, è stabile; ogni scarto va letto contro quel mezzo
>    punto, non contro soglie da manuale.
>
> I numeri sono ottimisti per costruzione (binomiale su stime registro+campione): la
> finestra vera necessaria è semmai più lunga, non più corta.

# **Chi se ne va? Ritenzione di coorte per genere, 2021-2024**
La fuga di talenti del bando, guardata per genere. Dalle età singole si segue ogni
coorte: chi aveva `a` anni nel 2021 ne ha `a+3` nel 2024, e il rapporto fra i due stock
è la ritenzione netta. A queste età la mortalità è trascurabile, quindi lo scarto da 100
è migrazione netta — impastata però con l'aggiustamento post-censuario delle stime, che
non è separabile: si leggono i pattern contro il benchmark nazionale, non i decimali.
"Netta" significa che conta anche chi arriva, non solo chi parte.

In [19]:
singole = (popolazione[
        popolazione["territorio"].isin(CONFRONTO)
        & popolazione["cittadinanza"].eq("TOTAL")
        & popolazione["genere"].isin(["M", "F"])
        & popolazione["eta_anni"].notna()]
    .assign(eta=lambda d: d["eta_anni"].astype(int)))

p21 = singole[singole["anno"].eq(2021)]
p24 = singole[singole["anno"].eq(2024)]

COORTI = {"15-19 nel 2021": (15, 19), "20-24 nel 2021": (20, 24), "25-29 nel 2021": (25, 29)}
righe = []
for etichetta, (a0, a1) in COORTI.items():
    base = p21[p21["eta"].between(a0, a1)].groupby(["territorio", "genere"])["valore"].sum()
    dopo = p24[p24["eta"].between(a0 + 3, a1 + 3)].groupby(["territorio", "genere"])["valore"].sum()
    parziale = (100 * dopo / base).rename("ritenzione_%").reset_index()
    parziale["coorte"] = etichetta
    righe.append(parziale)
coorti = pd.concat(righe, ignore_index=True)
coorti["nome_territorio"] = coorti["territorio"].map(NOMI)
coorti["ritenzione_%"] = coorti["ritenzione_%"].round(1)
coorti.to_csv(PROCESSED / "genere_coorti.csv", index=False)

(coorti.pivot_table(index="coorte", columns=["nome_territorio", "genere"], values="ritenzione_%")
    .reindex(columns=pd.MultiIndex.from_product([ORDINE, ["F", "M"]])))

Bagheria        Palermo        Sicilia        Italia       
                      F      M       F      M       F      M      F      M
coorte                                                                    
15-19 nel 2021    100.4   98.5   100.8  101.1   100.7  103.0  101.8  104.8
20-24 nel 2021    102.0   99.7    99.8   97.7    99.2   98.3  102.7  104.1
25-29 nel 2021     96.3  101.2    98.8   96.9    97.6   97.6  103.0  103.8

**📌 Risultato chiave** — Ritenzione di coorte 2021-2024: Bagheria sta **sotto l'Italia in ogni coorte e per entrambi i generi**. La firma di genere sta nel *quando*: i ragazzi si perdono presto (15-19: 98.5 contro 104.8), le ragazze **dopo i 25** (25-29: 96.3 contro 103.0, la cella peggiore del comune) — proprio quando il vantaggio formativo dovrebbe convertirsi in lavoro.

> Due pattern, letti contro il benchmark nazionale (che sta sopra 100 ovunque grazie
> all'immigrazione):
>
> 1. **Bagheria non raggiunge il livello nazionale in nessuna cella**: ogni coorte, di
>    entrambi i generi, trattiene meno giovani di quanto faccia l'Italia. Il drenaggio
>    riguarda tutti.
> 2. La firma di genere sta nel **quando**: i ragazzi si perdono presto (coorte 15-19
>    nel 2021: 98.5 contro il 104.8 nazionale), le ragazze **dopo i 25** — la coorte
>    25-29 femminile è la peggiore di Bagheria (96.3 contro 103.0, quasi 7 punti sotto).
>    È l'età in cui il percorso formativo finisce: coerente con la decomposizione per
>    stato, le ragazze restano finché studiano e il territorio ne perde una quota
>    proprio quando il vantaggio educativo dovrebbe convertirsi in lavoro.
>
> La misura è netta e su una finestra di tre anni: non distingue chi parte da chi
> arriva, né dice dove vanno. Per i flussi origine-destinazione servono altre fonti —
> il thread mobilità è il posto naturale dove cercarle.

# **La ritenzione per età: quando esattamente, e chi resta**
Le coorti quinquennali dicono "prima i ragazzi, dopo i 25 le ragazze"; le età singole
permettono di vedere *dove* la curva si piega. Conteggi comunali per età di ~250-340
persone: il profilo usa una media mobile su tre età (somme di conteggi, poi rapporto) e
si leggono i pattern, non i decimali. In coda, il conto del gruppo che resta ma è fuori
da lavoro, studio e ricerca — quello che `idee/04` chiama "il 19% invisibile" — scisso
per genere.

In [20]:
r21 = singole[singole["anno"].eq(2021)].groupby(["territorio", "genere", "eta"])["valore"].sum()
r24 = singole[singole["anno"].eq(2024)].groupby(["territorio", "genere", "eta"])["valore"].sum()

righe = []
for territorio in CONFRONTO:
    for genere in ("F", "M"):
        for a in range(13, 32):
            n0, n3 = r21.get((territorio, genere, a)), r24.get((territorio, genere, a + 3))
            righe.append({"territorio": territorio, "nome_territorio": NOMI[territorio],
                          "genere": genere, "eta_2021": a, "eta_2024": a + 3,
                          "n_2021": n0, "n_2024": n3,
                          "ritenzione_pct": round(100 * n3 / n0, 1)})
profilo = pd.DataFrame(righe)
for _, gruppo in profilo.groupby(["territorio", "genere"]):
    liscio = (100 * gruppo["n_2024"].rolling(3, center=True).sum()
              / gruppo["n_2021"].rolling(3, center=True).sum())
    profilo.loc[gruppo.index, "ritenzione_rolling3_pct"] = liscio.round(1)
profilo.to_csv(PROCESSED / "genere_ritenzione_eta.csv", index=False)

print("Ritenzione 2021→2024 per età (media mobile 3 età), Bagheria e Italia:")
print(profilo[profilo["territorio"].isin([BAGHERIA, ITALIA])]
      .pivot_table(index="eta_2021", columns=["nome_territorio", "genere"],
                   values="ritenzione_rolling3_pct")
      .loc[14:30, [("Bagheria", "F"), ("Bagheria", "M"), ("Italia", "F"), ("Italia", "M")]]
      .to_string())

# Il gruppo fuori da lavoro, studio e ricerca (2024), per genere: dimensioni e composizione.
fuori = largo.loc[(BAGHERIA, 2024)]
print("\nBagheria 2024, 15-24 — fuori da lavoro, studio e ricerca attiva:")
for genere in ("F", "M"):
    r = fuori.loc[genere]
    tot = r[["4", "24", "7"]].sum()
    print(f"  {genere}: {tot:.0f} persone = {100 * tot / r['99']:.1f}% della fascia "
          f"(casalinghe/i {r['4']:.0f}, altra condizione {r['7']:.0f}, pensione {r['24']:.0f}; "
          f"in cerca, per confronto: {r['12']:.0f})")
tot_f = fuori.loc["F", ["4", "24", "7"]].sum()
tot_m = fuori.loc["M", ["4", "24", "7"]].sum()
print(f"  quota femminile del gruppo: {100 * tot_f / (tot_f + tot_m):.1f}%")

Ritenzione 2021→2024 per età (media mobile 3 età), Bagheria e Italia:
nome_territorio Bagheria        Italia       
genere                 F      M      F      M
eta_2021                                     
14                  99.8   99.4  101.2  102.8
15                 100.6   98.8  101.2  103.8
16                  99.5   99.2  101.4  104.7
17                 100.5   99.4  101.8  105.2
18                 100.3   98.3  102.1  105.2
19                 101.0   97.8  102.4  104.6
20                 103.3   99.2  102.4  104.3
21                 101.5  100.7  102.6  104.2
22                 103.3  100.4  102.8  104.2
23                 100.5   98.7  102.9  104.1
24                  99.6   97.5  102.9  104.0
25                  97.1   97.6  102.9  103.9
26                  96.9  100.0  103.0  103.9
27                  97.6  102.7  103.0  103.9
28                  96.7  103.7  103.0  103.8
29                  97.9  101.1  102.9  103.6
30                  98.4  100.0  102.8  103.4

Bagheria 

**📌 Risultato chiave** — Il profilo per età localizza le due rotture: le ragazze tengono (≈100-103) **fino ai 23 anni del 2021** e cedono da lì in poi — 96.7-97.9 sulle età 25-29 contro ~103 dell'Italia; i ragazzi stanno sotto la pari già a 17-19 e di nuovo a 23-24, ma **recuperano dopo i 26** (102-104: rientri netti), le ragazze no. E il "19% invisibile" di `idee/04` **non è un gruppo femminile**: 573 ragazze e 549 ragazzi (51% F). Femminile è l'etichetta — 387 casalinghe contro 50 — non la dimensione.

> Tre precisazioni che le idee scritte finora non contenevano:
>
> 1. la finestra utile per le ragazze è **22-25 anni**: prima la curva è sopra la pari,
>    dopo la perdita è già avvenuta. "Dopo i 25" era giusto ma generico — il cedimento
>    parte fra le età 24 e 25 (del 2021) e riguarda quindi le 27-32enni del 2024;
> 2. per i ragazzi il profilo è **a due onde con rientro**: uscita precoce (17-19),
>    seconda uscita a 23-24, saldo positivo dopo i 26 — pendolarismo lungo di ritorno o
>    rientri, i dati non distinguono;
> 3. il gruppo fuori-da-tutto è di **entrambi i generi in parti quasi uguali**; ciò che
>    è di genere è la *composizione*: le ragazze hanno un ruolo dichiarato (casalinga),
>    i ragazzi un residuo ("altra condizione"). `idee/doppia_fuga.md` e `idee/04` vanno
>    corrette su questo punto: un intervento "per le invisibili" che ignorasse i 549
>    ragazzi sbaglierebbe platea di metà.

# **La finestra 22-25 regge? Le transizioni una per una**

Il profilo qui sopra è un salto pooled 2021→2024: tre transizioni annuali compresse in
una. Prima di appendere una policy alla finestra 22-25 va dichiarato quanto della
finestra sta nel pooled e quanto si rivedrebbe in un anno qualsiasi.

In [21]:
# Le tre transizioni annuali dentro il salto 2021→2024: rapporti N(a+1, t+1) / N(a, t)
# dalle stesse età singole del profilo sopra. Dichiarano quanto della finestra 22-25 sta
# nel pooled triennale e quanto reggerebbe su un anno solo.
conteggi_eta = {anno: singole[singole["anno"].eq(anno)]
                  .groupby(["territorio", "genere", "eta"])["valore"].sum()
            for anno in (2021, 2022, 2023, 2024)}

righe = []
for t0 in (2021, 2022, 2023):
    for territorio in CONFRONTO:
        for genere in ("F", "M"):
            for a in range(15, 31):
                base = conteggi_eta[t0].get((territorio, genere, a))
                dopo = conteggi_eta[t0 + 1].get((territorio, genere, a + 1))
                righe.append({"territorio": territorio, "nome_territorio": NOMI[territorio],
                              "genere": genere, "transizione": f"{t0}-{t0 + 1}",
                              "eta_iniziale": a, "n_iniziale": round(base),
                              "rapporto_pct": round(100 * dopo / base, 1)})
transizioni = pd.DataFrame(righe)
transizioni.to_csv(PROCESSED / "genere_ritenzione_transizioni.csv", index=False)

finestra = transizioni[transizioni["territorio"].isin([BAGHERIA, ITALIA])
                       & transizioni["eta_iniziale"].between(22, 25)]
print("Rapporti annuali per età iniziale 22-25 (Italia come metro del rumore):")
print(finestra.pivot_table(index=["genere", "eta_iniziale"],
                           columns=["nome_territorio", "transizione"], values="rapporto_pct")
      [[("Bagheria", c) for c in ("2021-2022", "2022-2023", "2023-2024")]
       + [("Italia", c) for c in ("2021-2022", "2022-2023", "2023-2024")]].to_string())

# Ordine di grandezza del solo rumore di conteggio su una cella di Bagheria: se ogni
# residente restasse con probabilità r ~ 0.98 su n ~ 280, sd(rapporto) ~ sqrt(r(1-r)/n)
# ~ 0.8 pp. L'escursione osservata è più larga: le transizioni singole contengono flussi
# reali anno per anno, non solo rumore — ma nessuna da sola fa una finestra.
import numpy as np
n_tipico = transizioni.query("territorio == @BAGHERIA and 22 <= eta_iniziale <= 25")["n_iniziale"].median()
sd_conteggio = 100 * np.sqrt(0.98 * 0.02 / n_tipico)
escursione = (transizioni[transizioni["territorio"].eq(BAGHERIA)
                          & transizioni["eta_iniziale"].between(22, 25)]
              .groupby(["genere", "eta_iniziale"])["rapporto_pct"].agg(["min", "max"])
              .assign(escursione=lambda d: (d["max"] - d["min"]).round(1)))
print(f"\nBagheria, escursione min-max fra le tre transizioni (n mediano {n_tipico:.0f}, "
      f"solo conteggio darebbe ~±{sd_conteggio:.1f} pp):")
print(escursione.to_string())

sotto_cento = transizioni.query("territorio == @BAGHERIA and genere == 'F' "
                                "and 22 <= eta_iniziale <= 25 and rapporto_pct < 100")
print(f"\nCelle F 22-25 sotto quota 100: {len(sotto_cento)} su 12, "
      f"di cui {len(sotto_cento.query('transizione == \"2023-2024\"'))} nella transizione 2023-2024")

Rapporti annuali per età iniziale 22-25 (Italia come metro del rumore):
nome_territorio      Bagheria                        Italia                    
transizione         2021-2022 2022-2023 2023-2024 2021-2022 2022-2023 2023-2024
genere eta_iniziale                                                            
F      22               101.0     101.0     100.4     101.3     100.7     100.7
       23               103.9      99.0     102.0     101.3     100.9     100.8
       24               100.7     104.5      96.5     101.2     100.8     100.8
       25                97.8      99.0      96.8     101.3     100.7     100.7
M      22               100.0     101.3     100.6     101.2     101.3     101.5
       23                99.0     100.3     101.3     101.2     101.4     101.8
       24                97.8     100.0      99.3     101.2     101.4     101.5
       25                99.3      97.4      99.0     101.2     101.2     101.5

Bagheria, escursione min-max fra le tre transiz

**📌 Risultato chiave** — La finestra 22-25 è una lettura **pooled, e va usata come
tale**. Nelle tre transizioni annuali il segno femminile c'è ma balla: 5 celle su 12
sotto quota 100 sulle età 22-25, con escursioni fino a **8 pp sulla stessa età** (le
24enni fanno 100.7, 104.5, 96.5) contro il ~±0.8 pp che darebbe il solo rumore di
conteggio su celle da ~290 ragazze — e contro l'Italia, che nelle stesse celle femminili
non esce mai dall'intervallo 100.7-101.3. La transizione più negativa è il 2023→2024
(96.5 e 96.8 sulle età 24 e 25). Conseguenza operativa, identica a quella dei KPI: il
claim si formula sul triennio 2021→2024 (e sulle scale quinquennale e decennale, dove
ritorna), la transizione singola serve da controllo di direzione e non da titolo. Un
anno solo di dati post-intervento non potrà dire nulla sulla ritenzione: anche questa
lettura entra nella finestra triennale della sezione «Il KPI si può misurare?».

# **Il bilancio dei giovani: la platea del 2035 è già nata**

La ritenzione dice chi se ne va; il registro anagrafico dice anche chi arriverà. Chi
avrà 15-24 anni nel 2029 o nel 2034 oggi ha 10-19 o 5-14 anni ed è già contabile per
età singola: il futuro demografico del target, a migrazione ferma, è un'operazione di
conteggio. Ogni intervento della proposal agirà su questa platea, non su quella del
2024.

In [22]:
# Chi avrà 15-24 anni nel 2029 o nel 2034 è già nato e già residente: la platea futura del
# target si conta, non si prevede. Contabilità a saldo migratorio zero, con mortalità
# trascurabile a queste età (ordine di 0.1 per mille l'anno): per Bagheria, dove la
# ritenzione osservata sta sotto quota 100, il conteggio è quindi un tetto; per l'Italia,
# che sulle stesse età guadagna residenti, è semmai un pavimento.
p2024 = conteggi_eta[2024]

def platea(territorio, genere, a0, a1):
    return sum(p2024.get((territorio, genere, a)) for a in range(a0, a1 + 1))

righe = []
for territorio in CONFRONTO:
    for genere in ("F", "M"):
        oggi = platea(territorio, genere, 15, 24)
        a_2029 = platea(territorio, genere, 10, 19)   # 10-19enni del 2024
        a_2034 = platea(territorio, genere, 5, 14)    # 5-14enni del 2024
        righe.append({"territorio": territorio, "nome_territorio": NOMI[territorio],
                      "genere": genere,
                      "platea_2024": round(oggi), "platea_2029": round(a_2029),
                      "platea_2034": round(a_2034),
                      "var_2029_pct": round(100 * (a_2029 / oggi - 1), 1),
                      "var_2034_pct": round(100 * (a_2034 / oggi - 1), 1)})
platea_15_24 = pd.DataFrame(righe)
platea_15_24.to_csv(PROCESSED / "genere_platea.csv", index=False)

# Verifica incrociata sulla tavola indipendente delle classi quinquennali: stesso
# censimento ma query, file raw e percorso di parsing diversi. Y5-9 + Y10-14 deve
# ridare la somma delle età singole, per entrambi i generi.
quinquennali = leggi("censpop_demografia_classi_long.csv")
q_5_14 = (quinquennali[quinquennali["territorio"].eq(BAGHERIA) & quinquennali["anno"].eq(2024)
                       & quinquennali["eta"].isin(["Y5-9", "Y10-14"])
                       & quinquennali["genere"].isin(["M", "F"])
                       & quinquennali["cittadinanza"].eq("TOTAL")]
          .groupby("genere")["valore"].sum())
for genere in ("F", "M"):
    scarto = abs(q_5_14[genere] - platea(BAGHERIA, genere, 5, 14))
    assert scarto < 1, f"classi vs età singole, {genere}: scarto {scarto:.1f}"

print("Platea 15-24 futura a saldo migratorio zero (residenti 2024 fatti invecchiare):")
print(platea_15_24.pivot_table(index="nome_territorio", columns="genere",
                               values=["platea_2024", "platea_2029", "platea_2034"],
                               aggfunc="first").reindex(ORDINE).to_string())
print()
for genere in ("F", "M"):
    r = platea_15_24.query("territorio == @BAGHERIA and genere == @genere").iloc[0]
    print(f"  Bagheria {genere}: {r['platea_2024']:,} oggi -> {r['platea_2029']:,} nel 2029 "
          f"({r['var_2029_pct']:+.1f}%) -> {r['platea_2034']:,} nel 2034 ({r['var_2034_pct']:+.1f}%)")
bench = platea_15_24[platea_15_24["territorio"].ne(BAGHERIA)]
div_bag = abs(platea_15_24.query("territorio == @BAGHERIA and genere == 'F'")["var_2034_pct"].item()
              - platea_15_24.query("territorio == @BAGHERIA and genere == 'M'")["var_2034_pct"].item())
print(f"\n  Benchmark al 2034: fra {bench['var_2034_pct'].min():+.1f}% e "
      f"{bench['var_2034_pct'].max():+.1f}%, con F e M vicini in ogni territorio; "
      f"a Bagheria i due generi divergono di {div_bag:.1f} punti.")

Platea 15-24 futura a saldo migratorio zero (residenti 2024 fatti invecchiare):
                platea_2024          platea_2029          platea_2034         
genere                    F        M           F        M           F        M
nome_territorio                                                               
Bagheria               2882     3022        2651     2946        2435     2846
Palermo               32672    34363       31681    33072       28950    30260
Sicilia              243292   265707      230183   247066      210834   222117
Italia              2822968  3088740     2713088  2908421     2445360  2591788

  Bagheria F: 2,882 oggi -> 2,651 nel 2029 (-8.0%) -> 2,435 nel 2034 (-15.5%)
  Bagheria M: 3,022 oggi -> 2,946 nel 2029 (-2.5%) -> 2,846 nel 2034 (-5.8%)

  Benchmark al 2034: fra -16.4% e -11.4%, con F e M vicini in ogni territorio; a Bagheria i due generi divergono di 9.7 punti.


**📌 Risultato chiave** — La platea femminile del target **si restringe più in fretta
di quella maschile, e in modo più sbilanciato che in ogni territorio di confronto**. A
saldo migratorio zero le ragazze 15-24 di Bagheria passano da **2.882 (2024) a 2.651
nel 2029 (-8.0%) e 2.435 nel 2034 (-15.5%)**; i coetanei calano del 2.5% e del 5.8%.
Nei benchmark il calo al 2034 sta fra l'11.4% e il 16.4% ma è quasi simmetrico fra i
generi; a Bagheria i due generi divergono di 9.7 punti. Due conseguenze per la
proposal. *Dimensionamento*: i KPI espressi in teste vanno riparametrati — un
intervento tarato sulle 2.882 ragazze di oggi che parta nel 2027 lavora su una platea
già in rotta verso le 2.651 del 2029; quelli in tasso restano validi. *Urgenza*: il
conteggio è un **tetto**, perché la ritenzione osservata (sezioni precedenti) sta sotto
quota 100: il 2034 reale sarà verosimilmente sotto le 2.435 contate. L'asimmetria di
genere del calo, però, non nasce dalla migrazione dei giovani: viene da più a monte, ed
è la sezione successiva.

# **L'audit della platea: la sex ratio 5-14 che sale**

Un -15.5% femminile contro un -5.8% maschile non può venire da un calo delle nascite in
generale, che colpirebbe entrambi i generi: o fra i bambini di Bagheria mancano
specificamente le femmine, o una delle due tavole mente. Prima di usare il numero,
l'audit.

In [23]:
# Da dove viene l'asimmetria della platea: la sex ratio (maschi per 100 femmine) dei
# 5-14enni. Le classi quinquennali la seguono su 2001, 2011 e 2018-2024, con le età
# singole come tavola di controllo dal 2021: il 2001 e il 2011 sono nella norma, la
# salita è tutta successiva — lo stesso decennio del "muro recente" di fig10.
cinque_14 = (quinquennali[quinquennali["eta"].isin(["Y5-9", "Y10-14"])
                          & quinquennali["genere"].isin(["M", "F"])
                          & quinquennali["cittadinanza"].eq("TOTAL")
                          & quinquennali["territorio"].isin(CONFRONTO)]
             .groupby(["anno", "territorio", "genere"])["valore"].sum().unstack("genere"))
sex_ratio = (cinque_14.assign(m_per_100f=lambda d: (100 * d["M"] / d["F"]).round(1))
             .reset_index())
sex_ratio["nome_territorio"] = sex_ratio["territorio"].map(NOMI)
sex_ratio.to_csv(PROCESSED / "genere_sex_ratio_5_14.csv", index=False)

print("Maschi per 100 femmine, età 5-14 (classi quinquennali):")
print(sex_ratio.pivot_table(index="anno", columns="nome_territorio", values="m_per_100f")
      [ORDINE].to_string())

# Controllo sulla tavola sorella (età singole, 2024) e scomposizione per cittadinanza:
# l'anomalia deve ritrovarsi identica e non essere un effetto di composizione straniera.
singole_5_14 = (popolazione[popolazione["territorio"].eq(BAGHERIA) & popolazione["anno"].eq(2024)
                            & popolazione["eta_anni"].between(5, 14) & popolazione["genere"].isin(["M", "F"])]
                .groupby(["cittadinanza", "genere"])["valore"].sum().unstack("genere"))
per_100 = (100 * singole_5_14["M"] / singole_5_14["F"]).round(1)
assert abs(per_100["TOTAL"] - sex_ratio.query("territorio == @BAGHERIA and anno == 2024")["m_per_100f"].item()) < 0.15
print(f"\nBagheria 2024 dalle età singole: {per_100['TOTAL']:.1f} totale, "
      f"{per_100['ITL']:.1f} fra gli italiani ({singole_5_14.loc['ITL'].sum():.0f} bambini) — "
      f"gli stranieri sono {singole_5_14.loc['FRGAPO'].sum():.0f} in tutto e non spostano il rapporto.")

# Compatibilità col caso: quota maschile osservata contro l'attesa data dalla quota
# italiana dello stesso anno. Le finestre 2001, 2011 e 2018 non condividono quasi nessuna
# coorte di nascita; il 2024 condivide col 2018 le nate 2010-2013. Un singolo z alto può
# essere sorte; una salita che attraversa finestre quasi disgiunte no.
print("\nScarto dalla quota maschile italiana (z della binomiale):")
for anno in (2001, 2011, 2018, 2024):
    m_b, f_b = cinque_14.loc[(anno, BAGHERIA), ["M", "F"]]
    m_i, f_i = cinque_14.loc[(anno, ITALIA), ["M", "F"]]
    n, p_oss, p_att = m_b + f_b, m_b / (m_b + f_b), m_i / (m_i + f_i)
    z = (p_oss - p_att) / (p_att * (1 - p_att) / n) ** 0.5
    print(f"  {anno}: {100 * p_oss:.1f}% maschi contro attesa {100 * p_att:.1f}%  "
          f"(n = {n:,.0f}, z = {z:+.1f})")

Maschi per 100 femmine, età 5-14 (classi quinquennali):
nome_territorio  Bagheria  Palermo  Sicilia  Italia
anno                                               
2001                103.7    105.6    105.1   105.5
2011                107.2    105.4    105.8   106.1
2018                110.5    103.7    105.5   106.1
2019                111.6    104.1    105.5   106.1
2020                113.5    104.4    105.5   106.1
2021                114.0    104.1    105.0   106.0
2022                114.3    104.3    105.1   106.0
2023                117.2    104.7    105.2   106.0
2024                116.9    104.5    105.4   106.0

Bagheria 2024 dalle età singole: 116.9 totale, 116.8 fra gli italiani (5220 bambini) — gli stranieri sono 61 in tutto e non spostano il rapporto.

Scarto dalla quota maschile italiana (z della binomiale):
  2001: 50.9% maschi contro attesa 51.3%  (n = 7,029, z = -0.7)
  2011: 51.7% maschi contro attesa 51.5%  (n = 6,058, z = +0.4)
  2018: 52.5% maschi contro attesa 51.

**📌 Risultato chiave** — Fra i bambini di 5-14 anni Bagheria conta **117 maschi ogni
100 femmine (2024)** contro il 104-106 stabile di Palermo, Sicilia e Italia — e ~106 è
anche il livello naturale alla nascita. La deviazione è **recente e progressiva**:
103.7 nel 2001 e 107.2 nel 2011 (entrambi nella norma: z -0.7 e +0.4 rispetto alla
quota maschile italiana), poi 110.5 nel 2018 (z +1.5) e 116.9 nel 2024 (**z +3.5** su
~5.300 bambini). Regge ai controlli: identica su due tavole indipendenti dello stesso
censimento (classi quinquennali ed età singole), tutta dentro la popolazione italiana
(gli stranieri sono 61 in tutto), non attribuibile al caso sull'ultima finestra. Il
decennio in cui parte è lo stesso del "muro" sul lavoro femminile di fig10. **Il
meccanismo resta aperto** — sorte accumulata sulle nascite, migrazione selettiva di
famiglie, dinamica di registro — e i due check che lo chiuderebbero sono piccoli e
dichiarati: i nati per sesso del comune (demo.istat) e la stessa serie sulle dieci
gemelle (estensione della tavola quinquennale); entrambi fetch nuovi, da decidere in
team. Fino ad allora la platea femminile del 2034 si usa col suo numero (2.435 ragazze
contate oggi), ma l'asimmetria di genere si presenta **insieme a questo audit**, mai da
sola.

# **La componente straniera: il ricambio che non c'è**

Nel bilancio demografico italiano le uscite dei giovani sono in parte compensate dagli
ingressi di cittadini stranieri. Resta da misurare quanto questo canale esista a
Bagheria — con un vincolo noto: la componente straniera si può contare (`SETA_1`
incrocia cittadinanza, età singola e genere), ma non se ne può misurare l'occupazione
(nelle tavole lavoro comunali la cittadinanza è solo `TOTAL`).

In [24]:
# Il canale di ricambio che altrove attenua il calo: la popolazione straniera nel 15-34.
# Le età singole partono dal 2021, quindi la serie è 2021-2024; la scomposizione
# ITL + FRGAPO deve ricostruire il TOTAL riga per riga (stessa regola dei totali SDMX).
quindici_34 = popolazione[popolazione["territorio"].isin(CONFRONTO)
                          & popolazione["anno"].between(2021, 2024)
                          & popolazione["genere"].isin(["M", "F"])
                          & popolazione["eta_anni"].between(15, 34)]
per_citt = (quindici_34[quindici_34["cittadinanza"].isin(["ITL", "FRGAPO"])]
            .groupby(["territorio", "anno", "genere", "cittadinanza"])["valore"].sum()
            .unstack("cittadinanza"))
totale = quindici_34[quindici_34["cittadinanza"].eq("TOTAL")].groupby(
    ["territorio", "anno", "genere"])["valore"].sum()
assert (per_citt.sum(axis=1) - totale).abs().max() < 0.001, "ITL + FRGAPO non ricostruisce TOTAL"

stranieri = (per_citt.assign(totale=totale,
                             quota_stranieri_pct=lambda d: (100 * d["FRGAPO"] / d["totale"]).round(1))
             .reset_index())
stranieri["nome_territorio"] = stranieri["territorio"].map(NOMI)
stranieri.to_csv(PROCESSED / "genere_stranieri.csv", index=False)

print("Quota di cittadini stranieri nella popolazione 15-34 (%, M+F):")
insieme = (stranieri.groupby(["nome_territorio", "anno"])[["FRGAPO", "totale"]].sum()
           .assign(quota=lambda d: (100 * d["FRGAPO"] / d["totale"]).round(1)))
print(insieme["quota"].unstack("anno")[[2021, 2022, 2023, 2024]].reindex(ORDINE).to_string())

print("\nBagheria, variazione 2021 -> 2024 del 15-34 per cittadinanza e genere:")
bag = stranieri[stranieri["territorio"].eq(BAGHERIA)].set_index(["anno", "genere"])
for genere in ("F", "M"):
    d_itl = bag.loc[(2024, genere), "ITL"] - bag.loc[(2021, genere), "ITL"]
    d_frg = bag.loc[(2024, genere), "FRGAPO"] - bag.loc[(2021, genere), "FRGAPO"]
    print(f"  {genere}: italiani {d_itl:+,.0f}, stranieri {d_frg:+,.0f}")
tot_itl = bag.groupby("anno")["ITL"].sum()
tot_frg = bag.groupby("anno")["FRGAPO"].sum()
print(f"  insieme: italiani {tot_itl[2024] - tot_itl[2021]:+,.0f}, "
      f"stranieri {tot_frg[2024] - tot_frg[2021]:+,.0f} "
      f"({tot_frg[2024]:,.0f} stranieri 15-34 in tutto al 2024)")

Quota di cittadini stranieri nella popolazione 15-34 (%, M+F):
anno             2021  2022  2023  2024
nome_territorio                        
Bagheria          1.3   1.5   1.6   1.6
Palermo           4.4   4.4   4.8   5.1
Sicilia           5.4   5.6   5.9   6.4
Italia           11.7  11.8  12.0  12.4

Bagheria, variazione 2021 -> 2024 del 15-34 per cittadinanza e genere:
  F: italiani -219, stranieri +11
  M: italiani -131, stranieri +26
  insieme: italiani -350, stranieri +37 (195 stranieri 15-34 in tutto al 2024)


**📌 Risultato chiave** — Il canale di ricambio a Bagheria **non esiste**: gli
stranieri sono l'**1.6% del 15-34** (195 persone al 2024) contro il 5.1% di Palermo, il
6.4% della Sicilia e il 12.4% dell'Italia — un ordine di grandezza sotto il benchmark
nazionale. Fra 2021 e 2024 il 15-34 perde **350 residenti italiani** (di cui **219
ragazze**, il 63%, da fonte demografica indipendente dalla tavola lavoro) e ne
guadagna **37 stranieri**: la compensazione copre un decimo della perdita. La fuga di
Bagheria è quindi al netto di niente — dove Palermo e l'Italia sostituiscono in parte
chi parte, Bagheria somma le uscite e basta. Per la proposal è un vincolo di contesto
più che un target (le leve comunali sull'attrattività internazionale sono deboli), ma
tocca il punto del bando sulla migrazione: ogni ragionamento sui flussi giovanili di
Bagheria è un ragionamento su chi esce, perché quasi nessuno entra.

# **Contesto storico 2011**
Gli indicatori di genere di 8milaCensus. Sono calcolati sui **15 anni e più**, non sui giovani:
servono come sfondo, non come termine di paragone con le serie qui sopra.

In [25]:
GENERE_2011 = ["L1", "L2", "L6", "L7", "L10", "L11", "I1"]

storico = (ottomila[
        ottomila["territorio"].isin(CONFRONTO)
        & ottomila["anno"].eq(2011)
        & ottomila["indicatore"].isin(GENERE_2011)]
    .merge(indicatori[["indicatore", "nome_indicatore"]], on="indicatore")
    .pivot(index=["indicatore", "nome_indicatore"], columns="territorio", values="valore"))
storico = storico[CONFRONTO].rename(columns={c: NOMI[c] for c in CONFRONTO}).round(1)
storico

,territorio,Bagheria,Palermo,Sicilia,Italia
indicatore,nome_indicatore,,,,
I1,Differenziali di genere per l'istruzione superiore,98.9,102.5,100.7,101.5
L1,Partecipazione al mercato del lavoro maschile,56.7,57.8,57.5,60.7
L10,Tasso di occupazione maschile,43.1,45.0,46.9,54.8
L11,Tasso di occupazione femminile,18.1,25.5,24.0,36.1
L2,Partecipazione al mercato del lavoro femminile,28.7,35.9,33.0,41.8
L6,Tasso di disoccupazione maschile,24.1,22.1,18.5,9.8
L7,Tasso di disoccupazione femminile,36.9,29.1,27.1,13.6


**📌 Risultato chiave** — Sfondo di lungo periodo (2011, 15+, fonte diversa): tasso di occupazione femminile a Bagheria **18.1% contro 36.1% nazionale**, la metà, e disoccupazione femminile 36.9% contro 13.6%. Lo svantaggio femminile locale non è un fatto recente. Non confrontabile con le serie 2018-2024: fascia e fonte diverse, citare sempre con entrambe.

> `L10`/`L11` danno il gap occupazionale complessivo del 2011, `L1`/`L2` quello sulla
> partecipazione. Non sono confrontabili con le serie 2018-2024 di questo notebook: fascia
> diversa (15+ contro 15-24) e fonte diversa. Vanno citati con anno e fascia, sempre.

# **Bagheria nella distribuzione siciliana**
I quattro territori di confronto dicono se Bagheria è sopra o sotto la media, non dove si
colloca fra i comuni. L'unica fonte che copre **tutti i comuni siciliani** con un dato di
genere è 8milaCensus: indicatore `L11`, tasso di occupazione femminile, **2011, 15 anni e
più**. Fascia e anno diversi dalle serie di questo notebook — è un ritratto del contesto,
mai da accostare al 15-24 del censimento permanente.

Prepara la tabella per la mappa: valori uniti ai confini comunali (`pipeline/build.py`,
confini ISTAT 2026 generalizzati, EPSG:32633).

In [26]:
# I vertici arrivano già proiettati da pipeline/build.py: qui si uniscono ai valori e basta.
poligoni = pd.read_csv(PROCESSED / "comuni_sicilia_poligoni.csv",
                       dtype={"territorio": str, "nome_comune": str})
centroidi = pd.read_csv(PROCESSED / "comuni_sicilia_centroidi.csv",
                        dtype={"territorio": str, "nome_comune": str})

occ_femminile = (ottomila[
        ottomila["anno"].eq(2011)
        & ottomila["indicatore"].eq("L11")
        & ottomila["livello"].eq("1")][["territorio", "valore"]]
    .rename(columns={"valore": "occupazione_femminile_2011"}))

# Left join dai confini: un comune senza dato resta nella mappa come area vuota, non sparisce.
mappa = poligoni.merge(occ_femminile, on="territorio", how="left")
scoperti = sorted(set(poligoni["territorio"]) - set(occ_femminile["territorio"]))
mancanti = sorted(set(occ_femminile["territorio"]) - set(poligoni["territorio"]))
assert not mancanti, f"comuni con dato 2011 privi di confine: {mancanti}"
mappa.to_csv(PROCESSED / "genere_mappa_occupazione_femminile.csv", index=False)

# Posizione di Bagheria nella distribuzione dei 390 comuni.
valori = occ_femminile.set_index("territorio")["occupazione_femminile_2011"]
bagheria = valori[BAGHERIA]
percentile = 100 * (valori < bagheria).mean()
posizione = int((valori < bagheria).sum()) + 1

# Etichette della mappa: Bagheria, i 5 comuni più vicini e i due estremi regionali.
# Vicinanza = distanza euclidea fra centroidi già proiettati (EPSG:32633), in km:
# è la prossimità fisica, non l'adiacenza amministrativa (due comuni possono confinare
# ed essere lontani di centroide, e viceversa).
xy = centroidi.set_index("territorio")[["x", "y"]]
distanza_km = (xy - xy.loc[BAGHERIA]).pow(2).sum(axis=1).pow(0.5).div(1000)
vicini = distanza_km.drop(BAGHERIA).nsmallest(5)
estremi = [valori.idxmin(), valori.idxmax()]

etichette = pd.concat([
    pd.DataFrame({"territorio": [BAGHERIA], "ruolo": "Bagheria", "distanza_km": 0.0}),
    pd.DataFrame({"territorio": vicini.index, "ruolo": "vicino",
                  "distanza_km": vicini.to_numpy()}),
    pd.DataFrame({"territorio": estremi, "ruolo": ["minimo", "massimo"],
                  "distanza_km": distanza_km.reindex(estremi).to_numpy()}),
], ignore_index=True)
etichette = etichette.merge(centroidi, on="territorio", how="left")
etichette["valore"] = etichette["territorio"].map(valori)
assert etichette["valore"].notna().all(), "etichetta senza dato 2011"
etichette.to_csv(PROCESSED / "genere_mappa_etichette.csv", index=False)

print(f"comuni siciliani con dato 2011: {len(valori)}   confini disponibili: {poligoni['territorio'].nunique()}")
print("senza dato 2011:", [centroidi.set_index('territorio').loc[c, 'nome_comune'] for c in scoperti] or "nessuno")
print(f"\nBagheria: {bagheria:.1f}%  →  {posizione}° comune su {len(valori)} in ordine crescente "
      f"({percentile:.0f}° percentile)")
print(f"mediana siciliana {valori.median():.1f}%   min {valori.min():.1f}%   max {valori.max():.1f}%")
print("\nvicini di Bagheria (distanza fra centroidi, occupazione femminile 2011):")
for _, r in etichette[etichette["ruolo"].eq("vicino")].iterrows():
    print(f"  {r['nome_comune']:<22} {r['distanza_km']:5.1f} km   {r['valore']:.1f}%")
for ruolo in ("minimo", "massimo"):
    r = etichette[etichette["ruolo"].eq(ruolo)].iloc[0]
    print(f"  {ruolo:<8} regionale: {r['nome_comune']} ({r['valore']:.1f}%)")

comuni siciliani con dato 2011: 390   confini disponibili: 391
senza dato 2011: ['Misiliscemi']

Bagheria: 18.1%  →  49° comune su 390 in ordine crescente (12° percentile)
mediana siciliana 23.6%   min 13.0%   max 39.4%

vicini di Bagheria (distanza fra centroidi, occupazione femminile 2011):
  Santa Flavia             2.9 km   18.6%
  Ficarazzi                3.2 km   20.1%
  Villabate                4.7 km   16.6%
  Casteldaccia             7.9 km   20.1%
  Misilmeri                8.1 km   16.2%
  minimo   regionale: Francofonte (13.0%)
  massimo  regionale: Maniace (39.4%)


**📌 Risultato chiave** — Bagheria è nel **12° percentile siciliano** per occupazione femminile: solo 48 comuni su 390 stavano più in basso nel 2011 (49ª posizione in ordine crescente), contro una mediana regionale del 23.6%. Non è un comune medio della Sicilia, è nella coda bassa di una regione già ultima in Italia. Dato 2011, 15+, fonte 8milaCensus: contesto, non confrontabile con le serie 15-24 di questo notebook.

> Due avvertenze per l'uso in mappa. **Misiliscemi** (istituito nel 2021 staccandosi da
> Trapani) non ha un dato 2011 e resta in bianco: nel 2011 il suo territorio era dentro
> Trapani, e attribuirgli il valore trapanese sarebbe un'imputazione, non un dato.
> I **confini sono al 2026** mentre il dato è 2011: per tutti gli altri 390 comuni la
> corrispondenza è verificata dal join qui sopra (nessun comune con dato resta senza
> confine), e ISTAT non pubblica più le annate storiche su quello storage.

# **I comuni vicini: le stesse due serie, un pezzo di costa alla volta**
La mappa colloca Bagheria nella coda bassa siciliana insieme al suo vicinato, ma su un dato
del 2011 e sui 15 anni e più. Qui le due serie portanti del thread, il **gap occupazionale
15-24** (2018-2024) e la **ritenzione di coorte per età singola** (2021 a 2024), vengono
ricostruite con le stesse definizioni per i **cinque comuni geograficamente più vicini**,
selezionati sulla distanza fra centroidi nella cella della mappa.

I raw SDMX di questi comuni stanno in file separati (`censpop_lavoro_vicini`,
`censpop_popolazione_vicini`): le tavole condivise restano a quattro territori e gli altri
thread non cambiano numeri. Anche le uscite sono separate, `*_vicini.csv`, e le figure le
uniscono in lettura.

> **Attenzione alla taglia.** Sono comuni fra i 10.000 e i 28.000 abitanti: sulla fascia
> 15-24 i conteggi campionari sono piccoli e gli intervalli di confidenza larghi. Le loro
> serie vanno lette come contesto locale, non come stime puntuali da confrontare anno su
> anno con Bagheria.

In [27]:
# Le due serie del thread per i cinque comuni vicini. Stessi filtri e stesse formule dei
# territori di confronto: cambia solo la platea, così le curve sono confrontabili.
vicini_anagrafica = (pd.read_csv(PROCESSED / "genere_mappa_etichette.csv", dtype={"territorio": str})
                     .query("ruolo == 'vicino'")
                     .sort_values("distanza_km"))
CODICI_VICINI = vicini_anagrafica["territorio"].tolist()

CODICE_VICINATO = "VICINI5"
NOME_VICINATO = f"vicinato ({len(CODICI_VICINI)} comuni)"

istr_lav_vicini = leggi("censpop_istr_lav_vicini_long.csv")
lavoro_vicini = istr_lav_vicini[istr_lav_vicini["tavola"].eq("lavoro")]
istruzione_vicini = istr_lav_vicini[istr_lav_vicini["tavola"].eq("istruzione")]
pop_vicini = leggi("censpop_popolazione_vicini_long.csv")
assert set(lavoro_vicini["territorio"]) == set(CODICI_VICINI), "raw e selezione non coincidono"

# --- Gap occupazionale 15-24: definizione di analisi_condizione_15_24 (occupati = 1,
# popolazione = 99) e CI di Newcombe come per i territori di confronto.
base_v = lavoro_vicini[
    lavoro_vicini["eta"].eq("Y15-24")
    & lavoro_vicini["cittadinanza"].eq("TOTAL")
    & lavoro_vicini["titolo_studio"].eq("ALL")
    & lavoro_vicini["genere"].isin(["M", "F"])]
conteggi_v = (base_v.pivot_table(index=["territorio", "anno"], columns=["condizione", "genere"],
                                 values="valore", aggfunc="sum")
              .rename(columns={"1": "occupati", "99": "popolazione"}, level=0))
conteggi_v.columns = [f"{misura}_{gen}" for misura, gen in conteggi_v.columns]

def riga_gap(territorio, nome, anno, r):
    g, lo, hi = gap_con_ci(r["occupati_M"], r["popolazione_M"], r["occupati_F"], r["popolazione_F"])
    return {"territorio": territorio, "nome_territorio": nome, "anno": anno,
            "tasso_F": 100 * r["occupati_F"] / r["popolazione_F"],
            "tasso_M": 100 * r["occupati_M"] / r["popolazione_M"],
            "gap": 100 * g, "gap_lo": 100 * lo, "gap_hi": 100 * hi}


# Gli anni incompleti alla fonte (il 2020 sulla classe 15-24) restano fuori: niente riga,
# nessun valore inventato.
completi = conteggi_v[conteggi_v["popolazione_M"].gt(0) & conteggi_v["popolazione_F"].gt(0)]
righe = [riga_gap(territorio, NOMI[territorio], anno, r)
         for (territorio, anno), r in completi.iterrows()]
# Il vicinato come territorio unico: conteggi sommati e poi il rapporto, mai la media dei
# cinque rapporti (peserebbe Ficarazzi quanto Misilmeri). Il denominatore diventa dello
# stesso ordine di Bagheria e l'intervallo di confidenza si stringe di conseguenza.
righe += [riga_gap(CODICE_VICINATO, NOME_VICINATO, anno, r)
          for anno, r in completi.groupby(level="anno").sum().iterrows()]
ci_gap_vicini = pd.DataFrame(righe)
ci_gap_vicini["rapporto_M_F"] = ci_gap_vicini["tasso_M"] / ci_gap_vicini["tasso_F"]
ci_gap_vicini = ci_gap_vicini.round({"tasso_F": 1, "tasso_M": 1, "gap": 1, "gap_lo": 1,
                                     "gap_hi": 1, "rapporto_M_F": 2})
ci_gap_vicini.to_csv(PROCESSED / "genere_gap_occupazione_ci_vicini.csv", index=False)

# --- Istruzione 9-24 del vicinato e riga della forbice (fig05). Stessa definizione della
# cella sui titoli: almeno il diploma = USE_IF + BL + ML_RDD sul totale ALL. Il pivot somma
# i cinque comuni, quindi la quota è già quella del vicinato aggregato.
scuola_v = istruzione_vicini[
    istruzione_vicini["eta"].eq("Y9-24")
    & istruzione_vicini["cittadinanza"].eq("TOTAL")
    & istruzione_vicini["genere"].isin(["M", "F"])]
titoli_v = scuola_v.pivot_table(index=["anno", "genere"], columns="titolo_studio",
                                values="valore", aggfunc="sum")
scarto_v = (titoli_v[TITOLI].sum(axis=1) - titoli_v["ALL"]).abs().max()
assert scarto_v == 0, f"le categorie non ricostruiscono il totale, scarto {scarto_v}"
diploma_v = (100 * titoli_v[ALMENO_DIPLOMA].sum(axis=1) / titoli_v["ALL"]).round(1)

occ_v = ci_gap_vicini[ci_gap_vicini["territorio"].eq(CODICE_VICINATO)
                      & ci_gap_vicini["anno"].eq(anno_comune)].iloc[0]
forbice_vicini = pd.DataFrame([{
    "nome_territorio": NOME_VICINATO,
    "vantaggio_istruzione_F_pp": round(diploma_v[(anno_comune, "F")] - diploma_v[(anno_comune, "M")], 2),
    "rapporto_M_F_occupazione": occ_v["rapporto_M_F"],
    "tasso_occupazione_F": occ_v["tasso_F"],
    "tasso_occupazione_M": occ_v["tasso_M"],
    "almeno_diploma_F": diploma_v[(anno_comune, "F")],
    "almeno_diploma_M": diploma_v[(anno_comune, "M")],
    "anno": anno_comune,
}])
forbice_vicini.to_csv(PROCESSED / "genere_forbice_vicini.csv", index=False)

# --- Ritenzione per eta singola 2021 -> 2024, stessa media mobile a tre eta.
singole_v = (pop_vicini[
        pop_vicini["cittadinanza"].eq("TOTAL")
        & pop_vicini["genere"].isin(["M", "F"])
        & pop_vicini["eta_anni"].notna()]
    .assign(eta=lambda d: d["eta_anni"].astype(int)))
v21 = singole_v[singole_v["anno"].eq(2021)].groupby(["territorio", "genere", "eta"])["valore"].sum()
v24 = singole_v[singole_v["anno"].eq(2024)].groupby(["territorio", "genere", "eta"])["valore"].sum()

righe = []
for territorio in CODICI_VICINI:
    for genere in ("F", "M"):
        for a in range(13, 32):
            n0, n3 = v21.get((territorio, genere, a)), v24.get((territorio, genere, a + 3))
            righe.append({"territorio": territorio, "nome_territorio": NOMI[territorio],
                          "genere": genere, "eta_2021": a, "eta_2024": a + 3,
                          "n_2021": n0, "n_2024": n3,
                          "ritenzione_pct": round(100 * n3 / n0, 1) if n0 and n3 else None})
profilo_vicini = pd.DataFrame(righe)
for _, gruppo in profilo_vicini.groupby(["territorio", "genere"]):
    liscio = (100 * gruppo["n_2024"].rolling(3, center=True).sum()
              / gruppo["n_2021"].rolling(3, center=True).sum())
    profilo_vicini.loc[gruppo.index, "ritenzione_rolling3_pct"] = liscio.round(1)

# Il vicinato come un territorio solo: si sommano le coorti dei cinque comuni e poi si fa
# il rapporto, non la media dei cinque rapporti. Il denominatore diventa dello stesso
# ordine di Bagheria e la serie smette di oscillare di dieci punti per il rumore comunale.
# Le righe aggregate stanno nello stesso file con un codice territorio riconoscibile: chi
# somma la tabella per territorio deve escluderlo, altrimenti conta due volte le persone.
CODICE_VICINATO = "VICINI5"
pool = (profilo_vicini.groupby(["genere", "eta_2021", "eta_2024"], as_index=False)
        [["n_2021", "n_2024"]].sum()
        .assign(territorio=CODICE_VICINATO,
                nome_territorio=f"vicinato ({len(CODICI_VICINI)} comuni)"))
pool["ritenzione_pct"] = (100 * pool["n_2024"] / pool["n_2021"]).round(1)
for _, gruppo in pool.groupby("genere"):
    liscio = (100 * gruppo["n_2024"].rolling(3, center=True).sum()
              / gruppo["n_2021"].rolling(3, center=True).sum())
    pool.loc[gruppo.index, "ritenzione_rolling3_pct"] = liscio.round(1)
profilo_vicini = pd.concat([profilo_vicini, pool[profilo_vicini.columns]], ignore_index=True)
profilo_vicini.to_csv(PROCESSED / "genere_ritenzione_eta_vicini.csv", index=False)

# --- Composizione per stato, 15-24 (fig02). Stessi codici CL_FORZE_LAV della cella
# condivisa: 1 occupati, 12 in cerca, 5 studenti, 4 casalinghe/i, 7 altra condizione,
# 24 pensione, denominatore 99. Il pivot somma i cinque comuni prima della quota.
base_stati = lavoro_vicini[
    lavoro_vicini["eta"].eq("Y15-24")
    & lavoro_vicini["cittadinanza"].eq("TOTAL")
    & lavoro_vicini["titolo_studio"].eq("ALL")
    & lavoro_vicini["genere"].isin(["M", "F", "T"])]
largo_v = base_stati.pivot_table(index=["anno", "genere"], columns="condizione",
                                 values="valore", aggfunc="sum")
sei_stati_v = pd.DataFrame({
    "occupati": largo_v["1"], "in cerca": largo_v["12"], "studenti": largo_v["5"],
    "casalinghe/i": largo_v["4"], "altra condizione": largo_v["7"], "pensione": largo_v["24"],
})
composizione_vicini = ((100 * sei_stati_v.div(largo_v["99"], axis=0)).round(1)
    .reset_index()
    .assign(territorio=CODICE_VICINATO, nome_territorio=NOME_VICINATO)
    .melt(id_vars=["territorio", "nome_territorio", "anno", "genere"],
          var_name="stato", value_name="quota")
    # il 2020 manca alla fonte sulla classe 15-24: niente riga, come nel file condiviso
    .dropna(subset=["quota"]))
composizione_vicini.to_csv(PROCESSED / "genere_composizione_stato_dettaglio_vicini.csv", index=False)

# --- Ritenzione di coorte (fig03): stesse tre coorti della cella condivisa, coorti dei
# cinque comuni sommate prima del rapporto.
righe = []
for etichetta, (a0, a1) in COORTI.items():
    base = (singole_v[singole_v["anno"].eq(2021) & singole_v["eta"].between(a0, a1)]
            .groupby("genere")["valore"].sum())
    dopo = (singole_v[singole_v["anno"].eq(2024) & singole_v["eta"].between(a0 + 3, a1 + 3)]
            .groupby("genere")["valore"].sum())
    parziale = (100 * dopo / base).rename("ritenzione_%").reset_index()
    parziale["coorte"] = etichetta
    righe.append(parziale)
coorti_vicini = pd.concat(righe, ignore_index=True).assign(
    territorio=CODICE_VICINATO, nome_territorio=NOME_VICINATO)
coorti_vicini["ritenzione_%"] = coorti_vicini["ritenzione_%"].round(1)
coorti_vicini[["territorio", "genere", "ritenzione_%", "coorte", "nome_territorio"]].to_csv(
    PROCESSED / "genere_coorti_vicini.csv", index=False)

print(f"Composizione femminile 15-24 nel {anno_comune}, vicinato contro Bagheria e Palermo:")
confronto_stati = pd.concat([
    composizione_vicini,
    leggi("genere_composizione_stato_dettaglio.csv").query("territorio in [@BAGHERIA, @PALERMO]"),
])
confronto_stati["quota"] = pd.to_numeric(confronto_stati["quota"])
print(confronto_stati.query("anno == @anno_comune and genere == 'F'")
      .pivot_table(index="stato", columns="nome_territorio", values="quota").to_string())

print("\nRitenzione di coorte del vicinato:")
print(coorti_vicini.pivot_table(index="coorte", columns="genere", values="ritenzione_%").to_string())

print("\nGap occupazionale M-F 15-24, ultimo anno disponibile per comune:")
ultimo_anno = ci_gap_vicini.sort_values("anno").groupby("nome_territorio").tail(1)
print(ultimo_anno.set_index("nome_territorio")
      .reindex(vicini_anagrafica["nome_comune"])[["anno", "tasso_F", "tasso_M", "gap", "gap_lo", "gap_hi"]]
      .to_string())
print(f"\nVicinato aggregato {anno_comune}: gap {occ_v['gap']:.1f} punti",
      f"(CI {occ_v['gap_lo']:.1f}-{occ_v['gap_hi']:.1f}), rapporto {occ_v['rapporto_M_F']:.2f}x,",
      f"vantaggio educativo femminile {forbice_vicini['vantaggio_istruzione_F_pp'].iloc[0]:.1f} punti")
print("\nAmpiezza mediana del CI sul gap:",
      f"{(ci_gap_vicini['gap_hi'] - ci_gap_vicini['gap_lo']).median():.1f} punti nei vicini",
      f"contro {(ci_gap['gap_hi'] - ci_gap['gap_lo']).median():.1f} nei quattro territori di confronto")
donne_2225 = pool[pool["genere"].eq("F") & pool["eta_2021"].between(22, 25)]["n_2021"].sum()
print(f"\nVicinato aggregato: coorte femminile 22-25 del 2021 = {int(donne_2225)} persone,",
      "contro le", int(profilo[profilo["territorio"].eq(BAGHERIA) & profilo["genere"].eq("F")
                               & profilo["eta_2021"].between(22, 25)]["n_2021"].sum()), "di Bagheria")
print(pd.concat([profilo, pool])
      .query("genere == 'F' and 22 <= eta_2021 <= 25")
      .pivot_table(index="eta_2021", columns="nome_territorio", values="ritenzione_rolling3_pct")
      .to_string())

print("\nRitenzione femminile 22-25 anni (media mobile), per comune:")
print(profilo_vicini[profilo_vicini["genere"].eq("F") & profilo_vicini["eta_2021"].between(22, 25)]
      .pivot_table(index="eta_2021", columns="nome_territorio", values="ritenzione_rolling3_pct")
      .reindex(columns=vicini_anagrafica["nome_comune"]).to_string())

Composizione femminile 15-24 nel 2024, vicinato contro Bagheria e Palermo:
nome_territorio   Bagheria  Palermo  vicinato (5 comuni)
stato                                                   
altra condizione       6.4      5.0                  6.0
casalinghe/i          13.4     11.3                 13.6
in cerca               6.9      7.2                  8.0
occupati               8.2      9.6                  8.9
pensione               0.1      0.1                  0.1
studenti              65.1     66.8                 63.5

Ritenzione di coorte del vicinato:
genere              F     M
coorte                     
15-19 nel 2021   98.6  99.1
20-24 nel 2021  100.1  98.0
25-29 nel 2021  102.4  99.7

Gap occupazionale M-F 15-24, ultimo anno disponibile per comune:
              anno  tasso_F  tasso_M   gap  gap_lo  gap_hi
nome_comune                                               
Santa Flavia  2024      7.8     17.9  10.1     6.3    13.9
Ficarazzi     2024      9.1     14.7   5.6     2.4


Ampiezza mediana del CI sul gap: 5.7 punti nei vicini contro 0.6 nei quattro territori di confronto

Vicinato aggregato: coorte femminile 22-25 del 2021 = 1756 persone, contro le 1143 di Bagheria
nome_territorio  Bagheria  Italia  Palermo  Sicilia  vicinato (5 comuni)
eta_2021                                                                
22                  103.3   102.8     99.9     99.4                101.3
23                  100.5   102.9     99.9     98.6                100.4
24                   99.6   102.9     98.8     97.9                101.0
25                   97.1   102.9     98.8     97.5                101.7

Ritenzione femminile 22-25 anni (media mobile), per comune:
nome_comune  Santa Flavia  Ficarazzi  Villabate  Casteldaccia  Misilmeri
eta_2021                                                                
22                   92.8      110.9      101.0         109.4       98.2
23                   93.6      104.5       98.5         109.6       99.3
24          

# **Su 1.000 ragazze: istruzione e lavoro sulla stessa fascia**
La tavola istruzione pubblica solo il 9-24, quella del lavoro solo il 15-24: finora i due
gap convivevano su fasce diverse. Ma sotto i 15 anni un diploma è **strutturalmente
impossibile** (la secondaria di II grado si completa a 18-19 anni, le qualifiche IFP non
prima dei 17): il conteggio delle diplomate 9-24 *è* il conteggio 15-24, identico. Il
denominatore 15-24 arriva dalle età singole della demografia (disponibili dal 2021), e la
coerenza fra le due tavole si verifica con un assert, non si assume.

Così i due tassi vivono sulla **stessa popolazione** e si possono leggere su base 1.000.
Quello che l'estrazione **non** compra è la condizionalità: l'incrocio titolo × condizione
non esiste a livello comunale (sezione «Verifica di fattibilità»), quindi "quante delle
diplomate lavorano" resta non calcolabile. Le due quote si mostrano **in parallelo, mai
come stadi di un funnel** — una 16enne può lavorare senza diploma, gli insiemi non sono
annidati.

Il 18-24 usa la stessa logica dei bounds sulle casalinghe: nessun diploma sotto i 18 è un
**bound superiore** (qualche qualifica IFP si ottiene a 17).

In [28]:
# Diplomate/i 9-24 = 15-24 (nessun titolo sotto i 15); denominatori dalle età singole.
def pop_eta_singole(df, da_eta, a_eta, chiavi):
    """Somma delle età singole [da, a] — richiede il dettaglio per anno di età (2021+)."""
    base = df[df["eta_anni"].between(da_eta, a_eta)
              & df["cittadinanza"].eq("TOTAL") & df["stato_civile"].eq("ALL")
              & df["genere"].isin(["M", "F"])]
    return base.groupby(chiavi)["valore"].sum()


def diplomati_9_24(df, chiavi):
    base = df[df["eta"].eq("Y9-24") & df["cittadinanza"].eq("TOTAL")
              & df["genere"].isin(["M", "F"])]
    tab = base.pivot_table(index=chiavi, columns="titolo_studio", values="valore", aggfunc="sum")
    return pd.DataFrame({"diplomati": tab[ALMENO_DIPLOMA].sum(axis=1), "pop_9_24": tab["ALL"]})


def blocco_fasce(istr, pop, chiavi):
    return (diplomati_9_24(istr, chiavi)
            .join(pop_eta_singole(pop, 9, 14, chiavi).rename("pop_9_14"))
            .join(pop_eta_singole(pop, 15, 24, chiavi).rename("pop_15_24"))
            .join(pop_eta_singole(pop, 18, 24, chiavi).rename("pop_18_24")))


istruzione_confronto = istr_lav[istr_lav["tavola"].eq("istruzione")
                                & istr_lav["territorio"].isin(CONFRONTO)]
# Vicinato come territorio unico: conteggi dei cinque comuni sommati prima dei rapporti.
vicinato_fasce = (blocco_fasce(istruzione_vicini, pop_vicini, ["anno", "genere"])
                  .reset_index().assign(territorio=CODICE_VICINATO))
fasce = (pd.concat([
    blocco_fasce(istruzione_confronto, popolazione[popolazione["territorio"].isin(CONFRONTO)],
                 ["territorio", "anno", "genere"]).reset_index(),
    vicinato_fasce,
], ignore_index=True)
    .dropna(subset=["pop_15_24"])  # le età singole partono dal 2021: il 2018-2019 esce qui
    .set_index(["territorio", "anno", "genere"]))

# Coerenza fra le tavole: la popolazione 9-24 di istruzione deve ricostruirsi ESATTAMENTE
# dalle età singole della demografia. Se un giorno divergono, l'estrazione non è più lecita.
scarto = (fasce["pop_9_24"] - fasce["pop_9_14"] - fasce["pop_15_24"]).abs().max()
assert scarto == 0, f"tavole incoerenti: scarto massimo {scarto} persone"
assert (fasce["diplomati"] <= fasce["pop_15_24"]).all()

# Occupati 15-24: territori di confronto da `condizione`, vicinato dai conteggi già sommati.
occ_vicinato = (completi.reset_index().groupby("anno")[["occupati_F", "occupati_M"]].sum()
                .reset_index()
                .melt(id_vars="anno", var_name="misura", value_name="occupati")
                .assign(genere=lambda d: d["misura"].str[-1], territorio=CODICE_VICINATO))
occ = (pd.concat([
    condizione[condizione["territorio"].isin(CONFRONTO) & condizione["genere"].isin(["M", "F"])]
        [["territorio", "anno", "genere", "occupati"]],
    occ_vicinato[["territorio", "anno", "genere", "occupati"]],
], ignore_index=True)
    .set_index(["territorio", "anno", "genere"])["occupati"])

per_1000 = fasce.join(occ, how="inner").reset_index()
per_1000["nome_territorio"] = per_1000["territorio"].map(NOMI | {CODICE_VICINATO: NOME_VICINATO})
per_1000["almeno_diploma_15_24_%"] = (100 * per_1000["diplomati"] / per_1000["pop_15_24"]).round(1)
per_1000["occupazione_15_24_%"] = (100 * per_1000["occupati"] / per_1000["pop_15_24"]).round(1)
per_1000["almeno_diploma_18_24_bound_%"] = (100 * per_1000["diplomati"] / per_1000["pop_18_24"]).round(1)
per_1000["per_1000_diploma"] = (1000 * per_1000["diplomati"] / per_1000["pop_15_24"]).round().astype(int)
per_1000["per_1000_occupati"] = (1000 * per_1000["occupati"] / per_1000["pop_15_24"]).round().astype(int)
per_1000 = per_1000[["territorio", "nome_territorio", "anno", "genere", "pop_15_24",
                     "diplomati", "occupati", "almeno_diploma_15_24_%", "occupazione_15_24_%",
                     "per_1000_diploma", "per_1000_occupati", "pop_18_24",
                     "almeno_diploma_18_24_bound_%"]].sort_values(["territorio", "anno", "genere"])
per_1000.to_csv(PROCESSED / "genere_per_1000.csv", index=False)

ultimo = per_1000[per_1000["anno"].eq(per_1000["anno"].max())]
print(f"=== {ultimo['anno'].iloc[0]}, per 1.000 residenti 15-24 ===")
print(ultimo.pivot_table(index="nome_territorio", columns="genere",
                         values=["per_1000_diploma", "per_1000_occupati"]).astype(int))

# Il primato del vantaggio educativo, riletto sulle fasce allineate all'età da diploma.
vantaggio = ultimo.pivot_table(index="nome_territorio", columns="genere",
                               values=["almeno_diploma_15_24_%", "almeno_diploma_18_24_bound_%"])
print("\n=== vantaggio educativo F-M (pp) ===")
print(pd.DataFrame({
    "15-24": vantaggio["almeno_diploma_15_24_%", "F"] - vantaggio["almeno_diploma_15_24_%", "M"],
    "18-24 (bound)": vantaggio["almeno_diploma_18_24_bound_%", "F"] - vantaggio["almeno_diploma_18_24_bound_%", "M"],
}).round(1))

=== 2024, per 1.000 residenti 15-24 ===
                    per_1000_diploma      per_1000_occupati     
genere                             F    M                 F    M
nome_territorio                                                 
Bagheria                         510  462                82  165
Italia                           534  495               173  269
Palermo                          473  445                96  165
Sicilia                          509  462               104  203
vicinato (5 comuni)              470  455                89  177

=== vantaggio educativo F-M (pp) ===
                     15-24  18-24 (bound)
nome_territorio                          
Bagheria               4.8            5.7
Italia                 3.9            6.2
Palermo                2.8            4.5
Sicilia                4.7            7.1
vicinato (5 comuni)    1.5            2.5


**📌 Risultato chiave** — Su 1.000 ragazze 15-24 di Bagheria (2024): **510 con almeno il
diploma, 82 occupate**; sui coetanei: 462 e 165. La coerenza fra tavola istruzione ed età
singole è esatta (scarto zero, verificato dall'assert), quindi i due tassi poggiano sulla
stessa popolazione. Sulla fascia allineata il vantaggio educativo femminile resta (+4,8 pp
sul 15-24, +5,7 sul 18-24) ma il **primato di fig05 non regge fuori dal 9-24**: sul 18-24
Sicilia (+7,1) e Italia (+6,2) superano Bagheria — parte del primato era composizione per
età, come la «Verifica di composizione» sospettava. Regge su ogni fascia il distacco dal
vicinato (+5,7 contro +2,5) e la conversione peggiore: i claim della proposal usano questi
due, non il primato assoluto. Serie 2021-2024 stabile (48,8 → 51,0 sul 15-24 F).

# **Il gap delle madri: tre censimenti, 1991-2011**
La sezione precedente fotografa il 2011; qui si usa la profondità dei tre censimenti.
La domanda di `idee/trentanni_di_bagheria.md`: il vantaggio educativo femminile è una
novità, o c'era già ai tempi delle madri — e si è mai convertito in occupazione?
Indicatori **15+ (I1: 6+)**, mai giovanili: la traiettoria si legge accanto alle serie
2018-2024, mai dentro. `I1` è un rapporto M/F × 100 sull'"almeno diploma": sotto 100 le
donne sono più istruite. Il percentile è la posizione fra i 390 comuni in ordine
crescente del valore: per disoccupazione (e per I1 letto come vantaggio maschile) alto
significa peggio. Caveat: le definizioni possono variare fra censimenti; la lettura è
sulla traiettoria delle posizioni, non sui decimali dei livelli.

In [29]:
MADRI = ["L2", "L11", "L10", "L7", "I1"]

valori_confronto = (ottomila[ottomila["territorio"].isin(CONFRONTO)
                             & ottomila["indicatore"].isin(MADRI)]
                    .pivot_table(index=["indicatore", "anno"], columns="territorio", values="valore")
                    [CONFRONTO].rename(columns=NOMI).round(1)
                    .reindex(MADRI, level=0))

comuni_madri = ottomila[ottomila["livello"].eq("1") & ottomila["indicatore"].isin(MADRI)]
righe = []
for (ind, anno), d in comuni_madri.groupby(["indicatore", "anno"]):
    v = d.set_index("territorio")["valore"].dropna()
    righe.append({"indicatore": ind, "anno": anno,
                  "percentile_390": round(100 * (v < v[BAGHERIA]).mean(), 1)})
percentili = pd.DataFrame(righe)

gap_madri = (valori_confronto.reset_index()
             .merge(percentili, on=["indicatore", "anno"])
             .merge(indicatori[["indicatore", "nome_indicatore"]], on="indicatore"))
gap_madri.to_csv(PROCESSED / "genere_gap_madri.csv", index=False)

print("Valori (15+; I1: 6+) e percentile di Bagheria fra i 390 comuni:")
print(gap_madri.set_index(["indicatore", "anno"])
      [["Bagheria", "percentile_390", "Palermo", "Sicilia", "Italia"]].to_string())

Valori (15+; I1: 6+) e percentile di Bagheria fra i 390 comuni:
                 Bagheria  percentile_390  Palermo  Sicilia  Italia
indicatore anno                                                    
L2         1991      20.6             7.9     28.9     28.0    35.3
           2001      27.2            40.8     33.5     30.0    37.6
           2011      28.7            31.5     35.9     33.0    41.8
L11        1991      10.9            22.1     18.0     15.7    27.7
           2001      15.1            23.6     21.5     19.5    32.0
           2011      18.1            12.3     25.5     24.0    36.1
L10        1991      44.2            63.6     45.3     45.0    55.8
           2001      43.8            54.9     44.4     44.6    54.8
           2011      43.1            14.6     45.0     46.9    54.8
L7         1991      47.3            45.9     37.6     44.1    21.7
           2001      44.5            84.4     35.7     34.8    14.8
           2011      36.9            94.6     29.1  

**📌 Risultato chiave** — In trent'anni le donne di Bagheria **sono entrate nel mercato del lavoro e il mercato non le ha assorbite**: la partecipazione femminile sale da 20.6 a 28.7 (dal 8° al 32° percentile siciliano), ma l'occupazione femminile scivola dal 22° al **12° percentile** e la disoccupazione femminile crolla dal 46° al **95°** — peggio di quasi tutti. Intanto il differenziale educativo si chiude e si ribalta: I1 da 102.6 a **98.9**, sotto la parità già nel 2011, unico territorio del panel. Il vantaggio educativo non convertito **non è un incidente del censimento permanente: è il regime del territorio da almeno un decennio**.

> Tre letture della traiettoria:
>
> 1. **partecipazione su, assorbimento no** — L2 +8.1 punti in vent'anni mentre L7 passa
>    da 47.3 a 36.9 *restando* fra i peggiori comuni della Sicilia (95° percentile 2011):
>    il calo del tasso è il ciclo, la posizione è la struttura;
> 2. **il sorpasso educativo è del 2011**, non di oggi: ai tempi delle madri (1991) gli
>    uomini erano ancora più istruiti (I1 102.6). Le ragazze del censimento permanente
>    sono la prima generazione figlia del sorpasso — e trovano lo stesso muro;
> 3. l'occupazione femminile assoluta migliora (10.9 → 18.1) ma **la posizione peggiora**:
>    è la "corsa che arretra" di `idee/02`, osservata sul lato femminile. E non è solo
>    femminile: anche la posizione maschile crolla nel decennio 2001-2011 (L10 dal 55°
>    al 15° percentile) — lo scivolamento relativo è dell'intero mercato locale; la
>    specificità femminile è il muro della disoccupazione (95°) e il sorpasso educativo
>    che non si converte.

# **Le gemelle di Bagheria**
Palermo, Sicilia e Italia dicono se Bagheria è sotto la media, non se è anomala *fra i
comuni che le somigliano*. Il gruppo di controllo si costruisce sui **390 comuni al
2011** con variabili strutturali **non-outcome**: dimensione (`P1`), densità (`P7`),
struttura per età (`P11`, `P12`), stranieri (`S1`), patrimonio abitativo (`A1`, `A4`),
distanza dal capoluogo. Escluse per costruzione: istruzione e lavoro (sono gli esiti da
confrontare), famiglie `F*` (confrontate nella sezione successiva), mobilità `M*` (è il
meccanismo in esame nel parent geografia), vulnerabilità `V*` (l'indice incorpora
istruzione e occupazione). Distanza di Mahalanobis (le variabili sono correlate; le code
asimmetriche entrano in log), k=10; robustezza su metodo e leave-one-variable-out.
Il matching è al 2011 e si usa per confronti 2011: per le serie 2018-2024 il censimento
permanente non copre gli altri comuni — limite di fonte, dichiarato.

In [30]:
import numpy as np

com_2011 = (ottomila[ottomila["livello"].eq("1") & ottomila["anno"].eq(2011)]
            .pivot_table(index="territorio", columns="indicatore", values="valore"))
xy = centroidi.set_index("territorio")
com_2011["dist_palermo_km"] = np.hypot(xy["x"] - xy.loc[PALERMO, "x"],
                                       xy["y"] - xy.loc[PALERMO, "y"]) / 1000

MATCHING = ["P1", "P7", "P11", "P12", "S1", "A1", "A4", "dist_palermo_km"]
X = com_2011[MATCHING].copy()
X["P1"], X["P7"], X["S1"] = np.log10(X["P1"]), np.log10(X["P7"]), np.log1p(X["S1"])
assert not X.isna().any().any(), "matrice di matching incompleta"


def distanze_da_bagheria(matrice, metodo="mahalanobis"):
    if metodo == "mahalanobis":
        inversa = np.linalg.inv(np.cov(matrice.values.T))
        diff = matrice.values - matrice.loc[BAGHERIA].values
        d = pd.Series(np.sqrt(np.einsum("ij,jk,ik->i", diff, inversa, diff)), index=matrice.index)
    elif metodo == "pca4":  # kNN sulle prime 4 componenti principali, sferizzate
        Z = ((matrice - matrice.mean()) / matrice.std(ddof=0)).values
        _, S, Vt = np.linalg.svd(Z - Z.mean(0), full_matrices=False)
        proi = pd.DataFrame((Z @ Vt.T[:, :4]) / S[:4], index=matrice.index)
        d = np.sqrt(((proi - proi.loc[BAGHERIA]) ** 2).sum(axis=1))
    else:  # euclide su z-score
        Z = (matrice - matrice.mean()) / matrice.std(ddof=0)
        d = np.sqrt(((Z - Z.loc[BAGHERIA]) ** 2).sum(axis=1))
    return d.drop(BAGHERIA).sort_values()


K = 10
dist = distanze_da_bagheria(X)
gemelle = dist.head(K)
PROVINCE = {"081": "TP", "082": "PA", "083": "ME", "084": "AG", "085": "CL",
            "086": "EN", "087": "CT", "088": "RG", "089": "SR"}
anagrafica = pd.DataFrame({
    "rank": range(1, K + 1),
    "territorio": gemelle.index,
    "nome_comune": [NOMI[t] for t in gemelle.index],
    "provincia": [PROVINCE[t[:3]] for t in gemelle.index],
    "distanza_matching": gemelle.round(2).values,
    "popolazione_2011": com_2011.loc[gemelle.index, "P1"].astype(int).values,
    "dist_palermo_km": com_2011.loc[gemelle.index, "dist_palermo_km"].round(1).values,
})
anagrafica.to_csv(PROCESSED / "genere_gemelle.csv", index=False)
print(anagrafica.to_string(index=False))

overlap = {metodo: len(set(distanze_da_bagheria(X, metodo).head(K).index) & set(gemelle.index))
           for metodo in ("zscore", "pca4")}
lovo = {v: len(set(distanze_da_bagheria(X.drop(columns=v)).head(K).index) & set(gemelle.index))
        for v in MATCHING}
print(f"\nRobustezza — overlap col gruppo base: z-score {overlap['zscore']}/{K}, "
      f"PCA-4 {overlap['pca4']}/{K}; leave-one-variable-out min "
      f"{min(lovo.values())}/{K} (senza {min(lovo, key=lovo.get)})")

 rank territorio     nome_comune provincia  distanza_matching  popolazione_2011  dist_palermo_km
    1     082070 Termini Imerese        PA               1.74             26201             40.0
    2     082067    Santa Flavia        PA               2.21             10751             20.5
    3     082020          Capaci        PA               2.24             11030              7.4
    4     082073          Trabia        PA               2.41             10360             30.5
    5     082048       Misilmeri        PA               2.51             27570             17.5
    6     082005       Altofonte        PA               2.68             10266             12.2
    7     082071       Terrasini        PA               2.84             11985             20.4
    8     081008           Erice        TP               2.84             28012             64.6
    9     084028 Porto Empedocle        AG               2.84             16841             93.3
   10     084041         Sciac

In [31]:
# Bagheria dentro il suo gruppo: posizione sugli indicatori femminili, giovanili e
# familiari del 2011. "Gemelle sotto Bagheria" = quante hanno un valore più basso.
POSIZIONE = ["L11", "L7", "I1", "L4", "F4", "F7", "M2"]
righe = []
for ind in POSIZIONE:
    nel_gruppo = com_2011.loc[gemelle.index, ind]
    valore = com_2011.loc[BAGHERIA, ind]
    righe.append({"indicatore": ind,
                  "nome": indicatori.set_index("indicatore").loc[ind, "nome_indicatore"][:52],
                  "Bagheria": round(valore, 1),
                  "mediana gemelle": round(nel_gruppo.median(), 1),
                  "min": round(nel_gruppo.min(), 1), "max": round(nel_gruppo.max(), 1),
                  "gemelle sotto Bagheria": int((nel_gruppo < valore).sum())})
pd.DataFrame(righe).set_index("indicatore")

,nome,Bagheria,mediana gemelle,min,max,gemelle sotto Bagheria
indicatore,,,,,,
L11,Tasso di occupazione femminile,18.1,19.8,16.2,26.8,3
L7,Tasso di disoccupazione femminile,36.9,30.5,21.3,41.3,8
I1,Differenziali di genere per l'istruzione super...,98.9,101.2,97.0,107.0,1
L4,Incidenza giovani 15-29 anni che non studiano ...,40.1,41.2,30.4,46.8,4
F4,Incidenza di giovani che vivono da soli,3.6,4.4,3.6,8.6,0
F7,Incidenza di coppie giovani con figli,11.7,11.8,7.7,14.0,5
M2,Mobilità fuori comune per studio o lavoro,14.3,24.0,3.6,30.7,2


**📌 Risultato chiave** — Le dieci gemelle sono riconoscibili (Santa Flavia, Capaci, Misilmeri, Altofonte, Trabia, Terrasini, Termini Imerese nella cintura/costa palermitana, più Erice, Porto Empedocle, Sciacca: costieri di taglia media) e il gruppo è stabile (overlap 8/10 e 6/10 sugli altri metodi, LOVO mai sotto 7/10). Dentro il gruppo Bagheria è **nella norma su NEET e occupazione femminile** — ma è **estrema dove il thread ha già puntato**: il differenziale educativo più favorevole alle donne (I1 98.9, una sola gemella più in basso), la disoccupazione femminile fra le peggiori (8/10 sotto), **l'autonomia giovanile al minimo del gruppo** (F4 3.6%: nessuna gemella più in basso) e quasi la mobilità minima (M2: 2/10 sotto).

> La risposta all'obiezione "ma non è così in tutta la Sicilia?": sui livelli generali sì,
> Bagheria sta come chi le somiglia — il problema femminile giovanile è di scala regionale
> e la proposal deve dirlo. Ma il **profilo** di Bagheria dentro il gruppo è specifico:
> più capitale umano femminile relativo, più donne in coda al mercato, meno giovani
> autonomi, meno mobilità. È il ritratto strutturale della talent trap di genere, e rende
> i target dei KPI difendibili: "la mediana delle gemelle" è un obiettivo che "la media
> nazionale" non può essere. Matching al 2011: per il 2018-2024 il gruppo non è
> osservabile (il censimento permanente comunale copre solo i quattro territori del brief).

In [32]:
# Export per fig08: la stessa posizione della cella sopra, ma con il gruppo intero
# (min-q1-mediana-q3-max) e il percentile sui 390 comuni, così la figura disegna la
# distribuzione e non solo un rank. `verso` è l'unica scelta interpretativa e sta qui,
# non in R: +1 = alto è meglio, -1 = alto è peggio, 0 = descrittivo, nessun verso "buono".
# F7 e M2 sono a 0 di proposito: il notebook mostra che sono tratti di fascia costiera
# (F7) e che la mobilità (M2) non ha un segno univoco senza il thread pendolarismo.
VERSO = {"L11": 1, "L7": -1, "I1": -1, "L4": -1, "F4": 1, "F7": 0, "M2": 0}

nomi = indicatori.set_index("indicatore")["nome_indicatore"]
righe = []
for ind in POSIZIONE:
    nel_gruppo = com_2011.loc[gemelle.index, ind]
    valore = com_2011.loc[BAGHERIA, ind]
    regione = com_2011[ind].dropna()
    righe.append({
        "indicatore": ind,
        "nome": nomi.loc[ind],
        "verso": VERSO[ind],
        "bagheria": round(valore, 1),
        "gemelle_min": round(nel_gruppo.min(), 1),
        "gemelle_q1": round(nel_gruppo.quantile(0.25), 1),
        "gemelle_mediana": round(nel_gruppo.median(), 1),
        "gemelle_q3": round(nel_gruppo.quantile(0.75), 1),
        "gemelle_max": round(nel_gruppo.max(), 1),
        "gemelle_sotto": int((nel_gruppo < valore).sum()),
        "n_gemelle": int(nel_gruppo.notna().sum()),
        "percentile_390": round((regione < valore).mean() * 100, 1),
        "n_regione": int(regione.size),
    })

posizionamento = pd.DataFrame(righe)
posizionamento.to_csv(PROCESSED / "genere_posizionamento.csv", index=False)
posizionamento


,indicatore,nome,verso,bagheria,gemelle_min,gemelle_q1,gemelle_mediana,gemelle_q3,gemelle_max,gemelle_sotto,n_gemelle,percentile_390,n_regione
0,L11,Tasso di occupazione femminile,1,18.1,16.2,17.8,19.8,22.7,26.8,3,10,12.3,390
1,L7,Tasso di disoccupazione femminile,-1,36.9,21.3,28.0,30.5,35.1,41.3,8,10,94.6,390
2,I1,Differenziali di genere per l'istruzione super...,-1,98.9,97.0,100.1,101.2,103.4,107.0,1,10,37.4,390
3,L4,Incidenza giovani 15-29 anni che non studiano ...,-1,40.1,30.4,37.4,41.2,42.7,46.8,4,10,87.4,390
4,F4,Incidenza di giovani che vivono da soli,1,3.6,3.6,4.0,4.4,4.9,8.6,0,10,3.3,390
5,F7,Incidenza di coppie giovani con figli,0,11.7,7.7,10.6,11.8,13.0,14.0,5,10,81.0,390
6,M2,Mobilità fuori comune per studio o lavoro,0,14.3,3.6,16.3,24.0,27.4,30.7,2,10,25.4,390


# **La nuvola dei 390: istruzione × occupazione femminile (2011)**
Il quadrante della sezione "I due gap a confronto" ha quattro territori; qui lo stesso
piano si costruisce per **tutti i comuni siciliani** — `I1` (differenziale educativo,
6+) contro `L11` (occupazione femminile, 15+), 2011. Serve a dare scala: Bagheria smette
di essere un caso isolato e diventa un punto in una distribuzione. Indicatori 15+/6+ e
fonte 2011: contesto strutturale, mai da unire al quadrante giovanile 2018-2024 — in
`fig06` i due pannelli sono affiancati e etichettati, non sovrapposti.

In [33]:
from scipy import stats

nuvola = (com_2011[["I1", "L11"]].dropna().reset_index()
          .assign(nome_comune=lambda d: d["territorio"].map(NOMI),
                  gemella=lambda d: d["territorio"].isin(gemelle.index),
                  evidenzia=lambda d: d["territorio"].map({BAGHERIA: "Bagheria", PALERMO: "Palermo"})))
nuvola.to_csv(PROCESSED / "genere_nuvola_390.csv", index=False)

rho, p_rho = stats.spearmanr(nuvola["I1"], nuvola["L11"])
vantaggio_f = nuvola["I1"] < 100
print(f"Spearman I1 × L11 su {len(nuvola)} comuni (2011): rho = {rho:.2f} (p = {p_rho:.1e})")
print(f"Comuni con vantaggio educativo femminile (I1 < 100): {vantaggio_f.sum()} su {len(nuvola)}")
print(f"  occupazione femminile mediana: {nuvola.loc[vantaggio_f, 'L11'].median():.1f}% fra questi, "
      f"{nuvola.loc[~vantaggio_f, 'L11'].median():.1f}% fra gli altri")
print(f"Bagheria: I1 = {com_2011.loc[BAGHERIA, 'I1']:.1f}, L11 = {com_2011.loc[BAGHERIA, 'L11']:.1f} "
      f"— vantaggio educativo femminile con occupazione femminile al 12° percentile")

Spearman I1 × L11 su 390 comuni (2011): rho = -0.24 (p = 1.2e-06)
Comuni con vantaggio educativo femminile (I1 < 100): 170 su 390
  occupazione femminile mediana: 24.9% fra questi, 22.4% fra gli altri
Bagheria: I1 = 98.9, L11 = 18.1 — vantaggio educativo femminile con occupazione femminile al 12° percentile


**📌 Risultato chiave** — Sui 390 comuni la relazione va nel verso "giusto": **dove le donne sono relativamente più istruite, l'occupazione femminile è più alta** (Spearman fra I1 e L11: -0.24, p<0.001; mediana L11 24.9% nei 170 comuni con I1<100 contro 22.4% negli altri). Bagheria contraddice il pattern: sta **nel quadrante vantaggio-educativo-femminile / occupazione-bassa**, con L11 al 12° percentile. La nuvola è il pannello di contesto di `fig06`, con le gemelle evidenziate.

> Correlazione **ecologica** su dati 2011: orienta l'ipotesi ("l'istruzione femminile di
> solito si converte, qui no"), non dimostra alcun nesso individuale. Il valore aggiunto è
> la scala: il paradosso di Bagheria non è "la Sicilia va così" — la maggioranza dei
> comuni col suo profilo educativo occupa di più.

# **Formazione familiare precoce e autonomia abitativa (2011)**
Il canale ipotizzato da `idee/casalinghe_a_venti_anni.md` e dal parent famiglia-welfare,
misurato con gli indicatori `F` mai usati finora: giovani soli (`F4`), monogenitori
giovani (`F5`), coppie giovani senza e con figli (`F6`, `F7`). Avvertenza di
denominatore: `F5`-`F7` sono quote sul **totale delle famiglie**, quindi risentono della
struttura per età del comune (un comune più giovane ha meccanicamente più coppie
giovani); `F4` è invece rapportato alla popolazione giovane ed è il più pulito dei
quattro. Dati 1991-2011, mai in serie con il 2018-2024.

In [34]:
FAMIGLIA = ["F4", "F5", "F6", "F7"]
righe = []
for ind in FAMIGLIA:
    for anno in (1991, 2001, 2011):
        comuni_f = (ottomila[ottomila["livello"].eq("1") & ottomila["anno"].eq(anno)
                             & ottomila["indicatore"].eq(ind)]
                    .set_index("territorio")["valore"].dropna())
        confronto = (ottomila[ottomila["territorio"].isin(CONFRONTO)
                              & ottomila["anno"].eq(anno) & ottomila["indicatore"].eq(ind)]
                     .set_index("territorio")["valore"])
        righe.append({"indicatore": ind, "anno": anno,
                      "bagheria": round(comuni_f[BAGHERIA], 1),
                      "percentile_390": round(100 * (comuni_f < comuni_f[BAGHERIA]).mean(), 1),
                      "mediana_gemelle": round(comuni_f[gemelle.index].median(), 1),
                      "palermo": round(confronto[PALERMO], 1),
                      "italia": round(confronto[ITALIA], 1)})
famiglia = (pd.DataFrame(righe)
            .merge(indicatori[["indicatore", "nome_indicatore"]], on="indicatore"))
famiglia.to_csv(PROCESSED / "genere_famiglia_precoce.csv", index=False)
famiglia.set_index(["indicatore", "anno"]).drop(columns="nome_indicatore")

bagheria  percentile_390  mediana_gemelle  palermo  italia
indicatore anno                                                            
F4         1991       1.3             9.0              2.0      2.1     2.9
           2001       1.7             6.4              2.6      2.6     4.6
           2011       3.6             3.3              4.4      4.0     7.0
F5         1991       1.3            54.4              1.0      1.7     1.0
           2001       0.9            56.9              0.7      0.8     1.0
           2011       0.9            49.0              0.8      0.7     1.0
F6         1991       5.0            80.0              4.8      4.5     5.4
           2001       4.7            84.4              4.2      4.1     5.0
           2011       3.4            81.8              3.6      2.9     3.4
F7         1991      25.6            88.5             24.6     20.9    16.3
           2001      18.4            86.7             17.8     14.3    11.0
           2011      11.7            81.0             11.8      9.3     7.4

**📌 Risultato chiave** — Il canale familiare ha numeri estremi in entrambe le direzioni: **giovani che vivono da soli al 3° percentile siciliano** (F4 = 3.6%, la metà dell'Italia, ultima anche fra le gemelle) e **coppie giovani con figli all'81°** (F7 = 11.7% delle famiglie, contro il 7.4% nazionale) — con la stessa posizione relativa già nel 1991 (9° e 88° percentile). A Bagheria la transizione all'età adulta passa dalla famiglia — formata presto — e quasi mai dall'autonomia abitativa. È il contesto strutturale in cui "casalinga a vent'anni" è un esito di sistema, non una scelta anomala.

> Cautele: `F7` risente della struttura per età (denominatore = tutte le famiglie) e il
> confronto giusto è il percentile e le gemelle, non l'Italia — e fra le gemelle F7 è
> nella norma (mediana 11.8): la formazione familiare precoce è un tratto di tutta la
> fascia costiera comparabile, non un'anomalia di Bagheria; `F4` non ne risente ed è
> al minimo del gruppo di controllo. La traiettoria 1991→2011 (F7 25.6 → 11.7) segue il
> calo generale della fecondità: a cambiare è il livello nazionale, non la posizione
> relativa di Bagheria, che resta nel quinto più alto. Il **motivo** (scelta, vincolo,
> norma) non è nei dati — resta il limite dichiarato del thread. Lo stato civile per età
> (fetch SDMX) direbbe se la formazione familiare precoce passa dai matrimoni.

# **Trend paralleli: il controfattuale, testato sul passato**
`idee/misurare_il_dopo.md` propone un disegno differenza-nelle-differenze per la
valutazione. Il DiD post-intervento oggi non esiste (nessun intervento è partito), ma il
suo prerequisito — **trend paralleli nel pre-periodo** — si testa già: (a) sul tasso di
occupazione femminile 15-24, 2018-2024, contro i tre benchmark (GLM binomiale a link
identità, stessa convenzione LPM del notebook: pendenze in pp/anno); (b) sul lungo
periodo, `L11` di Bagheria contro le gemelle sui tre censimenti. Sei punti temporali e
conteggi comunali: il test ha la potenza che ha — "non rifiutato" non è "dimostrato".

In [35]:
femmine = giovani[giovani["genere"].eq("F")].assign(
    p=lambda d: d["occupati"] / d["popolazione"], anno_c=lambda d: d["anno"] - 2021)
parallelo = smf.glm("p ~ anno_c * C(nome_territorio, Treatment('Bagheria'))", data=femmine,
                    family=sm.families.Binomial(link=sm.families.links.Identity()),
                    var_weights=femmine["popolazione"]).fit()
intervalli = parallelo.conf_int()

righe = [{"termine": "pendenza Bagheria (pp/anno)",
          "stima": 100 * parallelo.params["anno_c"],
          "CI 95% basso": 100 * intervalli.loc["anno_c", 0],
          "CI 95% alto": 100 * intervalli.loc["anno_c", 1],
          "p": parallelo.pvalues["anno_c"]}]
for nome in BENCHMARK:
    chiave = f"anno_c:C(nome_territorio, Treatment('Bagheria'))[T.{nome}]"
    righe.append({"termine": f"differenza di pendenza: {nome} - Bagheria",
                  "stima": 100 * parallelo.params[chiave],
                  "CI 95% basso": 100 * intervalli.loc[chiave, 0],
                  "CI 95% alto": 100 * intervalli.loc[chiave, 1],
                  "p": parallelo.pvalues[chiave]})
pendenze_f = pd.DataFrame(righe).round({"stima": 2, "CI 95% basso": 2, "CI 95% alto": 2, "p": 3})
pendenze_f.to_csv(PROCESSED / "genere_pretrend.csv", index=False)
print("(a) Tasso di occupazione femminile 15-24, 2018-2024 (2020 mancante alla fonte):")
print(pendenze_f.to_string(index=False))

righe = []
for anno in (1991, 2001, 2011):
    serie = (ottomila[ottomila["livello"].eq("1") & ottomila["anno"].eq(anno)
                      & ottomila["indicatore"].eq("L11")].set_index("territorio")["valore"])
    righe.append({"anno": anno, "bagheria": serie[BAGHERIA],
                  "gemelle_mediana": serie[gemelle.index].median(),
                  "gemelle_q1": serie[gemelle.index].quantile(0.25),
                  "gemelle_q3": serie[gemelle.index].quantile(0.75)})
storico_gemelle = pd.DataFrame(righe).round(1)
storico_gemelle.to_csv(PROCESSED / "genere_pretrend_gemelle.csv", index=False)
print("\n(b) Occupazione femminile 15+ (L11), Bagheria contro le gemelle:")
print(storico_gemelle.to_string(index=False))

(a) Tasso di occupazione femminile 15-24, 2018-2024 (2020 mancante alla fonte):
                                   termine  stima  CI 95% basso  CI 95% alto     p
               pendenza Bagheria (pp/anno)   0.65          0.48         0.81 0.000
differenza di pendenza: Palermo - Bagheria  -0.10         -0.27         0.08 0.291
differenza di pendenza: Sicilia - Bagheria  -0.17         -0.33         0.00 0.054
 differenza di pendenza: Italia - Bagheria  -0.08         -0.25         0.08 0.328

(b) Occupazione femminile 15+ (L11), Bagheria contro le gemelle:
 anno  bagheria  gemelle_mediana  gemelle_q1  gemelle_q3
 1991      10.9             10.8         9.5        14.5
 2001      15.1             15.0        14.2        17.9
 2011      18.1             19.8        17.8        22.7


**📌 Risultato chiave** — Il prerequisito del disegno di valutazione regge: la pendenza del tasso femminile di Bagheria (+0.65 pp/anno, CI 0.48-0.81) è **indistinguibile da Palermo e Italia** (differenze -0.10 e -0.08, p=0.29 e 0.33; la Sicilia è al margine, -0.17, p=0.054). E sul lungo periodo Bagheria era **identica alle sue gemelle** nel 1991 (10.9 contro 10.8) e nel 2001 (15.1 contro 15.0), poi **se ne stacca nel decennio 2001-2011** (18.1 contro una mediana di 19.8, con 7 gemelle su 10 sopra): lo scarto dalle comparabili è recente, non eterno.

> Per la proposal: il controfattuale dichiarato in anticipo è **Palermo** (trend parallelo
> non rifiutato sul pre-periodo, stessa fonte, lettura annuale), con le gemelle come
> ancora di lungo periodo per i target. Limiti scritti: sei punti e potenza bassa — il
> test non dimostra i paralleli, non li rifiuta; il DiD 2018-2024 contro le gemelle non è
> possibile (il censimento permanente comunale copre solo i quattro territori del brief) —
> estenderlo richiederebbe un fetch nuovo, decisione di team. La divergenza 2001-2011
> dalle gemelle data anche il problema: qualunque spiegazione della talent trap femminile
> deve essere compatibile con un peggioramento **relativo** concentrato in quel decennio.

# **Il ponte fra i due censimenti: dal 2011 al 2024**

Il lungo periodo di questo thread si ferma al 2011 perché 8milaCensus è l'ultimo censimento
decennale. Ma la classe **15 anni e più** esiste anche nel censimento permanente
(`AGE_NOCLASS = Y_GE15`), con gli stessi numeratori e denominatori del codebook: i quattro
indicatori di lavoro si ricalcolano per il 2018-2024, e scaricando i 390 comuni siciliani
anche i percentili regionali.

Serve a tre cose:

1. la posizione di Bagheria fra i 390 comuni arriva al 2024, non più al 2011;
2. il confronto con le dieci gemelle diventa una serie recente e non solo storica —
   è il pre-periodo del DiD, misurato invece che assunto;
3. il salto 2011 → 2018 mette a confronto **due rilevazioni con disegni diversi** —
   censimento decennale universale a questionario contro censimento permanente campionario
   appoggiato ai registri. Se danno lo stesso livello, il livello non è un artefatto del
   disegno della rilevazione.

Il punto 3 non è una validazione esterna: i numeri restano ISTAT in entrambi i casi, e la
coerenza va misurata indicatore per indicatore, non dichiarata. Le due fonti restano due:
mai una linea continua fra 2011 e 2018, sempre uno stacco visibile e la fonte per blocco.

In [36]:
# I quattro indicatori 15+ di fig10 ricostruiti dal censimento permanente. Le formule sono
# quelle del codebook 8milaCensus (data/processed/indicatori.csv), non un'approssimazione:
#   L2  = attive F / residenti F 15+       L11 = occupate F / residenti F 15+
#   L10 = occupati M / residenti M 15+     L7  = in cerca F / attive F 15+
# Codici CUR_ACT_STAT: 1 occupato, 12 in cerca di occupazione, 22 forze di lavoro, 99 totale.
# I1 resta fuori: 8milaCensus lo calcola sulla popolazione 6+, la tavola istruzione del
# permanente parte da 9+ e non ha una classe 15+. Sarebbe un altro indicatore, non un seguito.
RECENTI = ["L2", "L11", "L10", "L7"]


def tassi_15piu(tabella: pd.DataFrame) -> pd.DataFrame:
    """L2/L7/L10/L11 dalla tavola lavoro sulla classe Y_GE15, definizioni 8milaCensus."""
    base = tabella[tabella["eta"].eq("Y_GE15") & tabella["cittadinanza"].eq("TOTAL")
                   & tabella["titolo_studio"].eq("ALL") & tabella["genere"].isin(["M", "F"])]
    c = base.pivot_table(index=["territorio", "anno"], columns=["genere", "condizione"],
                         values="valore", aggfunc="sum")
    return pd.DataFrame({"L2":  100 * c[("F", "22")] / c[("F", "99")],
                         "L11": 100 * c[("F", "1")] / c[("F", "99")],
                         "L10": 100 * c[("M", "1")] / c[("M", "99")],
                         "L7":  100 * c[("F", "12")] / c[("F", "22")]})


sicilia_390 = leggi("censpop_lavoro_15piu_sicilia_long.csv")
assert sicilia_390["territorio"].nunique() == 390, "platea diversa da quella dei percentili 2011"
tassi_390 = tassi_15piu(sicilia_390)
tassi_confronto = tassi_15piu(istr_lav[istr_lav["tavola"].eq("lavoro")])

# --- (a) coerenza fra le due rilevazioni: 2011 decennale contro 2018 permanente --------
val_2011 = (ottomila[ottomila["anno"].eq(2011) & ottomila["indicatore"].isin(RECENTI)
                     & ottomila["territorio"].isin(CONFRONTO)]
            .pivot_table(index="territorio", columns="indicatore", values="valore"))
val_2018 = tassi_confronto.xs(2018, level="anno")

# --- (b) il salto interno al permanente fra 2019 e 2021, sugli stessi indicatori --------
# Il 2020 manca alla fonte su ogni classe che contenga i 15-24: il confronto utile è 2019-2021.
salto = tassi_confronto.xs(2021, level="anno") - tassi_confronto.xs(2019, level="anno")

coerenza = pd.DataFrame([
    {"territorio": NOMI[t], "indicatore": ind,
     "decennale_2011": round(val_2011.loc[t, ind], 1),
     "permanente_2018": round(val_2018.loc[t, ind], 1),
     "scarto_2011_2018": round(val_2018.loc[t, ind] - val_2011.loc[t, ind], 1),
     "salto_2019_2021": round(salto.loc[t, ind], 1)}
    for ind in RECENTI for t in CONFRONTO])
coerenza.to_csv(PROCESSED / "genere_coerenza_fonti.csv", index=False)

print("Due rilevazioni indipendenti sullo stesso indicatore e la stessa fascia (15+):")
print(coerenza.pivot(index="indicatore", columns="territorio")
      [["decennale_2011", "permanente_2018", "scarto_2011_2018", "salto_2019_2021"]]
      .reindex(RECENTI).to_string())

Due rilevazioni indipendenti sullo stesso indicatore e la stessa fascia (15+):
           decennale_2011                        permanente_2018                        scarto_2011_2018                        salto_2019_2021                       
territorio       Bagheria Italia Palermo Sicilia        Bagheria Italia Palermo Sicilia         Bagheria Italia Palermo Sicilia        Bagheria Italia Palermo Sicilia
indicatore                                                                                                                                                            
L2                   28.7   41.8    35.9    33.0            30.4   44.0    37.3    35.9              1.7    2.2     1.4     2.9            -4.7   -1.6    -4.2    -4.3
L11                  18.1   36.1    25.5    24.0            18.8   36.8    25.0    24.8              0.7    0.7    -0.5     0.8             1.2    0.5     1.8     1.0
L10                  43.1   54.8    45.0    46.9            40.4   53.8    43.5    44.

In [37]:
# Percentile di Bagheria fra i 390 comuni, anno per anno, sulla stessa platea del 2011
# (i comuni ai confini 2011: Misiliscemi, nato nel 2021, resta fuori per non cambiare il
# denominatore fra le due epoche). Il percentile è un rango calcolato dentro l'anno, quindi
# assorbe lo scarto di definizione fra le fonti, che sposta tutti i comuni nello stesso verso:
# è il motivo per cui il pannello dei percentili può attraversare il 2011 e i livelli no.
percentili_recenti = pd.DataFrame([
    {"indicatore": ind, "anno": anno,
     "percentile_390": round(100 * (v < v[BAGHERIA]).mean(), 1), "comuni": int(v.notna().sum())}
    for ind in RECENTI
    for anno, serie in tassi_390[ind].groupby("anno")
    for v in [serie.droplevel("anno").dropna()]])

madri_recente = (tassi_confronto.loc[CONFRONTO, RECENTI].round(1).stack()
                 .rename("valore").reset_index().rename(columns={"level_2": "indicatore"})
                 .pivot_table(index=["indicatore", "anno"], columns="territorio", values="valore")
                 [CONFRONTO].rename(columns=NOMI).reset_index()
                 .merge(percentili_recenti, on=["indicatore", "anno"])
                 .merge(indicatori[["indicatore", "nome_indicatore"]], on="indicatore")
                 .assign(fonte="censimento permanente")
                 .sort_values(["indicatore", "anno"], key=lambda c: c.map(
                     {v: i for i, v in enumerate(RECENTI)}).fillna(c) if c.name == "indicatore" else c))
madri_recente.to_csv(PROCESSED / "genere_madri_recente.csv", index=False)

print("Percentile di Bagheria fra i 390 comuni, 2018-2024 (2020 assente alla fonte):")
print(madri_recente.pivot(index="anno", columns="indicatore", values="percentile_390")
      [RECENTI].to_string())
print("\nPer confronto, gli stessi percentili ai tre censimenti (8milaCensus):")
print(gap_madri[gap_madri["indicatore"].isin(RECENTI)]
      .pivot(index="anno", columns="indicatore", values="percentile_390")[RECENTI].to_string())

Percentile di Bagheria fra i 390 comuni, 2018-2024 (2020 assente alla fonte):
indicatore    L2   L11   L10    L7
anno                              
2018        21.5   8.2  15.4  92.6
2019        21.3   9.7  19.2  91.8
2021        17.9  13.3  34.9  95.6
2022        15.1  13.1  35.6  84.4
2023        14.1  12.6  28.5  83.1
2024        16.9  16.9  29.5  73.6

Per confronto, gli stessi percentili ai tre censimenti (8milaCensus):
indicatore    L2   L11   L10    L7
anno                              
1991         7.9  22.1  63.6  45.9
2001        40.8  23.6  54.9  84.4
2011        31.5  12.3  14.6  94.6


In [38]:
# Le dieci gemelle dentro il censimento permanente: la banda interquartile del gruppo
# diventa una serie 2018-2024 invece di tre punti storici. Bagheria viene dal file dei 390,
# le gemelle dal loro raw: stessa formula, stessa fascia, stessa fonte.
lavoro_gemelle = leggi("censpop_lavoro_gemelle_long.csv")
tassi_gemelle = tassi_15piu(lavoro_gemelle[lavoro_gemelle["tavola"].eq("lavoro")])
assert set(tassi_gemelle.index.get_level_values("territorio")) == set(gemelle.index), \
    "il raw delle gemelle non coincide con il gruppo del matching"

gemelle_recente = pd.DataFrame([
    {"anno": anno, "bagheria": tassi_390.loc[(BAGHERIA, anno), "L11"],
     "gemelle_mediana": v.median(), "gemelle_q1": v.quantile(0.25), "gemelle_q3": v.quantile(0.75)}
    for anno, serie in tassi_gemelle["L11"].groupby("anno")
    for v in [serie.droplevel("anno")]]).round(1).assign(fonte="censimento permanente")
gemelle_recente.to_csv(PROCESSED / "genere_pretrend_gemelle_recente.csv", index=False)

scarto = (gemelle_recente["bagheria"] - gemelle_recente["gemelle_mediana"]).round(1)
print("Occupazione femminile 15+ (L11), Bagheria contro le dieci gemelle:")
print(gemelle_recente.assign(scarto=scarto).to_string(index=False))
print(f"\nStorico per confronto (8milaCensus, scarto finale "
      f"{storico_gemelle['bagheria'].iloc[-1] - storico_gemelle['gemelle_mediana'].iloc[-1]:+.1f}):")
print(storico_gemelle.to_string(index=False))

Occupazione femminile 15+ (L11), Bagheria contro le dieci gemelle:
 anno  bagheria  gemelle_mediana  gemelle_q1  gemelle_q3                 fonte  scarto
 2018      18.8             21.1        19.1        23.0 censimento permanente    -2.3
 2019      19.4             21.5        19.6        24.2 censimento permanente    -2.1
 2021      20.6             23.1        20.8        25.3 censimento permanente    -2.5
 2022      21.1             23.7        21.3        25.4 censimento permanente    -2.6
 2023      22.2             25.3        22.7        27.0 censimento permanente    -3.1
 2024      23.7             26.0        23.8        27.9 censimento permanente    -2.3

Storico per confronto (8milaCensus, scarto finale -1.7):
 anno  bagheria  gemelle_mediana  gemelle_q1  gemelle_q3
 1991      10.9             10.8         9.5        14.5
 2001      15.1             15.0        14.2        17.9
 2011      18.1             19.8        17.8        22.7


**📌 Risultato chiave** — Il muro si segue fino al **2024**, e **non si è richiuso**.

*(a) Le due rilevazioni concordano — su due indicatori su quattro.* Sul tasso di occupazione
femminile 15+ il censimento decennale 2011 (universale, a questionario) e il censimento
permanente 2018 (campionario, sui registri) danno **18,1 e 18,8** a Bagheria, e lo scarto resta
entro ±1 punto anche su Palermo, Sicilia e Italia: il livello femminile non è un artefatto del
disegno della rilevazione. La stessa verifica **boccia L7 e L2**: dentro il permanente, fra 2019
e 2021, la disoccupazione femminile crolla di **15,5 punti** a Bagheria e di 4,5 in Italia —
è un cambio di misura, non un mercato che guarisce, e in figura va marcato.

*(b) Percentili sui 390 comuni.* L'occupazione femminile scende ancora dopo il 2011
(12° → **8° percentile nel 2018**) e risale solo in parte (**17° nel 2024**). Nel frattempo
l'occupazione **maschile** risale dal 15° al 30° e la **partecipazione femminile continua a
scendere** (32° nel 2011 → **17° nel 2024**). La lettura del 2011 — «un mercato che si è
ristretto per tutti» — non regge più al 2024: gli uomini recuperano, le donne no.

*(c) Le gemelle.* Bagheria sta sotto la mediana del gruppo in **tutti** gli anni 2018-2024, di
2,1-3,1 punti, contro i −1,7 del 2011. Lo stacco aperto nel decennio 2001-2011 non si è chiuso:
tredici anni dopo Bagheria è ancora appoggiata al primo quartile delle sue gemelle. Il
controfattuale del DiD ha adesso un pre-periodo **misurato** e non solo assunto.

# **I claim reggono al 2024? Tre verifiche**

Il ponte qui sopra ha portato *fig10* fino al 2024. Le altre figure non sono tutte
altrettanto fresche, e la freschezza non è l'unico problema: una fotografia del 2011
può essere obsoleta, ma una fotografia del 2024 costruita su **un anno solo** può
essere altrettanto fragile. Tre verifiche, una per figura:

1. **fig04** legge la posizione di Bagheria fra i 390 comuni siciliani al 2011. La
   graduatoria del 2011 predice ancora quella del 2024, o si è rimescolata?
2. **fig07** data la fuga con il profilo per età singola 2021-2024, tre anni di braccio.
   Le classi quinquennali della demografia (2001, 2011, 2018-2024) permettono di
   chiedersi **da quando**.
3. **fig05** è già al 2024, ma è una fotografia: le due classifiche che racconta reggono
   su tutti gli anni disponibili, o le decide l'ultimo?

In [39]:
# ---- (1) fig04: la graduatoria dei 390 comuni sopravvive al cambio di rilevazione? ----
# Spearman e non Pearson perché la domanda è sull'ordine, non sui livelli: i livelli le
# due rilevazioni li misurano in modo diverso (cella sopra), il rango dentro l'anno no.
# Se il rango del 2011 predice quello del 2024, un claim di posizionamento costruito sul
# 2011 non era una scommessa sul passato — era una previsione, e qui si verifica.
from scipy.stats import spearmanr

occ_390 = tassi_390["L11"].unstack("anno")
occ_390.insert(0, 2011, occ_femminile.set_index("territorio")["occupazione_femminile_2011"])
assert occ_390.notna().all().all(), "la platea 2011 e quella 2018-2024 non coincidono"
ANNI_390 = list(occ_390.columns)


def percentile_interno(s: pd.Series) -> pd.Series:
    """Quota di comuni strettamente sotto: la stessa convenzione di `percentili_recenti`."""
    return (100 * (s.rank(method="min") - 1) / len(s)).round(1)


def quintile_basso(anno: int) -> set:
    return set(occ_390.index[occ_390[anno].rank(pct=True) <= 0.2])


def rho(a: int, b: int) -> float:
    return round(float(spearmanr(occ_390[a], occ_390[b]).statistic), 3)


distribuzione_390 = pd.DataFrame([
    {"anno": anno,
     "fonte": "8milaCensus" if anno == 2011 else "censimento permanente",
     "bagheria": round(occ_390.loc[BAGHERIA, anno], 1),
     "percentile": percentile_interno(occ_390[anno])[BAGHERIA],
     "comuni_sotto": int((occ_390[anno] < occ_390.loc[BAGHERIA, anno]).sum()),
     "mediana": round(occ_390[anno].median(), 1),
     "q1": round(occ_390[anno].quantile(0.25), 1),
     "q3": round(occ_390[anno].quantile(0.75), 1),
     "minimo": round(occ_390[anno].min(), 1),
     "massimo": round(occ_390[anno].max(), 1),
     "rho_vs_2011": rho(2011, anno),
     "rho_vs_2024": rho(anno, 2024),
     "quintile_basso_ancora_tale_nel_2024_pct":
         round(100 * len(quintile_basso(anno) & quintile_basso(2024)) / len(quintile_basso(anno)), 1)}
    for anno in ANNI_390])
distribuzione_390.to_csv(PROCESSED / "genere_distribuzione_390.csv", index=False)

# Le due convenzioni di percentile (questa e quella di `percentili_recenti`) devono
# coincidere: se un giorno divergono, fig04 e fig10 raccontano ranghi diversi.
atteso = percentili_recenti.set_index(["indicatore", "anno"]).loc[("L11", 2024), "percentile_390"]
assert distribuzione_390.set_index("anno").loc[2024, "percentile"] == atteso, "due percentili diversi"

# Tavola per la mappa: gli stessi comuni ai due estremi della serie. `ruolo` marca chi va
# etichettato — Bagheria, i cinque vicini e gli estremi regionali di CIASCUN anno, che nel
# 2024 non sono gli stessi comuni del 2011.
ruolo = pd.Series("", index=occ_390.index)
for anno in (2011, 2024):
    ruolo[occ_390[anno].idxmin()] = f"minimo {anno}"
    ruolo[occ_390[anno].idxmax()] = f"massimo {anno}"
ruolo[CODICI_VICINI] = "vicino"
ruolo[BAGHERIA] = "Bagheria"

mappa_2011_2024 = (pd.DataFrame(index=occ_390.index)
    .assign(nome_comune=centroidi.set_index("territorio")["nome_comune"],
            x=xy["x"], y=xy["y"], distanza_km=distanza_km.round(1),
            occ_2011=occ_390[2011].round(1), occ_2024=occ_390[2024].round(1),
            pct_2011=percentile_interno(occ_390[2011]),
            pct_2024=percentile_interno(occ_390[2024]),
            ruolo=ruolo)
    .reset_index())
mappa_2011_2024.to_csv(PROCESSED / "genere_mappa_2011_2024.csv", index=False)

print("Occupazione femminile 15+, distribuzione dei 390 comuni siciliani:")
print(distribuzione_390.set_index("anno").to_string())
print(f"\nLa graduatoria del 2011 predice quella del 2024: rho = {rho(2011, 2024)}",
      f"su {len(occ_390)} comuni, attraverso 13 anni e due rilevazioni diverse.")
print(f"Dentro il solo permanente, 2018 vs 2024: rho = {rho(2018, 2024)}.")
print(f"Del quintile più basso del 2011, il "
      f"{distribuzione_390.set_index('anno').loc[2011, 'quintile_basso_ancora_tale_nel_2024_pct']:.0f}%"
      f" è ancora nel quintile più basso nel 2024 — Bagheria compresa: "
      f"{BAGHERIA in quintile_basso(2011) and BAGHERIA in quintile_basso(2024)}")
print("\nEstremi regionali e vicini etichettati sulla mappa:")
print(mappa_2011_2024[mappa_2011_2024["ruolo"].ne("")]
      [["nome_comune", "ruolo", "occ_2011", "occ_2024", "pct_2011", "pct_2024"]]
      .sort_values("occ_2024").to_string(index=False))

Occupazione femminile 15+, distribuzione dei 390 comuni siciliani:
                      fonte  bagheria  percentile  comuni_sotto  mediana    q1    q3  minimo  massimo  rho_vs_2011  rho_vs_2024  quintile_basso_ancora_tale_nel_2024_pct
anno                                                                                                                                                                    
2011            8milaCensus      18.1        12.3            48     23.6  19.7  27.4    13.0     39.4        1.000        0.848                                     74.0
2018  censimento permanente      18.8         8.2            32     24.3  20.9  27.4    14.8     36.0        0.883        0.919                                     83.3
2019  censimento permanente      19.4         9.7            38     25.0  21.6  28.0    14.8     35.9        0.879        0.925                                     83.3
2021  censimento permanente      20.6        13.3            52     25.7  22.7  28.7    

In [40]:
# ---- (2) fig07: da quando si perde chi ------------------------------------------------
# Le classi quinquennali della demografia coprono 2001, 2011 e 2018-2024: una coorte si
# segue spostandosi di una classe ogni cinque anni (chi ha 15-19 anni in t ne ha 25-29 in
# t+10). Il profilo per età singola di `genere_ritenzione_eta.csv` dice DOVE si perde chi;
# questo dice DA QUANDO, che tre anni di braccio non possono dire.
CLASSI_QUINQ = ["Y10-14", "Y15-19", "Y20-24", "Y25-29", "Y30-34", "Y35-39", "Y40-44", "Y45-49"]
PERIODI = [(2001, 2011), (2011, 2021), (2018, 2023), (2019, 2024)]

classi = leggi("censpop_demografia_classi_long.csv")
conteggi_classi = (classi[classi["genere"].isin(["M", "F"]) & classi["eta"].isin(CLASSI_QUINQ)]
                   .pivot_table(index=["territorio", "genere", "eta"], columns="anno", values="valore"))


def coorti_quinquennali(anno_da: int, anno_a: int) -> list[dict]:
    passi, resto = divmod(anno_a - anno_da, 5)
    assert resto == 0, "le classi sono quinquennali: il passo dev'essere multiplo di cinque"
    righe = []
    for t in CONFRONTO:
        for g in ("F", "M"):
            for i, eta_da in enumerate(CLASSI_QUINQ[:-passi]):
                eta_a = CLASSI_QUINQ[i + passi]
                n_da = conteggi_classi.loc[(t, g, eta_da), anno_da]
                n_a = conteggi_classi.loc[(t, g, eta_a), anno_a]
                righe.append({
                    "territorio": t, "nome_territorio": NOMI[t], "genere": g,
                    "anno_da": anno_da, "anno_a": anno_a, "anni": anno_a - anno_da,
                    "eta_da": eta_da, "eta_a": eta_a,
                    "n_da": int(n_da), "n_a": int(n_a),
                    "ritenzione_pct": round(100 * n_a / n_da, 1),
                    # 2011 è decennale e 2021 permanente: quel rapporto ha una gamba per
                    # fonte e va marcato. La distorsione nota va nel verso prudente: il
                    # censimento 2011 contò meno dell'anagrafe, quindi sta al denominatore
                    # del decennio che crolla e al numeratore di quello che tiene — il
                    # divario fra i due decenni è una stima per difetto in entrambe le gambe.
                    "fonti_diverse": anno_da < 2018 <= anno_a})
    return righe


ritenzione_decennale = pd.DataFrame([r for da, a in PERIODI for r in coorti_quinquennali(da, a)])
ritenzione_decennale.to_csv(PROCESSED / "genere_ritenzione_decennale.csv", index=False)

COORTE = "Y15-19"
decenni = ritenzione_decennale[ritenzione_decennale["eta_da"].eq(COORTE)]
for g, etichetta in (("F", "femmine"), ("M", "maschi")):
    print(f"Ritenzione della coorte {COORTE} — {etichetta} (100 = la coorte si conserva):")
    print(decenni[decenni["genere"].eq(g)]
          .assign(periodo=lambda d: d["anno_da"].astype(str) + "-" + d["anno_a"].astype(str))
          .pivot(index="periodo", columns="nome_territorio", values="ritenzione_pct")
          [ORDINE].to_string(), "\n")

ribaltamento = (decenni[decenni["anno_da"].isin([2001, 2011]) & decenni["territorio"].eq(BAGHERIA)]
                .pivot(index="genere", columns="anno_da", values="ritenzione_pct")
                .assign(scarto=lambda d: (d[2011] - d[2001]).round(1)))
print("Bagheria, scarto fra i due decenni sulla stessa coorte:")
print(ribaltamento.to_string())

Ritenzione della coorte Y15-19 — femmine (100 = la coorte si conserva):
nome_territorio  Bagheria  Palermo  Sicilia  Italia
periodo                                            
2001-2011           102.9     87.6     97.8   113.1
2011-2021            88.6     90.2     92.1   104.4
2018-2023            97.9     95.8     98.1   101.7
2019-2024           100.1     97.0     99.4   102.2 

Ritenzione della coorte Y15-19 — maschi (100 = la coorte si conserva):
nome_territorio  Bagheria  Palermo  Sicilia  Italia
periodo                                            
2001-2011            98.4     86.3     95.1   108.1
2011-2021            83.9     86.3     91.0   104.8
2018-2023            96.1     94.9     97.4   103.6
2019-2024            98.1     97.1    100.2   105.5 

Bagheria, scarto fra i due decenni sulla stessa coorte:
anno_da   2001  2011  scarto
genere                      
F        102.9  88.6   -14.3
M         98.4  83.9   -14.5


In [41]:
# ---- (3) fig05: la forbice è un anno o una tendenza? ----------------------------------
# Stesse formule di `genere_forbice.csv` (istruzione 9-24 «almeno il diploma», occupazione
# 15-24) ripetute su ogni anno disponibile, vicinato compreso. Ricalcolate qui e non
# riprese dalle tabelle a monte, che sono già filtrate sull'anno comune.
def serie_forbice(lavoro_t: pd.DataFrame, istruzione_t: pd.DataFrame, territorio: str) -> pd.DataFrame:
    occ = (lavoro_t[lavoro_t["eta"].eq("Y15-24") & lavoro_t["genere"].isin(["M", "F"])]
           .pivot_table(index="anno", columns=["condizione", "genere"], values="valore", aggfunc="sum"))
    # Il 2020 manca alla fonte su ogni classe che contenga i 15-24: niente riga, nessuna stima.
    occ = occ[occ[("99", "F")].gt(0) & occ[("99", "M")].gt(0)]
    istr = (istruzione_t[istruzione_t["eta"].eq("Y9-24") & istruzione_t["genere"].isin(["M", "F"])]
            .pivot_table(index="anno", columns=["titolo_studio", "genere"], values="valore", aggfunc="sum")
            .reindex(occ.index))
    tasso = lambda g: 100 * occ[("1", g)] / occ[("99", g)]
    diploma = lambda g: 100 * sum(istr[(k, g)] for k in ALMENO_DIPLOMA) / istr[("ALL", g)]
    return pd.DataFrame({
        "tasso_occupazione_F": tasso("F"), "tasso_occupazione_M": tasso("M"),
        "rapporto_M_F_occupazione": tasso("M") / tasso("F"),
        "almeno_diploma_F": diploma("F"), "almeno_diploma_M": diploma("M"),
        "vantaggio_istruzione_F_pp": diploma("F") - diploma("M"),
    }).assign(territorio=territorio,
              nome_territorio=NOMI.get(territorio, NOME_VICINATO)).round(2)


lavoro_confronto = istr_lav[istr_lav["tavola"].eq("lavoro")]
istruzione_tavola = istr_lav[istr_lav["tavola"].eq("istruzione")]
forbice_serie = pd.concat(
    [serie_forbice(lavoro_confronto[lavoro_confronto["territorio"].eq(t)],
                   istruzione_tavola[istruzione_tavola["territorio"].eq(t)], t) for t in CONFRONTO]
    + [serie_forbice(lavoro_vicini, istruzione_vicini, CODICE_VICINATO)]).reset_index()
forbice_serie.to_csv(PROCESSED / "genere_forbice_serie.csv", index=False)

# La serie deve riprodurre esattamente la fotografia che fig05 già disegna.
for colonna in ("rapporto_M_F_occupazione", "vantaggio_istruzione_F_pp", "tasso_occupazione_F"):
    atteso_fig = forbice.loc["Bagheria", colonna]
    ottenuto = forbice_serie.set_index(["territorio", "anno"]).loc[(BAGHERIA, anno_comune), colonna]
    assert abs(ottenuto - atteso_fig) < 0.02, f"{colonna}: serie {ottenuto} contro fotografia {atteso_fig}"

ORDINE_SERIE = ["Bagheria", NOME_VICINATO, "Palermo", "Sicilia", "Italia"]
for colonna, verso, titolo in (
        ("tasso_occupazione_F", "min", "tasso di occupazione femminile 15-24 (%)"),
        ("rapporto_M_F_occupazione", "max", "rapporto M/F sull'occupazione 15-24"),
        ("vantaggio_istruzione_F_pp", "max", "vantaggio educativo femminile 9-24 (punti)")):
    tabella = forbice_serie.pivot_table(index="anno", columns="nome_territorio", values=colonna)[ORDINE_SERIE]
    testa = tabella.idxmin(axis=1) if verso == "min" else tabella.idxmax(axis=1)
    print(f"{titolo} — in testa alla classifica «sbagliata»: "
          f"Bagheria in {(testa == 'Bagheria').sum()} anni su {len(testa)}")
    print(tabella.to_string())
    print(f"   per anno: {testa.to_dict()}\n")

tasso di occupazione femminile 15-24 (%) — in testa alla classifica «sbagliata»: Bagheria in 6 anni su 6
nome_territorio  Bagheria  vicinato (5 comuni)  Palermo  Sicilia  Italia
anno                                                                    
2018                 4.68                 5.94     6.34     7.52   14.07
2019                 5.16                 6.54     6.94     7.96   14.48
2021                 6.17                 7.46     7.53     8.42   15.00
2022                 7.69                 7.87     8.63     9.34   16.40
2023                 8.02                 8.09     9.21     9.94   16.90
2024                 8.19                 8.90     9.59    10.41   17.27
   per anno: {2018: 'Bagheria', 2019: 'Bagheria', 2021: 'Bagheria', 2022: 'Bagheria', 2023: 'Bagheria', 2024: 'Bagheria'}

rapporto M/F sull'occupazione 15-24 — in testa alla classifica «sbagliata»: Bagheria in 4 anni su 6
nome_territorio  Bagheria  vicinato (5 comuni)  Palermo  Sicilia  Italia
anno           

**📌 Risultato chiave** — Le tre verifiche danno tre esiti diversi, e vale la pena
tenerli distinti.

**fig04 — il claim del 2011 era una previsione, e ha tenuto.** Sui 390 comuni la
graduatoria del 2011 predice quella del 2024 con **ρ di Spearman 0.848**, attraverso
tredici anni e due disegni di rilevazione; dentro il solo censimento permanente
(2018 vs 2024) ρ sale a 0.919. Il **74%** dei comuni nel quintile più basso del 2011 è
ancora lì nel 2024, Bagheria compresa. Il livello però va aggiornato: Bagheria passa da
18.1% (12° percentile) a **23.7% (17° percentile)**, mentre la mediana regionale sale da
23.6% a 28.3%. Detta così: **nel 2024 Bagheria arriva dove stava la mediana siciliana nel
2011**, e i cinque comuni vicini restano tutti sotto la mediana anche adesso.

**fig07 — la fuga è databile, e cade nello stesso decennio del muro sul lavoro.** Sulla
coorte 15-19 anni seguita per dieci anni, Bagheria passa da **102.9% (2001-2011) a 88.6%
(2011-2021)** sulle femmine e da 98.4% a **83.9%** sui maschi: un ribaltamento di circa
quattordici punti. Nel primo decennio Bagheria *guadagnava* coorti mentre Palermo ne
perdeva il 13%; nel secondo ne perde più di Palermo sui maschi. I controlli interni al
permanente (2018-2023 e 2019-2024) confermano che la perdita è **attuale** e sta dove il
profilo per età singola la trova: transizione 20-24 → 25-29, femmine 92.8% e 93.3% contro
il 102% italiano. La finestra 22-25 non è un artefatto di tre anni di dati.

**fig05 — qui il problema non è l'età del dato, è che poggia su un anno solo.** Delle due
classifiche che la figura racconta, una regge e una no. Il **tasso di occupazione
femminile 15-24 di Bagheria è il minimo del panel in ognuno dei sei anni
disponibili** (4.7% →
8.2%), e il vantaggio educativo femminile cresce da 3.1 a 4.2 punti mentre nel vicinato
crolla da 2.9 a 0.5: la forbice si allarga davvero. Il **rapporto M/F sull'occupazione**
invece oscilla — 2.47 nel 2018, **1.89 nel 2023** (il migliore del gruppo locale, meglio
di vicinato e Sicilia), 2.01 nel 2024: in testa alla classifica «sbagliata»
in quattro anni su sei. «Prima in entrambe le classifiche» è vero nel 2024 per tre
centesimi e non lo era né nel 2022 né nel 2023: la figura va appoggiata sul livello
femminile e sulla forbice, non sul rapporto.

# **Sintesi finale**

I risultati delle sezioni precedenti, ricomposti nell'ordine in cui reggono la proposal.
Il riferimento fra virgolette è la sezione che produce la cifra: nulla qui è calcolato a
mano, tutto si rigenera rieseguendo il notebook.

---

### 1. Il paradosso istruzione / lavoro — il risultato centrale
* **Istruzione (9-24)**: 33.4% F contro 29.2% M, gap **-4.2 pp** — le ragazze studiano di più *(«Gap di genere sull'istruzione»)*.
* **Occupazione (15-24)**: 8.2% F contro 16.5% M, gap **+8.3 pp** — lavorano la metà *(«Gap di genere sull'occupazione»)*.
* **Meccanismo**: a 15-24 la differenza è assorbita dalla permanenza nello studio (65.1% F contro 56.4% M), non dall'inattività *(«Dentro il "fuori da lavoro e studio"»)*.
* I due segni sono **opposti in tutti e quattro i territori**: il problema non è che le ragazze si formino meno, è che il vantaggio formativo non si converte.

---

### 2. Il problema non è l'ampiezza del gap: è il livello
* **In punti** (8.3 pp) Bagheria sta sopra Palermo (6.9) ma sotto Sicilia (9.9) e Italia (9.7); il modello LPM lo conferma formalmente sul pooled 2022-2024 (+1.1 vs Palermo, p=0.03; -1.8/-1.9 vs Sicilia/Italia, p<0.001) *(«Il gap di Bagheria è un'anomalia locale?»)*.
* **In rapporto M/F** (2.01) Bagheria è invece la **peggiore del panel** (Palermo 1.72, Sicilia 1.95, Italia 1.56) *(«Punti percentuali o rapporto?»)*.
* Gli **effetti principali** dello stesso modello testano il livello: tasso femminile di Bagheria sotto Palermo di 1.2 pp, sotto la Sicilia di 1.9, sotto l'Italia di 8.9 (p ≤ 0.0001 sul pooled 2022-2024, p ≤ 0.009 anche sul solo 2024) *(«Il gap di Bagheria è un'anomalia locale?»)*.
* Il bersaglio della proposal è quindi il **livello dell'occupazione femminile — 8.2%, il più basso dei quattro territori** — non la chiusura di un divario "anomalo" che i dati non mostrano.

---

### 3. Il carico di cura, reso visibile
* Il **13.4% delle ragazze 15-24 (387 persone)** si dichiara casalinga, contro l'1.7% dei coetanei e il 4.6% delle coetanee italiane *(«Dentro gli "altri inattivi"»)*.
* Gradiente territoriale netto (Italia 4.6 < Sicilia 10.1 < Palermo 11.3 < Bagheria 13.4) e serie 2018-2024 stabile fra 320 e 420 ragazze: **fenomeno strutturale, non episodico**.
* **Il canale non è il matrimonio precoce**: al 1.1.2025 le ragazze 15-24 già coniugate sono **41** contro 387 casalinghe — almeno l'**89% delle casalinghe è nubile** — e la quota di già coniugate 20-24 di Bagheria (2.7%) sta *sotto* Palermo (3.2%) e Sicilia (2.9%) *(«Le casalinghe sono coniugate?»)*. Il ruolo si assume da nubili, nella famiglia d'origine: il servizio è di **attivazione**, non solo di conciliazione.
* È l'unico meccanismo del thread con un target nominabile e un KPI naturale già pronto.

---

### 4. Il gap si allarga — lentamente, ma davvero
* +0.21 pp/anno a Bagheria (CI 0.06-0.35, p=0.008), passo indistinguibile da Palermo e Italia *(«Il gap si sta allargando?»)*.
* Nello stesso periodo il rapporto M/F **scende** (2.47 → 2.01): entrambi i tassi salgono, quello maschile di più in valore assoluto. Ogni claim deve dichiarare quale delle due scale sta usando.

---

### 5. La fuga di talenti ha un tempismo di genere
* Bagheria trattiene meno dell'Italia in **ogni** coorte e per entrambi i generi: il drenaggio riguarda tutti *(«Ritenzione di coorte per genere»)*.
* Ma i ragazzi si perdono a 15-19 (98.5 contro 104.8), le ragazze **dopo i 25** (96.3 contro 103.0). Le ragazze restano finché studiano; il territorio le perde all'uscita dal percorso formativo — esattamente il punto in cui il vantaggio educativo dovrebbe diventare lavoro.
* La finestra 22-25 è una lettura **pooled 2021→2024**: nelle tre transizioni annuali il segno femminile compare ma oscilla (fino a 8 pp sulla stessa età, contro il ~±0.8 del solo rumore di conteggio) *(«La finestra 22-25 regge?»)*. La transizione singola è un controllo di direzione, non un titolo.

---

### 6. Sotto il gap di genere ce n'è uno territoriale
* La quota di "altri inattivi" 15-24 è 14-20% a Bagheria, Palermo e Sicilia contro il ~9% nazionale, **per entrambi i generi** *(«Dentro il "fuori da lavoro e studio"»)*.
* E il 2011 mostra radici lunghe: occupazione femminile 15+ al 18.1% contro il 36.1% nazionale *(«Contesto storico 2011»)*.
* Un intervento di genere agisce quindi su uno svantaggio doppio: essere giovane a Bagheria, ed esservi giovane donna.

---

### 7. Traduzione in persone (per il template della proposal)
Base 2024: **2.882 ragazze 15-24, 236 occupate** *(«Il gap in persone»)*.

| Scenario | Occupate in più | Uso |
|---|---|---|
| Tasso femminile di Palermo | **+40** | KPI realistico a 2-3 anni |
| Parità con i coetanei maschi | +239 | misura del problema |
| Tasso femminile nazionale | +262 | misura del problema |

---

### 8. Il KPI ha una finestra di lettura, non solo un valore
* "+40 occupate" = **+1.40 pp**, sotto l'MDE della lettura annuale (2.14 pp; potenza 46%, una moneta). Su **finestre pooled di tre anni** la potenza sale al 90% (MDE 1.21 pp); per le casalinghe (-2.1 pp verso Palermo): dal 68% al 99% *(«Il KPI si può misurare?»)*.
* Conseguenza: i KPI primari si valutano su trienni pooled (2022-2024 contro 2025-2027); la lettura annuale spetta a indicatori di processo (utenza per età e genere) che oggi **nessuno rileva** — è la saldatura quantitativa con la proposta del presidio di misurazione.

---

### 9. Dentro il gruppo dei pari: le gemelle
* Dieci comuni comparabili per struttura (matching Mahalanobis su variabili non-outcome, 2011; robustezza 8/10 e 6/10 sugli altri metodi, LOVO ≥7/10) *(«Le gemelle di Bagheria»)*: Santa Flavia, Capaci, Misilmeri, Altofonte, Trabia, Terrasini, Termini Imerese, Erice, Porto Empedocle, Sciacca.
* Sui livelli generali Bagheria sta come chi le somiglia (NEET 4/10, occupazione F 3/10): lo svantaggio giovanile è **di fascia territoriale**, e la proposal deve dirlo. Il profilo specifico è altrove: differenziale educativo pro-F più forte (1/10), disoccupazione F fra le peggiori (8/10 sotto), **autonomia giovanile al minimo** (F4: nessuna sotto), mobilità quasi minima (2/10).
* Su L11 Bagheria era **identica alle gemelle** nel 1991 e nel 2001; se ne stacca nel decennio 2001-2011 (18.1 contro 19.8) *(«Trend paralleli»)*: lo scarto è recente, e la diagnosi deve spiegare quel decennio.

---

### 10. Il gap delle madri: partecipazione su, assorbimento no
* 1991→2011 (15+): partecipazione femminile dal **8° al 32° percentile** siciliano (20.6→28.7), ma occupazione femminile dal 22° al **12°** e disoccupazione femminile dal 46° al **95°** *(«Il gap delle madri»)*: le donne sono entrate nel mercato, il mercato non le ha assorbite.
* Il **sorpasso educativo è del 2011** (I1 98.9, unico territorio del panel sotto la parità): le ragazze del censimento permanente sono la prima generazione figlia del sorpasso — e trovano lo stesso muro. Il vantaggio educativo non convertito è il **regime del territorio da almeno un decennio**, non un incidente della serie 2018-2024.

---

### 11. Il "19% invisibile" è di entrambi i generi; l'etichetta no
* Fuori da lavoro, studio e ricerca (2024): **573 ragazze e 549 ragazzi** — 51% femminile *(«La ritenzione per età»)*. Di genere è la **composizione**: 387 casalinghe contro 50; 183 contro 485 in "altra condizione". Un intervento "per le invisibili" che ignorasse i ragazzi sbaglierebbe platea di metà; uno neutro che ignorasse l'etichetta mancherebbe il meccanismo.
* Il profilo per età data le rotture: ragazze **sopra la pari fino ai 23**, cedimento dalle età 24-25 (2021) in poi; ragazzi a due onde (17-19 e 23-24) con **rientri netti dopo i 26**, che le ragazze non hanno.

---

### 12. Il contesto familiare: autonomia al minimo, famiglia precoce
* Giovani che vivono da soli (F4) al **3° percentile** siciliano, ultimo anche nel gruppo gemelle; coppie giovani con figli (F7) all'81° dei 390 — ma nella mediana delle gemelle: tratto di fascia, non anomalia locale *(«Formazione familiare precoce»)*. Posizioni già così nel 1991.
* Bounds sulle casalinghe: fra il **18.8% delle 18-24enni** (nessuna minorenne) e il 25.8% delle 20-24enni (tutte maggiori di 19) *(«Chi sono le casalinghe?»)*; eccesso sull'incidenza italiana: **254 ragazze**. L'età resta non osservata: bounds, non stime.

---

### 13. Il bilancio dei giovani: la platea, il suo audit, il ricambio assente
* **La platea del target è già nata e si sta restringendo**: a saldo migratorio zero le ragazze 15-24 passano da 2.882 (2024) a 2.651 nel 2029 (-8.0%) e 2.435 nel 2034 (**-15.5%**, contro il -5.8% dei coetanei); nei benchmark il calo è simile in ampiezza ma quasi simmetrico fra i generi *(«Il bilancio dei giovani»)*. I KPI in teste si riparametrano sulla platea corrente (`genere_platea.csv`); quelli in tasso restano validi. E il conteggio è un **tetto**: la ritenzione osservata sta sotto quota 100.
* **L'asimmetria ha un'origine anomala, documentata e replicata**: 117 maschi ogni 100 femmine fra i 5-14enni (2024) contro il 104-106 dei benchmark; nella norma nel 2001 e 2011, in salita da allora (z +3.5 sull'ultima finestra, identica su due tavole indipendenti, tutta nella popolazione italiana) *(«L'audit della platea»)*. Meccanismo aperto: il dato si cita **solo insieme al suo audit**.
* **Il ricambio migratorio non esiste**: stranieri all'**1.6% del 15-34** (195 persone) contro 5.1% Palermo, 6.4% Sicilia, 12.4% Italia; fra 2021 e 2024 il 15-34 perde 350 italiani (219 ragazze) e guadagna 37 stranieri *(«La componente straniera»)*: la fuga di Bagheria è al netto di niente.

---

### Cosa entra nella proposal, e con quale KPI
L'intervento è la colonna che il team deve riempire: qui ci sono solo evidenza, target e
KPI, che sono fatti misurabili e si rigenerano da questo notebook.

| Evidenza (sezione) | Target | KPI misurabile |
|---|---|---|
| Occupazione femminile 15-24 all'8.2%, la più bassa del panel, con rapporto M/F 2.01 *(«Punti percentuali o rapporto?», «Il gap in persone»)* | le 2.882 ragazze 15-24 residenti | +40 occupate = tasso femminile allineato a Palermo (9.6%) entro 2-3 anni |
| 387 ragazze 15-24 casalinghe, 13.4% contro il 4.6% nazionale, stabile dal 2018; almeno l'89% nubile *(«Dentro gli "altri inattivi"», «Le casalinghe sono coniugate?»)* | le ~390 casalinghe 15-24 | quota casalinghe 15-24: prima al livello di Palermo (11.3%), poi verso quello nazionale |
| Ritenzione femminile 25-29 al 96.3 contro il 103.0 nazionale *(«Ritenzione di coorte»)* | le coorti femminili in uscita dal percorso formativo | ritenzione netta 25-29 F portata almeno al livello di Palermo (98.8) |

**Finestra di lettura** *(«Il KPI si può misurare?»)*: tasso e quota casalinghe si valutano su **trienni pooled** (potenza 90% e 99%), non anno su anno; la ritenzione contro la variabilità osservata (±0.5 punti sulle letture annue). I KPI espressi in teste si riparametrano sulla platea corrente di `genere_platea.csv` — la platea femminile 15-24 cala dell'8% già entro il 2029 a migrazione ferma *(«Il bilancio dei giovani»)*. **Controfattuale dichiarato in anticipo** *(«Trend paralleli»)*: Palermo (pendenze indistinguibili sul pre-periodo 2018-2024), con la mediana delle gemelle come ancora dei target di lungo periodo.

### Cosa resta nell'analisi e non entra nella proposal
* **Decomposizione del gap per titolo di studio**: non calcolabile a livello comunale *(«Verifica di fattibilità»)*.
* **Tasso di disoccupazione femminile**: denominatore di 430-680 persone, oscillazioni in larga parte rumore *(«Dentro il "fuori da lavoro e studio"»)*.
* **Magnitudine del gap di istruzione**: ~1.5 dei 4.2 punti sono composizione per età; il segno regge, il numero va citato con la cautela *(«Verifica di composizione per età»)*.
* **Primato del vantaggio educativo**: vero sul 9-24 di fig05; sulle fasce allineate all'età da diploma non regge (15-24: +4.8 con la Sicilia a +4.7; 18-24: +5.7 contro 7.1 Sicilia e 6.2 Italia). Nei claim usare il distacco dal vicinato e la mancata conversione, non il primato assoluto *(«Su 1.000 ragazze»)*.
* **Indicatori 2011**: sfondo storico su 15+, mai in serie con il 2018-2024 *(«Contesto storico 2011»)*.
* **Correlazioni sui 390 comuni** (nuvola, mobilità): ecologiche e al 2011 — orientano le ipotesi, non dimostrano meccanismi *(«La nuvola dei 390»)*.
* **F5-F7**: denominatore = totale famiglie, quindi sensibili alla struttura per età; si citano percentili e gemelle, non i livelli *(«Formazione familiare precoce»)*.
* **Sex ratio 5-14**: anomalia replicata su due tavole ma senza meccanismo identificato; i check decisivi (nati per sesso del comune, stessa serie sulle gemelle) sono fetch nuovi, da decidere in team *(«L'audit della platea»)*.
* **Stato civile (DCIS_POPRES1)**: fonte diversa dal censimento permanente — stock al 1° gennaio contro media annua — mai in serie con SETA_1; osserva il matrimonio formale, non le convivenze né la maternità *(«Le casalinghe sono coniugate?»)*.
* **Transizioni annuali di ritenzione**: controllo di direzione della finestra 22-25, non misura autonoma *(«La finestra 22-25 regge?»)*.
* **Trend paralleli**: non rifiutati su sei punti ≠ dimostrati; il pre-periodo del confronto con le gemelle è ora misurato fino al 2024 *(«Il ponte fra i due censimenti»)*, ma il DiD resta incalcolabile finché un intervento non esiste *(«Trend paralleli»)*.

# **Tabelle esportate**
Le figure R di questo thread leggono da qui. Nessun ricalcolo in R.

In [42]:
pd.DataFrame(
    [(p.name, sum(1 for _ in p.open()) - 1) for p in sorted(PROCESSED.glob("genere_*.csv"))],
    columns=["file", "righe"])

,file,righe
0,genere_base_persone.csv,1
1,genere_casalinghe.csv,72
2,genere_casalinghe_bounds.csv,3
3,genere_coerenza_fonti.csv,16
4,genere_composizione_stato.csv,288
5,genere_composizione_stato_dettaglio.csv,432
6,genere_composizione_stato_dettaglio_vicini.csv,108
7,genere_coorti.csv,24
8,genere_coorti_vicini.csv,6
9,genere_distribuzione_390.csv,7
